In [3]:
# ================================================================================================
# L24 — CORRECTED VISUAL RE-RENDER
#
# Uses the EXACT hand geometry from the working l1.ipynb:
#
#   OpenVINO palm detector
#       ↓
#   aspect-ratio-preserving letterbox
#       ↓
#   2016-anchor palm decoding
#       ↓
#   weighted NMS
#       ↓
#   remove palm letterbox
#       ↓
#   MediaPipe-style rotated hand ROI
#       ↓
#   224×224 perspective crop
#       ↓
#   OpenVINO 21-landmark model
#       ↓
#   exact inverse perspective transform
#       ↓
#   skeleton on ORIGINAL frame
#
# Classification is NOT rerun.
# Frozen L24 predictions remain unchanged.
#
# Deletes ONLY the incorrect generated L24 visualization MP4s.
# ================================================================================================

from pathlib import Path
import math
import time

import cv2
import numpy as np
import pandas as pd
import openvino as ov

from IPython.display import Video, display


# ================================================================================================
# 1. PATHS
# ================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

L24_ROOT = (
    LANDMARK_ROOT /
    "audit" /
    "l24_cislr_mp4_openvino_demo"
)

VIDEO_OUTPUT_DIR = (
    L24_ROOT /
    "videos"
)

PREDICTIONS_CSV = (
    L24_ROOT /
    "reports" /
    "l24_mp4_predictions.csv"
)


# ================================================================================================
# IMPORTANT:
# These are the SAME model locations used by your working l1.ipynb.
# ================================================================================================

PALM_MODEL = (
    LANDMARK_ROOT /
    "models" /
    "pretrained" /
    "palm" /
    "palm_detection_full.tflite"
)

HAND_MODEL = (
    LANDMARK_ROOT /
    "models" /
    "pretrained" /
    "hand_landmark" /
    "hand_landmark_full.tflite"
)


DEVICE = "CPU"

PALM_SIZE = 192

LANDMARK_SIZE = 224

PALM_THRESHOLD = 0.50

NMS_THRESHOLD = 0.30

MAX_HANDS = 2


print("=" * 120)
print("L24 — CORRECTED L1-BASED HAND VISUALIZATION")
print("=" * 120)

print("Palm model:")
print(PALM_MODEL)

print("\nLandmark model:")
print(HAND_MODEL)

print("\nFrozen predictions:")
print(PREDICTIONS_CSV)

print("\nOutput:")
print(VIDEO_OUTPUT_DIR)


for path in [
    PALM_MODEL,
    HAND_MODEL,
    PREDICTIONS_CSV
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


print("\n[PASS] Required files found.")


# ================================================================================================
# 2. LOAD FROZEN L24 CLASSIFICATION RESULTS
# ================================================================================================

print("\n" + "=" * 120)
print("2. LOAD FROZEN L24 RESULTS")
print("=" * 120)


df = pd.read_csv(
    PREDICTIONS_CSV
)


if len(df) != 10:

    raise RuntimeError(
        f"Expected 10 rows, found {len(df)}"
    )


required_columns = [

    "uid",
    "expected_gloss",
    "predicted_gloss",
    "confidence",
    "correct",
    "source_video"
]


for column in required_columns:

    if column not in df.columns:

        raise RuntimeError(
            f"Missing column: {column}"
        )


print("Rows:", len(df))

print("\n[PASS] Frozen L24 results loaded.")
print("[PASS] VideoMAE classification will NOT be rerun.")


# ================================================================================================
# 3. DELETE ONLY WRONG GENERATED DEMO VIDEOS
# ================================================================================================

print("\n" + "=" * 120)
print("3. DELETE PREVIOUS INCORRECT VISUALIZATION VIDEOS")
print("=" * 120)


VIDEO_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


old_videos = list(
    VIDEO_OUTPUT_DIR.glob(
        "*.mp4"
    )
)


print(
    "Videos to delete:",
    len(old_videos)
)


for path in old_videos:

    print(
        "Deleting:",
        path.name
    )

    path.unlink()


print(
    f"\n[PASS] Deleted {len(old_videos)} generated demo videos."
)

print(
    "[PASS] Original CISLR videos untouched."
)


# ================================================================================================
# 4. LOAD EXACT L1 OPENVINO MODELS
# ================================================================================================

print("\n" + "=" * 120)
print("4. LOAD OPENVINO PALM + LANDMARK MODELS")
print("=" * 120)


core = ov.Core()


palm_model = core.read_model(
    str(PALM_MODEL)
)


hand_model = core.read_model(
    str(HAND_MODEL)
)


palm_net = core.compile_model(
    palm_model,
    DEVICE
)


hand_net = core.compile_model(
    hand_model,
    DEVICE
)


print(
    "OpenVINO:",
    ov.__version__
)


print(
    "Device:",
    DEVICE
)


print("\nPalm input:")

for x in palm_net.inputs:

    print(
        " ",
        x.shape
    )


print("Palm outputs:")

for x in palm_net.outputs:

    print(
        " ",
        x.shape
    )


print("\nLandmark input:")

for x in hand_net.inputs:

    print(
        " ",
        x.shape
    )


print("Landmark outputs:")

for x in hand_net.outputs:

    print(
        " ",
        x.shape
    )


print(
    "\n[PASS] Exact L1 OpenVINO models compiled."
)


# ================================================================================================
# 5. EXACT L1 PALM ANCHORS
# ================================================================================================

print("\n" + "=" * 120)
print("5. BUILD EXACT L1 PALM ANCHORS")
print("=" * 120)


def generate_anchors():

    num_layers = 4

    strides = [
        8,
        16,
        16,
        16
    ]

    anchors = []

    layer = 0


    while layer < num_layers:

        same = layer

        anchors_per_cell = 0


        while (
            same < num_layers
            and
            strides[same] == strides[layer]
        ):

            # Normal anchor
            anchors_per_cell += 1

            # Interpolated anchor
            anchors_per_cell += 1

            same += 1


        stride = strides[
            layer
        ]


        fh = int(
            math.ceil(
                PALM_SIZE /
                stride
            )
        )


        fw = int(
            math.ceil(
                PALM_SIZE /
                stride
            )
        )


        for y in range(fh):

            for x in range(fw):

                cx = (
                    x + 0.5
                ) / fw

                cy = (
                    y + 0.5
                ) / fh


                for _ in range(
                    anchors_per_cell
                ):

                    anchors.append(
                        [
                            cx,
                            cy,
                            1.0,
                            1.0
                        ]
                    )


        layer = same


    return np.asarray(
        anchors,
        dtype=np.float32
    )


ANCHORS = generate_anchors()


print(
    "Anchors:",
    ANCHORS.shape
)


assert ANCHORS.shape == (
    2016,
    4
)


print(
    "[PASS] Exact 2016-anchor L1 configuration."
)


# ================================================================================================
# 6. EXACT L1 PALM PREPROCESSING
#
# THIS IS IMPORTANT.
#
# The previous bad L24 code resized directly to 192×192.
# Your working L1 preserves aspect ratio and LETTERBOXES.
# ================================================================================================

def preprocess_palm(frame):

    h, w = frame.shape[
        :2
    ]


    scale = min(

        PALM_SIZE / w,

        PALM_SIZE / h
    )


    nw = int(
        round(
            w * scale
        )
    )


    nh = int(
        round(
            h * scale
        )
    )


    resized = cv2.resize(

        frame,

        (
            nw,
            nh
        ),

        interpolation=
            cv2.INTER_LINEAR
    )


    rgb = cv2.cvtColor(

        resized,

        cv2.COLOR_BGR2RGB
    )


    left = (
        PALM_SIZE -
        nw
    ) // 2


    top = (
        PALM_SIZE -
        nh
    ) // 2


    right = (
        PALM_SIZE -
        nw -
        left
    )


    bottom = (
        PALM_SIZE -
        nh -
        top
    )


    padded = cv2.copyMakeBorder(

        rgb,

        top,
        bottom,
        left,
        right,

        cv2.BORDER_CONSTANT,

        value=(
            0,
            0,
            0
        )
    )


    # EXACT L1 normalization:
    # RGB / 255
    #
    # NOT [-1,+1].

    tensor = (

        padded.astype(
            np.float32
        )

        /

        255.0
    )


    shape = list(
        palm_net.input(0).shape
    )


    if shape == [
        1,
        192,
        192,
        3
    ]:

        tensor = tensor[
            None
        ]


    elif shape == [
        1,
        3,
        192,
        192
    ]:

        tensor = np.transpose(

            tensor,

            (
                2,
                0,
                1
            )
        )[
            None
        ]


    else:

        raise RuntimeError(

            f"Unexpected palm input shape: "
            f"{shape}"
        )


    return tensor, {

        "scale":
            scale,

        "left":
            left,

        "top":
            top
    }


# ================================================================================================
# 7. EXACT L1 PALM INFERENCE
# ================================================================================================

def palm_inference(frame):

    tensor, info = preprocess_palm(
        frame
    )


    result = palm_net(
        [
            tensor
        ]
    )


    reg = None

    logits = None


    for out in palm_net.outputs:

        arr = np.asarray(
            result[
                out
            ]
        )


        if arr.size == (
            2016 *
            18
        ):

            reg = arr.reshape(
                2016,
                18
            )


        elif arr.size == 2016:

            logits = arr.reshape(
                2016
            )


    if (
        reg is None
        or
        logits is None
    ):

        raise RuntimeError(
            "Could not identify palm outputs."
        )


    return (
        reg,
        logits,
        info
    )


# ================================================================================================
# 8. EXACT L1 PALM DECODING
# ================================================================================================

def sigmoid(x):

    x = np.clip(
        x,
        -100,
        100
    )


    return (
        1.0 /
        (
            1.0 +
            np.exp(
                -x
            )
        )
    )


def decode_palms(
    reg,
    logits
):

    scores = sigmoid(
        logits
    )


    ids = np.where(

        scores >=
        PALM_THRESHOLD

    )[0]


    detections = []


    for i in ids:

        r = reg[
            i
        ]


        ax, ay, aw, ah = ANCHORS[
            i
        ]


        cx = (

            r[0] /
            192.0 *
            aw

            +

            ax
        )


        cy = (

            r[1] /
            192.0 *
            ah

            +

            ay
        )


        bw = (

            r[2] /
            192.0 *
            aw
        )


        bh = (

            r[3] /
            192.0 *
            ah
        )


        box = np.array(

            [
                cx - bw / 2,
                cy - bh / 2,
                cx + bw / 2,
                cy + bh / 2
            ],

            dtype=np.float32
        )


        kp = []


        for k in range(7):

            j = (
                4 +
                2 * k
            )


            x = (

                r[j] /
                192.0 *
                aw

                +

                ax
            )


            y = (

                r[j + 1] /
                192.0 *
                ah

                +

                ay
            )


            kp.append(
                [
                    x,
                    y
                ]
            )


        detections.append({

            "score":
                float(
                    scores[i]
                ),

            "bbox":
                box,

            "keypoints":
                np.asarray(
                    kp,
                    dtype=np.float32
                )
        })


    detections.sort(

        key=lambda d:
            d[
                "score"
            ],

        reverse=True
    )


    return detections


# ================================================================================================
# 9. EXACT L1 WEIGHTED NMS
# ================================================================================================

def IoU(
    a,
    b
):

    x1 = max(
        a[0],
        b[0]
    )


    y1 = max(
        a[1],
        b[1]
    )


    x2 = min(
        a[2],
        b[2]
    )


    y2 = min(
        a[3],
        b[3]
    )


    inter = (

        max(
            0.0,
            x2 - x1
        )

        *

        max(
            0.0,
            y2 - y1
        )
    )


    aa = (

        max(
            0.0,
            a[2] - a[0]
        )

        *

        max(
            0.0,
            a[3] - a[1]
        )
    )


    ab = (

        max(
            0.0,
            b[2] - b[0]
        )

        *

        max(
            0.0,
            b[3] - b[1]
        )
    )


    union = (
        aa +
        ab -
        inter
    )


    return (
        inter / union
        if union > 0
        else 0.0
    )


def weighted_nms(
    detections
):

    remaining = detections.copy()

    output = []


    while remaining:

        reference = remaining[
            0
        ]


        group = []

        rest = []


        for detection in remaining:

            if IoU(

                reference[
                    "bbox"
                ],

                detection[
                    "bbox"
                ]

            ) > NMS_THRESHOLD:

                group.append(
                    detection
                )


            else:

                rest.append(
                    detection
                )


        weights = np.asarray(

            [
                d[
                    "score"
                ]

                for d in group
            ],

            dtype=np.float32
        )


        total = weights.sum()


        boxes = np.stack(

            [
                d[
                    "bbox"
                ]

                for d in group
            ]
        )


        keypoints = np.stack(

            [
                d[
                    "keypoints"
                ]

                for d in group
            ]
        )


        merged_box = (

            boxes *
            weights[
                :,
                None
            ]

        ).sum(
            axis=0
        ) / total


        merged_keypoints = (

            keypoints *
            weights[
                :,
                None,
                None
            ]

        ).sum(
            axis=0
        ) / total


        output.append({

            "score":
                max(
                    d[
                        "score"
                    ]
                    for d in group
                ),

            "bbox":
                merged_box,

            "keypoints":
                merged_keypoints
        })


        remaining = rest


    output.sort(

        key=lambda d:
            d[
                "score"
            ],

        reverse=True
    )


    return output[
        :MAX_HANDS
    ]


# ================================================================================================
# 10. EXACT L1 LETTERBOX REMOVAL
#
# THIS FIXES THE PALM BOX POSITION.
# ================================================================================================

def palm_to_original(
    p,
    info
):

    x = (

        p[0] *
        PALM_SIZE

        -

        info[
            "left"
        ]

    ) / info[
        "scale"
    ]


    y = (

        p[1] *
        PALM_SIZE

        -

        info[
            "top"
        ]

    ) / info[
        "scale"
    ]


    return np.array(

        [
            x,
            y
        ],

        dtype=np.float32
    )


def palm_detection_to_original(
    detection,
    info
):

    box = detection[
        "bbox"
    ]


    p1 = palm_to_original(

        [
            box[0],
            box[1]
        ],

        info
    )


    p2 = palm_to_original(

        [
            box[2],
            box[3]
        ],

        info
    )


    keypoints = np.stack(

        [
            palm_to_original(
                p,
                info
            )

            for p in detection[
                "keypoints"
            ]
        ]
    )


    return {

        "score":
            detection[
                "score"
            ],

        "bbox":
            np.array(

                [
                    p1[0],
                    p1[1],
                    p2[0],
                    p2[1]
                ],

                dtype=np.float32
            ),

        "keypoints":
            keypoints
    }


# ================================================================================================
# 11. EXACT MEDIAPIPE-STYLE PALM -> ROTATED HAND ROI
#
# From your working L1:
#
# wrist -> middle MCP
# target angle = 90°
# shift_y = -0.5
# scale_x = scale_y = 2.6
# square_long = true
# ================================================================================================

def normalize_radians(
    angle
):

    return (

        angle

        -

        2 *
        math.pi

        *

        math.floor(

            (
                angle +
                math.pi
            )

            /

            (
                2 *
                math.pi
            )
        )
    )


def palm_to_hand_rect(
    detection
):

    box = detection[
        "bbox"
    ]


    x1, y1, x2, y2 = box


    box_w = (
        x2 -
        x1
    )


    box_h = (
        y2 -
        y1
    )


    cx = (
        x1 +
        x2
    ) / 2.0


    cy = (
        y1 +
        y2
    ) / 2.0


    long_side = max(
        box_w,
        box_h
    )


    rect_w = long_side

    rect_h = long_side


    # --------------------------------------------------------------------------------------------
    # wrist -> middle MCP
    # --------------------------------------------------------------------------------------------

    wrist = detection[
        "keypoints"
    ][0]


    middle = detection[
        "keypoints"
    ][2]


    angle = math.atan2(

        -(
            middle[1] -
            wrist[1]
        ),

        middle[0] -
        wrist[0]
    )


    rotation = normalize_radians(

        math.pi /
        2.0

        -

        angle
    )


    # --------------------------------------------------------------------------------------------
    # Exact MediaPipe-style shift.
    # --------------------------------------------------------------------------------------------

    shift_y = -0.5


    x_shift = (

        -rect_h *
        shift_y *
        math.sin(
            rotation
        )
    )


    y_shift = (

        rect_h *
        shift_y *
        math.cos(
            rotation
        )
    )


    cx += x_shift

    cy += y_shift


    # --------------------------------------------------------------------------------------------
    # Exact L1 scale.
    # --------------------------------------------------------------------------------------------

    rect_w *= 2.6

    rect_h *= 2.6


    return {

        "cx":
            float(
                cx
            ),

        "cy":
            float(
                cy
            ),

        "width":
            float(
                rect_w
            ),

        "height":
            float(
                rect_h
            ),

        "rotation":
            float(
                rotation
            )
    }


# ================================================================================================
# 12. EXACT ROTATED ROI
# ================================================================================================

def rect_corners(
    rect
):

    cx = rect[
        "cx"
    ]

    cy = rect[
        "cy"
    ]


    rw = rect[
        "width"
    ]

    rh = rect[
        "height"
    ]


    rotation = rect[
        "rotation"
    ]


    c = math.cos(
        rotation
    )

    s = math.sin(
        rotation
    )


    local = np.array(

        [
            [
                -rw / 2,
                -rh / 2
            ],

            [
                rw / 2,
                -rh / 2
            ],

            [
                rw / 2,
                rh / 2
            ],

            [
                -rw / 2,
                rh / 2
            ]
        ],

        dtype=np.float32
    )


    R = np.array(

        [
            [
                c,
                -s
            ],

            [
                s,
                c
            ]
        ],

        dtype=np.float32
    )


    points = (

        local @
        R.T

        +

        np.array(
            [
                cx,
                cy
            ],
            dtype=np.float32
        )
    )


    return points.astype(
        np.float32
    )


def crop_hand_roi(
    frame,
    rect
):

    src = rect_corners(
        rect
    )


    dst = np.array(

        [
            [
                0,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                LANDMARK_SIZE - 1
            ],

            [
                0,
                LANDMARK_SIZE - 1
            ]
        ],

        dtype=np.float32
    )


    M = cv2.getPerspectiveTransform(

        src,
        dst
    )


    M_inv = cv2.getPerspectiveTransform(

        dst,
        src
    )


    crop = cv2.warpPerspective(

        frame,

        M,

        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),

        flags=
            cv2.INTER_LINEAR,

        borderMode=
            cv2.BORDER_CONSTANT,

        borderValue=(
            0,
            0,
            0
        )
    )


    return (
        crop,
        M,
        M_inv,
        src
    )


# ================================================================================================
# 13. EXACT L1 LANDMARK PREPROCESSING
# ================================================================================================

def preprocess_hand(
    crop
):

    rgb = cv2.cvtColor(

        crop,

        cv2.COLOR_BGR2RGB
    )


    tensor = (

        rgb.astype(
            np.float32
        )

        /

        255.0
    )


    shape = list(
        hand_net.input(0).shape
    )


    if shape == [
        1,
        224,
        224,
        3
    ]:

        tensor = tensor[
            None
        ]


    elif shape == [
        1,
        3,
        224,
        224
    ]:

        tensor = np.transpose(

            tensor,

            (
                2,
                0,
                1
            )
        )[
            None
        ]


    else:

        raise RuntimeError(

            f"Unexpected landmark input shape: "
            f"{shape}"
        )


    return tensor


# ================================================================================================
# 14. LANDMARK INFERENCE
#
# Same output identification as working L1,
# but without printing network diagnostics every video frame.
# ================================================================================================

def run_hand_landmark(
    crop
):

    tensor = preprocess_hand(
        crop
    )


    result = hand_net(
        [
            tensor
        ]
    )


    screen_landmarks = None

    world_landmarks = None

    scalar_outputs = []


    for i, out in enumerate(
        hand_net.outputs
    ):

        arr = np.asarray(
            result[
                out
            ]
        )


        flat = arr.reshape(
            -1
        )


        if flat.size == 63:

            candidate = flat.reshape(

                21,
                3

            ).astype(
                np.float32
            )


            xy_abs = float(

                np.max(

                    np.abs(

                        candidate[
                            :,
                            :2
                        ]
                    )
                )
            )


            if xy_abs > 2.0:

                screen_landmarks = (
                    candidate
                )


            else:

                world_landmarks = (
                    candidate
                )


        elif flat.size == 1:

            scalar_outputs.append(

                (
                    i,
                    float(
                        flat[0]
                    )
                )
            )


    if screen_landmarks is None:

        return None


    return (

        screen_landmarks,

        world_landmarks,

        scalar_outputs
    )


# ================================================================================================
# 15. EXACT INVERSE LANDMARK PROJECTION
#
# THIS FIXES THE SKELETON LOCATION.
# ================================================================================================

def project_landmarks_back(

    landmarks,

    M_inv,

    rect
):

    xy = landmarks[
        :,
        :2
    ].astype(
        np.float32
    )


    pts = xy.reshape(

        -1,
        1,
        2
    )


    projected = cv2.perspectiveTransform(

        pts,

        M_inv

    ).reshape(

        -1,
        2
    )


    z = (

        landmarks[
            :,
            2
        ]

        /

        LANDMARK_SIZE

        *

        rect[
            "width"
        ]
    )


    return np.column_stack(

        [
            projected,
            z
        ]

    ).astype(
        np.float32
    )


# ================================================================================================
# 16. COMPLETE EXACT L1 HAND PIPELINE FOR ONE FRAME
# ================================================================================================

def detect_hands_exact_l1(
    frame
):

    reg, logits, palm_info = (
        palm_inference(
            frame
        )
    )


    decoded = decode_palms(

        reg,
        logits
    )


    filtered = weighted_nms(
        decoded
    )


    palms = [

        palm_detection_to_original(

            detection,
            palm_info
        )

        for detection
        in filtered
    ]


    results = []


    for palm in palms:

        rect = palm_to_hand_rect(
            palm
        )


        (
            crop,
            M,
            M_inv,
            corners

        ) = crop_hand_roi(

            frame,
            rect
        )


        landmark_result = (
            run_hand_landmark(
                crop
            )
        )


        if landmark_result is None:

            continue


        (
            landmarks_crop,
            world_landmarks,
            scalars

        ) = landmark_result


        landmarks_image = (
            project_landmarks_back(

                landmarks_crop,

                M_inv,

                rect
            )
        )


        results.append({

            "palm":
                palm,

            "rect":
                rect,

            "corners":
                corners,

            "landmarks":
                landmarks_image,

            "landmarks_crop":
                landmarks_crop,

            "world_landmarks":
                world_landmarks,

            "scalars":
                scalars
        })


    return results


# ================================================================================================
# 17. EXACT L1 SKELETON
# ================================================================================================

HAND_CONNECTIONS = [

    # Thumb
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),

    # Index
    (0, 5),
    (5, 6),
    (6, 7),
    (7, 8),

    # Middle
    (5, 9),
    (9, 10),
    (10, 11),
    (11, 12),

    # Ring
    (9, 13),
    (13, 14),
    (14, 15),
    (15, 16),

    # Little
    (13, 17),
    (17, 18),
    (18, 19),
    (19, 20),

    # Palm
    (0, 17)
]


# ================================================================================================
# 18. DRAW CORRECT PALM + ROI + SKELETON
# ================================================================================================

def draw_hand(
    frame,
    hand_result
):

    H, W = frame.shape[
        :2
    ]


    palm = hand_result[
        "palm"
    ]


    landmarks = hand_result[
        "landmarks"
    ]


    corners = hand_result[
        "corners"
    ]


    # --------------------------------------------------------------------------------------------
    # PALM DETECTION BOX
    #
    # This is now in ORIGINAL IMAGE COORDINATES after exact letterbox removal.
    # --------------------------------------------------------------------------------------------

    bbox = palm[
        "bbox"
    ]


    px1 = int(
        round(
            bbox[0]
        )
    )

    py1 = int(
        round(
            bbox[1]
        )
    )

    px2 = int(
        round(
            bbox[2]
        )
    )

    py2 = int(
        round(
            bbox[3]
        )
    )


    px1 = max(
        0,
        min(
            W - 1,
            px1
        )
    )


    py1 = max(
        0,
        min(
            H - 1,
            py1
        )
    )


    px2 = max(
        0,
        min(
            W - 1,
            px2
        )
    )


    py2 = max(
        0,
        min(
            H - 1,
            py2
        )
    )


    cv2.rectangle(

        frame,

        (
            px1,
            py1
        ),

        (
            px2,
            py2
        ),

        (
            0,
            255,
            255
        ),

        1,

        cv2.LINE_AA
    )


    cv2.putText(

        frame,

        f"Palm {palm['score']:.2f}",

        (
            px1,
            max(
                12,
                py1 - 4
            )
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.28,

        (
            0,
            255,
            255
        ),

        1,

        cv2.LINE_AA
    )


    # --------------------------------------------------------------------------------------------
    # Rotated hand ROI.
    #
    # Thin cyan line so it doesn't dominate the visualization.
    # --------------------------------------------------------------------------------------------

    corners_int = np.round(
        corners
    ).astype(
        np.int32
    )


    cv2.polylines(

        frame,

        [
            corners_int
        ],

        True,

        (
            255,
            255,
            0
        ),

        1,

        cv2.LINE_AA
    )


    # --------------------------------------------------------------------------------------------
    # Skeleton
    # --------------------------------------------------------------------------------------------

    for a, b in HAND_CONNECTIONS:

        x1 = int(
            round(
                landmarks[
                    a,
                    0
                ]
            )
        )


        y1 = int(
            round(
                landmarks[
                    a,
                    1
                ]
            )
        )


        x2 = int(
            round(
                landmarks[
                    b,
                    0
                ]
            )
        )


        y2 = int(
            round(
                landmarks[
                    b,
                    1
                ]
            )
        )


        # Same sanity range as working L1.

        if (

            -W <= x1 <=
            2 * W

            and

            -H <= y1 <=
            2 * H

            and

            -W <= x2 <=
            2 * W

            and

            -H <= y2 <=
            2 * H
        ):

            cv2.line(

                frame,

                (
                    x1,
                    y1
                ),

                (
                    x2,
                    y2
                ),

                (
                    0,
                    255,
                    0
                ),

                2,

                cv2.LINE_AA
            )


    # --------------------------------------------------------------------------------------------
    # 21 landmarks
    #
    # Smaller than previous visualization so the hand stays visible.
    # --------------------------------------------------------------------------------------------

    for landmark_id in range(
        21
    ):

        x = int(
            round(
                landmarks[
                    landmark_id,
                    0
                ]
            )
        )


        y = int(
            round(
                landmarks[
                    landmark_id,
                    1
                ]
            )
        )


        if (
            0 <= x < W
            and
            0 <= y < H
        ):

            radius = (

                4

                if landmark_id == 0

                else 3
            )


            cv2.circle(

                frame,

                (
                    x,
                    y
                ),

                radius,

                (
                    0,
                    0,
                    255
                ),

                -1,

                cv2.LINE_AA
            )


            # Tiny white outline.

            cv2.circle(

                frame,

                (
                    x,
                    y
                ),

                radius + 1,

                (
                    255,
                    255,
                    255
                ),

                1,

                cv2.LINE_AA
            )


    return frame


# ================================================================================================
# 19. SMALL TOP INFORMATION BAR
#
# Smaller than your current screenshot.
# ================================================================================================

def draw_top_bar(

    frame,

    expected,

    predicted,

    confidence,

    correct
):

    H, W = frame.shape[
        :2
    ]


    overlay = frame.copy()


    # --------------------------------------------------------------------------------------------
    # Much smaller bar.
    # --------------------------------------------------------------------------------------------

    bar_height = min(
        34,
        H
    )


    cv2.rectangle(

        overlay,

        (
            0,
            0
        ),

        (
            W,
            bar_height
        ),

        (
            0,
            0,
            0
        ),

        -1
    )


    cv2.addWeighted(

        overlay,

        0.55,

        frame,

        0.45,

        0,

        frame
    )


    status = (

        "PASS"

        if correct

        else

        "FAIL"
    )


    # --------------------------------------------------------------------------------------------
    # Put everything on ONE compact line.
    # --------------------------------------------------------------------------------------------

    text = (

        f"Expected: {expected}  |  "
        f"Pred: {predicted}  |  "
        f"{confidence * 100:.1f}%  |  "
        f"{status}  |  OpenVINO"
    )


    cv2.putText(

        frame,

        text,

        (
            7,
            22
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.32,

        (
            255,
            255,
            255
        ),

        1,

        cv2.LINE_AA
    )


    return frame


# ================================================================================================
# 20. TEST ONE FRAME BEFORE PROCESSING ALL 10 VIDEOS
#
# This is intentional:
# We verify that the exact L1 geometry works on a CISLR frame before spending time rendering.
# ================================================================================================

print("\n" + "=" * 120)
print("20. ONE-FRAME GEOMETRY TEST")
print("=" * 120)


test_video = Path(
    df.iloc[
        0
    ][
        "source_video"
    ]
)


cap = cv2.VideoCapture(
    str(
        test_video
    )
)


if not cap.isOpened():

    raise RuntimeError(
        test_video
    )


frame_total = int(

    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


# Middle frame is more likely to contain the sign.

target_frame = max(

    0,

    frame_total //
    2
)


cap.set(

    cv2.CAP_PROP_POS_FRAMES,

    target_frame
)


ok, test_frame = cap.read()

cap.release()


if not ok:

    raise RuntimeError(
        "Could not read test frame."
    )


test_results = detect_hands_exact_l1(
    test_frame
)


print(
    "Test video:",
    test_video.name
)


print(
    "Frame:",
    target_frame
)


print(
    "Hands detected:",
    len(
        test_results
    )
)


test_draw = test_frame.copy()


for hand_result in test_results:

    test_draw = draw_hand(

        test_draw,

        hand_result
    )


test_preview = (

    L24_ROOT /
    "l24_hand_geometry_preview.jpg"
)


cv2.imwrite(

    str(
        test_preview
    ),

    test_draw
)


print(
    "Preview saved:"
)

print(
    test_preview
)


print()
print(
    "[PASS] One-frame L1 geometry test complete."
)


# ================================================================================================
# 21. RENDER ALL 10 CORRECTED VIDEOS
# ================================================================================================

print("\n" + "=" * 120)
print("21. RENDER CORRECTED L24 VIDEOS")
print("=" * 120)


generated = []


for row_index, row in df.iterrows():

    print()
    print("-" * 120)


    source_video = Path(
        row[
            "source_video"
        ]
    )


    uid = str(
        row[
            "uid"
        ]
    )


    expected = str(
        row[
            "expected_gloss"
        ]
    )


    predicted = str(
        row[
            "predicted_gloss"
        ]
    )


    confidence = float(
        row[
            "confidence"
        ]
    )


    correct_raw = row[
        "correct"
    ]


    if isinstance(
        correct_raw,
        str
    ):

        correct = (

            correct_raw
            .strip()
            .lower()

            ==

            "true"
        )


    else:

        correct = bool(
            correct_raw
        )


    print(
        f"[{row_index + 1:02d}/10]"
    )


    print(
        "Expected :",
        expected
    )


    print(
        "Predicted:",
        predicted
    )


    print(
        "Source   :",
        source_video
    )


    if not source_video.exists():

        raise FileNotFoundError(
            source_video
        )


    cap = cv2.VideoCapture(
        str(
            source_video
        )
    )


    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open {source_video}"
        )


    fps = float(

        cap.get(
            cv2.CAP_PROP_FPS
        )
    )


    if (
        not np.isfinite(
            fps
        )
        or
        fps <= 0
    ):

        fps = 25.0


    width = int(

        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )


    height = int(

        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )


    output_path = (

        VIDEO_OUTPUT_DIR /

        (
            f"{row_index + 1:02d}_"
            f"{expected}_"
            f"{uid}_"
            f"openvino_l1_skeleton_demo.mp4"
        )
    )


    writer = cv2.VideoWriter(

        str(
            output_path
        ),

        cv2.VideoWriter_fourcc(
            *"mp4v"
        ),

        fps,

        (
            width,
            height
        )
    )


    if not writer.isOpened():

        cap.release()

        raise RuntimeError(
            f"Could not create {output_path}"
        )


    frame_count = 0

    frames_with_hand = 0

    total_hand_detections = 0


    while True:

        ok, frame = cap.read()


        if not ok:

            break


        frame_count += 1


        # ------------------------------------------------------------------------------------------
        # EXACT WORKING L1 HAND PIPELINE
        # ------------------------------------------------------------------------------------------

        hand_results = (
            detect_hands_exact_l1(
                frame
            )
        )


        if len(
            hand_results
        ) > 0:

            frames_with_hand += 1


        total_hand_detections += len(
            hand_results
        )


        for hand_result in hand_results:

            frame = draw_hand(

                frame,

                hand_result
            )


        # ------------------------------------------------------------------------------------------
        # SMALL TOP TEXT
        # ------------------------------------------------------------------------------------------

        frame = draw_top_bar(

            frame,

            expected,

            predicted,

            confidence,

            correct
        )


        writer.write(
            frame
        )


    cap.release()

    writer.release()


    if not output_path.exists():

        raise RuntimeError(
            f"Output missing: {output_path}"
        )


    generated.append(
        output_path
    )


    detection_percent = (

        100.0 *
        frames_with_hand /
        frame_count

        if frame_count > 0

        else 0.0
    )


    print(
        "Frames:",
        frame_count
    )


    print(
        "Frames with hand:",
        f"{frames_with_hand}/{frame_count}",
        f"({detection_percent:.1f}%)"
    )


    print(
        "Total hand detections:",
        total_hand_detections
    )


    print(
        "Saved:"
    )


    print(
        output_path
    )


# ================================================================================================
# 22. VERIFY
# ================================================================================================

print("\n" + "=" * 120)
print("22. VERIFY CORRECTED OUTPUT")
print("=" * 120)


final_videos = sorted(

    VIDEO_OUTPUT_DIR.glob(
        "*.mp4"
    )
)


print(
    "Generated videos:",
    len(
        final_videos
    )
)


if len(
    final_videos
) != 10:

    raise RuntimeError(

        f"Expected exactly 10 videos, "
        f"found {len(final_videos)}"
    )


total_mb = sum(

    p.stat().st_size

    for p in final_videos

) / (
    1024 ** 2
)


print(
    f"Total size: {total_mb:.2f} MB"
)


print()
print(
    "[PASS] Exactly 10 corrected videos exist."
)


# ================================================================================================
# 23. SCIENTIFIC RESULTS REMAIN UNCHANGED
# ================================================================================================

print("\n" + "=" * 120)
print("23. L24 CLASSIFICATION RESULTS — UNCHANGED")
print("=" * 120)


correct_count = int(

    df[
        "correct"
    ]
    .astype(str)
    .str.lower()
    .eq(
        "true"
    )
    .sum()
)


mean_confidence = float(

    df[
        "confidence"
    ].mean()
)


print(
    "Videos evaluated :",
    len(df)
)


print(
    "Correct          :",
    f"{correct_count}/{len(df)}"
)


print(
    "Accuracy         :",
    f"{100 * correct_count / len(df):.3f}%"
)


print(
    "Mean confidence  :",
    f"{100 * mean_confidence:.3f}%"
)


print()
print(
    "VideoMAE rerun      : NO"
)


print(
    "Training            : NO"
)


print(
    "Retraining          : NO"
)


print(
    "Weights modified    : NO"
)


print(
    "Locked test touched : NO"
)


# ================================================================================================
# 24. DISPLAY FIRST CORRECTED VIDEO
# ================================================================================================

print("\n" + "=" * 120)
print("24. DISPLAY FIRST CORRECTED VIDEO")
print("=" * 120)


first_video = final_videos[
    0
]


print(
    first_video
)


display(

    Video(

        str(
            first_video
        ),

        embed=True,

        html_attributes=
            "controls autoplay loop muted"
    )
)


# ================================================================================================
# 25. FINAL
# ================================================================================================

print()
print("=" * 120)
print("L24 VISUAL CORRECTION COMPLETE")
print("=" * 120)


print(
    "Palm preprocessing       : EXACT L1"
)


print(
    "Letterbox removal        : EXACT L1"
)


print(
    "Weighted NMS             : EXACT L1"
)


print(
    "Palm -> rotated ROI      : EXACT L1"
)


print(
    "Perspective transform    : EXACT L1"
)


print(
    "Inverse landmark mapping : EXACT L1"
)


print(
    "21-point skeleton        : EXACT L1"
)


print(
    "Top text                 : SMALL / SINGLE LINE"
)


print(
    "L24 classification       : UNCHANGED"
)


print(
    f"L24 accuracy             : "
    f"{correct_count}/{len(df)}"
)


print(
    "Locked test              : UNTOUCHED"
)


print()
print(
    "Preview:"
)

print(
    test_preview
)


print()
print(
    "Videos:"
)

print(
    VIDEO_OUTPUT_DIR
)


print()
print("=" * 120)

L24 — CORRECTED L1-BASED HAND VISUALIZATION
Palm model:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/palm/palm_detection_full.tflite

Landmark model:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/hand_landmark/hand_landmark_full.tflite

Frozen predictions:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l24_cislr_mp4_openvino_demo/reports/l24_mp4_predictions.csv

Output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l24_cislr_mp4_openvino_demo/videos

[PASS] Required files found.

2. LOAD FROZEN L24 RESULTS
Rows: 10

[PASS] Frozen L24 results loaded.
[PASS] VideoMAE classification will NOT be rerun.

3. DELETE PREVIOUS INCORRECT VISUALIZATION VIDEOS
Videos to delete: 10
Deleting: 05_pencil_z1VPeFV_ew4_openvino_hand_visual_demo.mp4
Deleting: 04_christmas_9LDQGtFRVhI_openvino_hand_visual_demo.mp4
Deleting: 02_interest_jMWvOjMUjYk_1_openvino_hand_visual_demo.mp4
Deleting: 01_mond


L24 VISUAL CORRECTION COMPLETE
Palm preprocessing       : EXACT L1
Letterbox removal        : EXACT L1
Weighted NMS             : EXACT L1
Palm -> rotated ROI      : EXACT L1
Perspective transform    : EXACT L1
Inverse landmark mapping : EXACT L1
21-point skeleton        : EXACT L1
Top text                 : SMALL / SINGLE LINE
L24 classification       : UNCHANGED
L24 accuracy             : 10/10
Locked test              : UNTOUCHED

Preview:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l24_cislr_mp4_openvino_demo/l24_hand_geometry_preview.jpg

Videos:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l24_cislr_mp4_openvino_demo/videos



In [7]:
# ================================================================================================
# CISLR — L24 FINAL INTEGRATED RANDOM OPENVINO DEMO
#
# EVERY RUN:
#   1. Randomly selects ONE L18B validation MP4
#   2. Runs FRESH frozen L22 OpenVINO VideoMAE sign recognition
#   3. Runs FRESH OpenVINO palm detection on video frames
#   4. Runs FRESH OpenVINO 21-point hand landmark inference
#   5. Uses EXACT L1 palm -> rotated ROI -> inverse projection geometry
#   6. Draws palm box + rotated ROI + 21-point skeleton
#   7. Saves/displays annotated MP4
#
# IMPORTANT:
#   - Validation only
#   - Locked test untouched
#   - No training
#   - No retraining
#   - No recognition threshold tuning
#   - Prediction is NOT loaded from CSV
# ================================================================================================

from pathlib import Path
import random
import time
import math
import cv2
import numpy as np
import pandas as pd
import openvino as ov

from IPython.display import Video, display


# ================================================================================================
# 1. PATHS
# ================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

VIDEO_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)

VALIDATION_MANIFEST = (
    ROOT /
    "audit/l18b_10class_dataset/manifests/l18b_validation_manifest.csv"
)

VIDEOMAE_XML = (
    ROOT /
    "audit/l22_openvino_export/model/l22_cislr_videomae_10class.xml"
)

VIDEOMAE_BIN = (
    ROOT /
    "audit/l22_openvino_export/model/l22_cislr_videomae_10class.bin"
)

PALM_MODEL = (
    ROOT /
    "models/pretrained/palm/palm_detection_full.tflite"
)

HAND_MODEL = (
    ROOT /
    "models/pretrained/hand_landmark/hand_landmark_full.tflite"
)

OUTPUT_DIR = (
    ROOT /
    "audit/l24_cislr_mp4_openvino_demo/random_integrated_demo"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================================================
# 2. CONFIG
# ================================================================================================

DEVICE = "CPU"

PALM_SIZE = 192
LANDMARK_SIZE = 224

PALM_THRESHOLD = 0.50
NMS_THRESHOLD = 0.30
MAX_HANDS = 2

NUM_CACHE_FRAMES = 32
NUM_MODEL_FRAMES = 16

RESIZE_SIZE = 256
CROP_SIZE = 224


VOCAB = {
    0: "monday",
    1: "interest",
    2: "thursday",
    3: "christmas",
    4: "pencil",
    5: "mother",
    6: "tuesday",
    7: "friday",
    8: "rubber",
    9: "wash",
}


IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
).reshape(1, 1, 3)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
).reshape(1, 1, 3)


# EXACT skeleton from your L1 notebook

HAND_CONNECTIONS = [

    # Thumb
    (0,1), (1,2), (2,3), (3,4),

    # Index
    (0,5), (5,6), (6,7), (7,8),

    # Middle
    (5,9), (9,10), (10,11), (11,12),

    # Ring
    (9,13), (13,14), (14,15), (15,16),

    # Little
    (13,17), (17,18), (18,19), (19,20),

    # Palm
    (0,17)
]


print("=" * 110)
print("L24 — FINAL INTEGRATED RANDOM OPENVINO DEMO")
print("=" * 110)

print("OpenVINO:", ov.__version__)
print("Device  :", DEVICE)

print("\nModels:")
print("Palm     :", PALM_MODEL)
print("Landmark :", HAND_MODEL)
print("VideoMAE :", VIDEOMAE_XML)


# ================================================================================================
# 3. VERIFY FILES
# ================================================================================================

for path in [
    VALIDATION_MANIFEST,
    VIDEOMAE_XML,
    VIDEOMAE_BIN,
    PALM_MODEL,
    HAND_MODEL
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("\n[PASS] All required artifacts found.")


# ================================================================================================
# 4. COMPILE ALL THREE MODELS WITH OPENVINO
# ================================================================================================

print("\n" + "=" * 110)
print("COMPILE OPENVINO MODELS")
print("=" * 110)

core = ov.Core()


# ---------------- PALM ----------------

palm_model = core.read_model(
    str(PALM_MODEL)
)

palm_net = core.compile_model(
    palm_model,
    DEVICE
)


# ---------------- LANDMARK ----------------

hand_model = core.read_model(
    str(HAND_MODEL)
)

hand_net = core.compile_model(
    hand_model,
    DEVICE
)


# ---------------- VIDEOMAE ----------------

videomae_model = core.read_model(
    str(VIDEOMAE_XML)
)

videomae_net = core.compile_model(
    videomae_model,
    DEVICE
)

videomae_input = videomae_net.input(0)
videomae_output = videomae_net.output(0)


print("\nPalm input     :", palm_net.input(0).shape)
print("Landmark input :", hand_net.input(0).shape)
print("VideoMAE input :", videomae_input.shape)
print("VideoMAE output:", videomae_output.shape)

assert list(videomae_input.shape) == [
    1, 16, 3, 224, 224
]

assert list(videomae_output.shape) == [
    1, 10
]

print("\n[PASS] All 3 models compiled with OpenVINO.")


# ================================================================================================
# 5. EXACT L1 PALM ANCHORS
# ================================================================================================

def generate_anchors():

    num_layers = 4
    strides = [8, 16, 16, 16]

    anchors = []

    layer = 0

    while layer < num_layers:

        same = layer
        anchors_per_cell = 0

        while (
            same < num_layers
            and
            strides[same] == strides[layer]
        ):

            anchors_per_cell += 1
            anchors_per_cell += 1

            same += 1

        stride = strides[layer]

        fh = int(
            math.ceil(
                PALM_SIZE / stride
            )
        )

        fw = int(
            math.ceil(
                PALM_SIZE / stride
            )
        )

        for y in range(fh):

            for x in range(fw):

                cx = (
                    x + 0.5
                ) / fw

                cy = (
                    y + 0.5
                ) / fh

                for _ in range(
                    anchors_per_cell
                ):

                    anchors.append(
                        [
                            cx,
                            cy,
                            1.0,
                            1.0
                        ]
                    )

        layer = same

    return np.asarray(
        anchors,
        dtype=np.float32
    )


ANCHORS = generate_anchors()

assert ANCHORS.shape == (
    2016,
    4
)

print(
    "\n[PASS] Palm anchors:",
    ANCHORS.shape
)


# ================================================================================================
# 6. EXACT L1 PALM PREPROCESSING
# ================================================================================================

def preprocess_palm(frame):

    h, w = frame.shape[:2]

    scale = min(
        PALM_SIZE / w,
        PALM_SIZE / h
    )

    nw = int(
        round(w * scale)
    )

    nh = int(
        round(h * scale)
    )

    resized = cv2.resize(
        frame,
        (nw, nh),
        interpolation=cv2.INTER_LINEAR
    )

    rgb = cv2.cvtColor(
        resized,
        cv2.COLOR_BGR2RGB
    )

    left = (
        PALM_SIZE - nw
    ) // 2

    top = (
        PALM_SIZE - nh
    ) // 2

    right = (
        PALM_SIZE -
        nw -
        left
    )

    bottom = (
        PALM_SIZE -
        nh -
        top
    )

    padded = cv2.copyMakeBorder(
        rgb,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=(0, 0, 0)
    )

    tensor = (
        padded.astype(
            np.float32
        )
        / 255.0
    )

    shape = list(
        palm_net.input(0).shape
    )

    if shape == [
        1, 192, 192, 3
    ]:

        tensor = tensor[None]

    elif shape == [
        1, 3, 192, 192
    ]:

        tensor = np.transpose(
            tensor,
            (2, 0, 1)
        )[None]

    else:

        raise RuntimeError(
            f"Unexpected palm input: {shape}"
        )

    return tensor, {
        "scale": scale,
        "left": left,
        "top": top
    }


# ================================================================================================
# 7. PALM INFERENCE
# ================================================================================================

def palm_inference(frame):

    tensor, info = (
        preprocess_palm(frame)
    )

    start = time.perf_counter()

    result = palm_net(
        [tensor]
    )

    latency = (
        time.perf_counter()
        - start
    ) * 1000.0

    reg = None
    logits = None

    for out in palm_net.outputs:

        arr = np.asarray(
            result[out]
        )

        if arr.size == (
            2016 * 18
        ):

            reg = arr.reshape(
                2016,
                18
            )

        elif arr.size == 2016:

            logits = arr.reshape(
                2016
            )

    if (
        reg is None
        or
        logits is None
    ):

        raise RuntimeError(
            "Could not identify palm outputs."
        )

    return (
        reg,
        logits,
        info,
        latency
    )


# ================================================================================================
# 8. EXACT PALM DECODING
# ================================================================================================

def sigmoid(x):

    x = np.clip(
        x,
        -100,
        100
    )

    return 1.0 / (
        1.0 +
        np.exp(-x)
    )


def decode_palms(
    reg,
    logits
):

    scores = sigmoid(
        logits
    )

    ids = np.where(
        scores >= PALM_THRESHOLD
    )[0]

    detections = []

    for i in ids:

        r = reg[i]

        ax, ay, aw, ah = (
            ANCHORS[i]
        )

        cx = (
            r[0] / 192.0 * aw
            + ax
        )

        cy = (
            r[1] / 192.0 * ah
            + ay
        )

        bw = (
            r[2] / 192.0 * aw
        )

        bh = (
            r[3] / 192.0 * ah
        )

        box = np.array(
            [
                cx - bw / 2,
                cy - bh / 2,
                cx + bw / 2,
                cy + bh / 2
            ],
            dtype=np.float32
        )

        kp = []

        for k in range(7):

            j = (
                4 +
                2 * k
            )

            x = (
                r[j]
                / 192.0
                * aw
                + ax
            )

            y = (
                r[j + 1]
                / 192.0
                * ah
                + ay
            )

            kp.append(
                [x, y]
            )

        detections.append(
            {
                "score":
                    float(
                        scores[i]
                    ),

                "bbox":
                    box,

                "keypoints":
                    np.asarray(
                        kp,
                        dtype=np.float32
                    )
            }
        )

    detections.sort(
        key=lambda d:
            d["score"],
        reverse=True
    )

    return detections


# ================================================================================================
# 9. EXACT WEIGHTED NMS
# ================================================================================================

def IoU(a, b):

    x1 = max(
        a[0],
        b[0]
    )

    y1 = max(
        a[1],
        b[1]
    )

    x2 = min(
        a[2],
        b[2]
    )

    y2 = min(
        a[3],
        b[3]
    )

    inter = (
        max(
            0.0,
            x2 - x1
        )
        *
        max(
            0.0,
            y2 - y1
        )
    )

    aa = (
        max(
            0.0,
            a[2] - a[0]
        )
        *
        max(
            0.0,
            a[3] - a[1]
        )
    )

    ab = (
        max(
            0.0,
            b[2] - b[0]
        )
        *
        max(
            0.0,
            b[3] - b[1]
        )
    )

    union = (
        aa +
        ab -
        inter
    )

    return (
        inter / union
        if union > 0
        else 0.0
    )


def weighted_nms(
    detections
):

    remaining = (
        detections.copy()
    )

    output = []

    while remaining:

        reference = (
            remaining[0]
        )

        group = []
        rest = []

        for d in remaining:

            if IoU(
                reference["bbox"],
                d["bbox"]
            ) > NMS_THRESHOLD:

                group.append(d)

            else:

                rest.append(d)

        weights = np.asarray(
            [
                d["score"]
                for d in group
            ],
            dtype=np.float32
        )

        total = (
            weights.sum()
        )

        boxes = np.stack(
            [
                d["bbox"]
                for d in group
            ]
        )

        kps = np.stack(
            [
                d["keypoints"]
                for d in group
            ]
        )

        merged_box = (
            boxes
            * weights[:, None]
        ).sum(
            axis=0
        ) / total

        merged_kps = (
            kps
            * weights[
                :,
                None,
                None
            ]
        ).sum(
            axis=0
        ) / total

        output.append(
            {
                "score":
                    max(
                        d["score"]
                        for d in group
                    ),

                "bbox":
                    merged_box,

                "keypoints":
                    merged_kps
            }
        )

        remaining = rest

    output.sort(
        key=lambda d:
            d["score"],
        reverse=True
    )

    return output[
        :MAX_HANDS
    ]


# ================================================================================================
# 10. EXACT L1 LETTERBOX REMOVAL
# ================================================================================================

def palm_to_original(
    p,
    info
):

    x = (
        p[0] * PALM_SIZE
        - info["left"]
    ) / info["scale"]

    y = (
        p[1] * PALM_SIZE
        - info["top"]
    ) / info["scale"]

    return np.array(
        [x, y],
        dtype=np.float32
    )


def palm_detection_to_original(
    det,
    info
):

    box = det["bbox"]

    p1 = palm_to_original(
        [
            box[0],
            box[1]
        ],
        info
    )

    p2 = palm_to_original(
        [
            box[2],
            box[3]
        ],
        info
    )

    keypoints = np.stack(
        [
            palm_to_original(
                p,
                info
            )
            for p
            in det["keypoints"]
        ]
    )

    return {
        "score":
            det["score"],

        "bbox":
            np.array(
                [
                    p1[0],
                    p1[1],
                    p2[0],
                    p2[1]
                ],
                dtype=np.float32
            ),

        "keypoints":
            keypoints
    }


# ================================================================================================
# 11. EXACT L1 PALM -> HAND ROTATED ROI
# ================================================================================================

def normalize_radians(
    angle
):

    return (
        angle
        -
        2 * math.pi
        *
        math.floor(
            (
                angle +
                math.pi
            )
            /
            (
                2 *
                math.pi
            )
        )
    )


def palm_to_hand_rect(
    det
):

    x1, y1, x2, y2 = (
        det["bbox"]
    )

    box_w = (
        x2 - x1
    )

    box_h = (
        y2 - y1
    )

    cx = (
        x1 + x2
    ) / 2.0

    cy = (
        y1 + y2
    ) / 2.0


    # square_long = true

    long_side = max(
        box_w,
        box_h
    )

    rect_w = long_side
    rect_h = long_side


    # wrist = keypoint 0
    # middle MCP = keypoint 2

    wrist = (
        det["keypoints"][0]
    )

    middle = (
        det["keypoints"][2]
    )

    angle = math.atan2(
        -(
            middle[1]
            -
            wrist[1]
        ),
        middle[0]
        -
        wrist[0]
    )

    rotation = (
        normalize_radians(
            math.pi / 2.0
            -
            angle
        )
    )


    # EXACT shift_y = -0.5

    shift_y = -0.5

    x_shift = (
        -rect_h
        * shift_y
        * math.sin(rotation)
    )

    y_shift = (
        rect_h
        * shift_y
        * math.cos(rotation)
    )

    cx += x_shift
    cy += y_shift


    # EXACT scale = 2.6

    rect_w *= 2.6
    rect_h *= 2.6


    return {
        "cx": float(cx),
        "cy": float(cy),

        "width":
            float(rect_w),

        "height":
            float(rect_h),

        "rotation":
            float(rotation)
    }


# ================================================================================================
# 12. EXACT ROTATED RECT CORNERS
# ================================================================================================

def rect_corners(
    rect
):

    cx = rect["cx"]
    cy = rect["cy"]

    rw = rect["width"]
    rh = rect["height"]

    rotation = (
        rect["rotation"]
    )

    c = math.cos(
        rotation
    )

    s = math.sin(
        rotation
    )

    local = np.array(
        [
            [-rw/2, -rh/2],
            [ rw/2, -rh/2],
            [ rw/2,  rh/2],
            [-rw/2,  rh/2]
        ],
        dtype=np.float32
    )

    R = np.array(
        [
            [c, -s],
            [s,  c]
        ],
        dtype=np.float32
    )

    points = (
        local @ R.T
        +
        np.array(
            [cx, cy],
            dtype=np.float32
        )
    )

    return points.astype(
        np.float32
    )


# ================================================================================================
# 13. EXACT 224x224 PERSPECTIVE ROI
# ================================================================================================

def crop_hand_roi(
    frame,
    rect
):

    src = rect_corners(
        rect
    )

    dst = np.array(
        [
            [0, 0],

            [
                LANDMARK_SIZE - 1,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                LANDMARK_SIZE - 1
            ],

            [
                0,
                LANDMARK_SIZE - 1
            ]
        ],
        dtype=np.float32
    )

    M = cv2.getPerspectiveTransform(
        src,
        dst
    )

    M_inv = cv2.getPerspectiveTransform(
        dst,
        src
    )

    crop = cv2.warpPerspective(
        frame,
        M,
        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)
    )

    return (
        crop,
        M,
        M_inv,
        src
    )


# ================================================================================================
# 14. EXACT LANDMARK PREPROCESSING
# ================================================================================================

def preprocess_hand(
    crop
):

    rgb = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2RGB
    )

    tensor = (
        rgb.astype(
            np.float32
        )
        /
        255.0
    )

    shape = list(
        hand_net.input(0).shape
    )

    if shape == [
        1,
        224,
        224,
        3
    ]:

        tensor = tensor[
            None
        ]

    elif shape == [
        1,
        3,
        224,
        224
    ]:

        tensor = np.transpose(
            tensor,
            (2,0,1)
        )[None]

    else:

        raise RuntimeError(
            f"Unexpected landmark input: {shape}"
        )

    return tensor


# ================================================================================================
# 15. EXACT LANDMARK OUTPUT IDENTIFICATION
# ================================================================================================

def run_hand_landmark(
    crop
):

    tensor = preprocess_hand(
        crop
    )

    start = time.perf_counter()

    result = hand_net(
        [tensor]
    )

    latency = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    screen_landmarks = None
    world_landmarks = None
    scalar_outputs = []


    for i, out in enumerate(
        hand_net.outputs
    ):

        arr = np.asarray(
            result[out]
        )

        flat = arr.reshape(
            -1
        )

        if flat.size == 63:

            candidate = (
                flat.reshape(
                    21,
                    3
                )
                .astype(
                    np.float32
                )
            )

            xy_abs = float(
                np.max(
                    np.abs(
                        candidate[
                            :,
                            :2
                        ]
                    )
                )
            )

            if xy_abs > 2.0:

                screen_landmarks = (
                    candidate
                )

            else:

                world_landmarks = (
                    candidate
                )

        elif flat.size == 1:

            scalar_outputs.append(
                (
                    i,
                    float(
                        flat[0]
                    )
                )
            )


    if screen_landmarks is None:

        raise RuntimeError(
            "Could not identify 21x3 screen landmarks."
        )


    return (
        screen_landmarks,
        world_landmarks,
        scalar_outputs,
        latency
    )


# ================================================================================================
# 16. EXACT INVERSE LANDMARK PROJECTION
# ================================================================================================

def project_landmarks_back(
    landmarks,
    M_inv,
    rect
):

    xy = (
        landmarks[
            :,
            :2
        ]
        .astype(
            np.float32
        )
    )

    pts = xy.reshape(
        -1,
        1,
        2
    )

    projected = (
        cv2.perspectiveTransform(
            pts,
            M_inv
        )
        .reshape(
            -1,
            2
        )
    )

    z = (
        landmarks[:,2]
        /
        LANDMARK_SIZE
        *
        rect["width"]
    )

    return np.column_stack(
        [
            projected,
            z
        ]
    ).astype(
        np.float32
    )


# ================================================================================================
# 17. COMPLETE OPENVINO HAND PIPELINE FOR ONE FRAME
# ================================================================================================

def detect_hands(
    frame
):

    reg, logits, info, palm_ms = (
        palm_inference(
            frame
        )
    )

    decoded = decode_palms(
        reg,
        logits
    )

    filtered = weighted_nms(
        decoded
    )

    palms = [
        palm_detection_to_original(
            d,
            info
        )
        for d in filtered
    ]

    results = []

    total_landmark_ms = 0.0


    for palm in palms:

        try:

            rect = palm_to_hand_rect(
                palm
            )

            (
                crop,
                M,
                M_inv,
                corners
            ) = crop_hand_roi(
                frame,
                rect
            )

            (
                lm_crop,
                world,
                scalars,
                lm_ms
            ) = run_hand_landmark(
                crop
            )

            lm_image = (
                project_landmarks_back(
                    lm_crop,
                    M_inv,
                    rect
                )
            )

            total_landmark_ms += (
                lm_ms
            )

            results.append(
                {
                    "palm":
                        palm,

                    "rect":
                        rect,

                    "corners":
                        corners,

                    "landmarks":
                        lm_image,

                    "scalars":
                        scalars
                }
            )

        except Exception as e:

            # A single failed ROI must not kill video rendering.
            continue


    return (
        results,
        palm_ms,
        total_landmark_ms
    )


# ================================================================================================
# 18. DRAW PALM + ROI + SKELETON
# ================================================================================================

def draw_hand(
    frame,
    result
):

    H, W = frame.shape[:2]

    palm = result[
        "palm"
    ]

    lm = result[
        "landmarks"
    ]

    corners = np.round(
        result["corners"]
    ).astype(
        np.int32
    )


    # --------------------------------------------------------------------------------------------
    # PALM BOUNDING BOX — YELLOW
    # --------------------------------------------------------------------------------------------

    x1, y1, x2, y2 = (
        palm["bbox"]
    )

    x1 = int(round(x1))
    y1 = int(round(y1))

    x2 = int(round(x2))
    y2 = int(round(y2))


    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        (0, 255, 255),
        2,
        cv2.LINE_AA
    )


    # --------------------------------------------------------------------------------------------
    # ROTATED HAND ROI — THIN CYAN
    # --------------------------------------------------------------------------------------------

    cv2.polylines(
        frame,
        [corners],
        True,
        (255, 255, 0),
        1,
        cv2.LINE_AA
    )


    # --------------------------------------------------------------------------------------------
    # GREEN SKELETON
    # --------------------------------------------------------------------------------------------

    for a, b in HAND_CONNECTIONS:

        x_a = int(
            round(
                lm[a,0]
            )
        )

        y_a = int(
            round(
                lm[a,1]
            )
        )

        x_b = int(
            round(
                lm[b,0]
            )
        )

        y_b = int(
            round(
                lm[b,1]
            )
        )


        valid_a = (
            -W <= x_a <= 2*W
            and
            -H <= y_a <= 2*H
        )

        valid_b = (
            -W <= x_b <= 2*W
            and
            -H <= y_b <= 2*H
        )


        if valid_a and valid_b:

            cv2.line(
                frame,
                (x_a, y_a),
                (x_b, y_b),
                (0,255,0),
                2,
                cv2.LINE_AA
            )


    # --------------------------------------------------------------------------------------------
    # 21 LANDMARKS — RED
    # --------------------------------------------------------------------------------------------

    for j in range(21):

        x = int(
            round(
                lm[j,0]
            )
        )

        y = int(
            round(
                lm[j,1]
            )
        )

        if (
            0 <= x < W
            and
            0 <= y < H
        ):

            radius = (
                5
                if j == 0
                else 3
            )

            # tiny white outline

            cv2.circle(
                frame,
                (x,y),
                radius + 1,
                (255,255,255),
                -1,
                cv2.LINE_AA
            )

            cv2.circle(
                frame,
                (x,y),
                radius,
                (0,0,255),
                -1,
                cv2.LINE_AA
            )


    return frame


# ================================================================================================
# 19. RANDOMLY SELECT ONE VALIDATION VIDEO
# ================================================================================================

df = pd.read_csv(
    VALIDATION_MANIFEST
)

if len(df) != 10:

    raise RuntimeError(
        f"Expected 10 validation videos; got {len(df)}"
    )


random_index = (
    random.SystemRandom()
    .randrange(
        len(df)
    )
)

row = df.iloc[
    random_index
]


UID = str(
    row["uid"]
)


# Robustly identify gloss column

if "gloss" in row.index:

    EXPECTED_GLOSS = str(
        row["gloss"]
    )

elif "feature_gloss" in row.index:

    EXPECTED_GLOSS = str(
        row["feature_gloss"]
    )

else:

    raise RuntimeError(
        "Cannot find gloss column in validation manifest."
    )


EXPECTED_GLOSS = (
    EXPECTED_GLOSS
    .strip()
    .lower()
)


reverse_vocab = {
    v: k
    for k,v
    in VOCAB.items()
}


if EXPECTED_GLOSS not in reverse_vocab:

    raise RuntimeError(
        f"Unexpected gloss: {EXPECTED_GLOSS}"
    )


EXPECTED_ID = (
    reverse_vocab[
        EXPECTED_GLOSS
    ]
)


# Resolve video

if (
    "video_path" in row.index
    and
    pd.notna(
        row["video_path"]
    )
):

    VIDEO_PATH = Path(
        str(
            row["video_path"]
        )
    )

else:

    VIDEO_PATH = (
        VIDEO_ROOT /
        f"{UID}.mp4"
    )


if not VIDEO_PATH.exists():

    VIDEO_PATH = (
        VIDEO_ROOT /
        f"{UID}.mp4"
    )


if not VIDEO_PATH.exists():

    raise FileNotFoundError(
        VIDEO_PATH
    )


print("\n" + "=" * 110)
print("RANDOM VIDEO")
print("=" * 110)

print(
    "Random index:",
    random_index
)

print(
    "UID         :",
    UID
)

print(
    "Expected    :",
    EXPECTED_ID,
    "->",
    EXPECTED_GLOSS
)

print(
    "Video       :",
    VIDEO_PATH
)


# ================================================================================================
# 20. READ ENTIRE ORIGINAL VIDEO
# ================================================================================================

cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)

if not cap.isOpened():

    raise RuntimeError(
        f"Cannot open {VIDEO_PATH}"
    )


fps = float(
    cap.get(
        cv2.CAP_PROP_FPS
    )
)

width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)


if (
    not np.isfinite(fps)
    or
    fps <= 0
):

    fps = 25.0


frames = []


while True:

    ok, frame = (
        cap.read()
    )

    if not ok:
        break

    frames.append(
        frame
    )


cap.release()


if not frames:

    raise RuntimeError(
        "No frames decoded."
    )


print(
    "\nDecoded frames:",
    len(frames)
)

print(
    "Resolution:",
    width,
    "x",
    height
)

print(
    "FPS:",
    fps
)


# ================================================================================================
# 21. EXACT L19 TEMPORAL SAMPLING
# ================================================================================================

def deterministic_indices(
    total,
    count
):

    return np.round(
        np.linspace(
            0,
            total - 1,
            count
        )
    ).astype(
        np.int64
    )


idx32 = deterministic_indices(
    len(frames),
    32
)

frames32 = [
    frames[i]
    for i in idx32
]


idx16 = deterministic_indices(
    len(frames32),
    16
)

frames16 = [
    frames32[i]
    for i in idx16
]


# ================================================================================================
# 22. EXACT L19 VIDEOMAE PREPROCESSING
# ================================================================================================

def preprocess_videomae_frame(
    frame
):

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    resized = cv2.resize(
        rgb,
        (
            RESIZE_SIZE,
            RESIZE_SIZE
        ),
        interpolation=cv2.INTER_LINEAR
    )

    offset = (
        RESIZE_SIZE -
        CROP_SIZE
    ) // 2

    crop = resized[
        offset:
        offset + CROP_SIZE,

        offset:
        offset + CROP_SIZE
    ]

    x = (
        crop.astype(
            np.float32
        )
        / 255.0
    )

    x = (
        x -
        IMAGENET_MEAN
    ) / IMAGENET_STD

    x = np.transpose(
        x,
        (2,0,1)
    )

    return np.ascontiguousarray(
        x,
        dtype=np.float32
    )


video_tensor = np.stack(
    [
        preprocess_videomae_frame(
            f
        )
        for f in frames16
    ],
    axis=0
)[None]


video_tensor = (
    np.ascontiguousarray(
        video_tensor,
        dtype=np.float32
    )
)


assert video_tensor.shape == (
    1,
    16,
    3,
    224,
    224
)


print(
    "\nVideoMAE tensor:",
    video_tensor.shape
)


# ================================================================================================
# 23. FRESH OPENVINO VIDEOMAE SIGN RECOGNITION
# ================================================================================================

# Warmup

_ = videomae_net(
    {
        videomae_input:
            video_tensor
    }
)


start = time.perf_counter()


ov_result = videomae_net(
    {
        videomae_input:
            video_tensor
    }
)


videomae_ms = (
    time.perf_counter()
    -
    start
) * 1000.0


logits = np.asarray(
    ov_result[
        videomae_output
    ]
).reshape(
    -1
)


shifted = (
    logits -
    np.max(logits)
)

exp_logits = np.exp(
    shifted
)

probabilities = (
    exp_logits /
    exp_logits.sum()
)


PREDICTED_ID = int(
    np.argmax(
        probabilities
    )
)

PREDICTED_GLOSS = (
    VOCAB[
        PREDICTED_ID
    ]
)

CONFIDENCE = float(
    probabilities[
        PREDICTED_ID
    ]
)

CORRECT = (
    PREDICTED_ID
    ==
    EXPECTED_ID
)


print("\n" + "=" * 110)
print("FRESH OPENVINO VIDEOMAE RESULT")
print("=" * 110)

print(
    "Expected :",
    EXPECTED_GLOSS
)

print(
    "Predicted:",
    PREDICTED_GLOSS
)

print(
    "Confidence:",
    f"{CONFIDENCE*100:.3f}%"
)

print(
    "Correct:",
    CORRECT
)

print(
    "VideoMAE inference:",
    f"{videomae_ms:.3f} ms"
)


# ================================================================================================
# 24. OUTPUT VIDEO
#
# Only previous RANDOM demo is deleted.
# Frozen historical L24 artifacts are not touched.
# ================================================================================================

for old in OUTPUT_DIR.glob(
    "*.mp4"
):

    old.unlink()


output_path = (
    OUTPUT_DIR /
    (
        f"random_{EXPECTED_GLOSS}_"
        f"{UID}_"
        f"pred_{PREDICTED_GLOSS}.mp4"
    )
)


writer = cv2.VideoWriter(
    str(output_path),
    cv2.VideoWriter_fourcc(
        *"mp4v"
    ),
    fps,
    (
        width,
        height
    )
)


if not writer.isOpened():

    raise RuntimeError(
        "Could not create output MP4."
    )


# ================================================================================================
# 25. RUN OPENVINO PALM + LANDMARK ON EVERY FRAME
# ================================================================================================

print("\n" + "=" * 110)
print("RUNNING OPENVINO PALM + LANDMARK ON VIDEO")
print("=" * 110)


total_palms = 0
frames_with_hands = 0

palm_times = []
landmark_times = []


for frame_number, original_frame in enumerate(
    frames
):

    annotated = (
        original_frame.copy()
    )


    # --------------------------------------------------------------------------------------------
    # Fresh OpenVINO palm + landmark inference
    # --------------------------------------------------------------------------------------------

    try:

        (
            hand_results,
            palm_ms,
            landmark_ms
        ) = detect_hands(
            original_frame
        )

    except Exception:

        hand_results = []
        palm_ms = np.nan
        landmark_ms = np.nan


    if np.isfinite(
        palm_ms
    ):

        palm_times.append(
            palm_ms
        )


    if np.isfinite(
        landmark_ms
    ):

        landmark_times.append(
            landmark_ms
        )


    if hand_results:

        frames_with_hands += 1

        total_palms += len(
            hand_results
        )


    # --------------------------------------------------------------------------------------------
    # Draw hands
    # --------------------------------------------------------------------------------------------

    for result in hand_results:

        annotated = draw_hand(
            annotated,
            result
        )


    # --------------------------------------------------------------------------------------------
    # Compact top bar
    # --------------------------------------------------------------------------------------------

    overlay = (
        annotated.copy()
    )

    bar_height = min(
        36,
        height
    )

    cv2.rectangle(
        overlay,
        (0,0),
        (
            width,
            bar_height
        ),
        (0,0,0),
        -1
    )

    cv2.addWeighted(
        overlay,
        0.60,
        annotated,
        0.40,
        0,
        annotated
    )


    status = (
        "PASS"
        if CORRECT
        else "FAIL"
    )


    text = (
        f"Expected: {EXPECTED_GLOSS} | "
        f"Pred: {PREDICTED_GLOSS} | "
        f"{CONFIDENCE*100:.1f}% | "
        f"{status} | OpenVINO"
    )


    cv2.putText(
        annotated,
        text,
        (7,23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.32,
        (255,255,255),
        1,
        cv2.LINE_AA
    )


    writer.write(
        annotated
    )


    # Progress

    if (
        frame_number + 1
    ) % 30 == 0:

        print(
            f"\rProcessed "
            f"{frame_number+1}/{len(frames)} frames",
            end=""
        )


writer.release()


print(
    f"\rProcessed {len(frames)}/{len(frames)} frames"
)


# ================================================================================================
# 26. SUMMARY
# ================================================================================================

detection_rate = (
    frames_with_hands
    /
    len(frames)
)


mean_palm_ms = (
    float(
        np.mean(
            palm_times
        )
    )
    if palm_times
    else np.nan
)


mean_landmark_ms = (
    float(
        np.mean(
            landmark_times
        )
    )
    if landmark_times
    else np.nan
)


print("\n" + "=" * 110)
print("L24 RANDOM INTEGRATED DEMO COMPLETE")
print("=" * 110)


print(
    "\nRandom video:",
    UID
)

print(
    "Expected:",
    EXPECTED_GLOSS
)

print(
    "Predicted:",
    PREDICTED_GLOSS
)

print(
    "Confidence:",
    f"{CONFIDENCE*100:.3f}%"
)

print(
    "Correct:",
    CORRECT
)


print("\nOPENVINO:")

print(
    "VideoMAE inference:",
    f"{videomae_ms:.3f} ms"
)

print(
    "Mean palm inference:",
    f"{mean_palm_ms:.3f} ms"
)

print(
    "Mean landmark inference:",
    f"{mean_landmark_ms:.3f} ms"
)


print("\nHAND VISUALIZATION:")

print(
    "Frames:",
    len(frames)
)

print(
    "Frames with hand:",
    frames_with_hands
)

print(
    "Detection rate:",
    f"{detection_rate*100:.2f}%"
)

print(
    "Detected hand instances:",
    total_palms
)


print("\nINTEGRITY:")

print(
    "Random validation selection : YES"
)

print(
    "Fresh VideoMAE inference    : YES"
)

print(
    "Fresh palm inference        : YES"
)

print(
    "Fresh landmark inference    : YES"
)

print(
    "Prediction loaded from CSV  : NO"
)

print(
    "Training                    : NO"
)

print(
    "Retraining                  : NO"
)

print(
    "Locked test loaded          : NO"
)

print(
    "Locked test evaluated       : NO"
)


print("\nSaved video:")

print(
    output_path
)

print(
    "Size:",
    f"{output_path.stat().st_size/(1024**2):.2f} MB"
)


# ================================================================================================
# 27. DISPLAY VIDEO IN JUPYTER
# ================================================================================================

display(
    Video(
        str(output_path),
        embed=True,
        html_attributes=
            "controls autoplay loop muted"
    )
)

L24 — FINAL INTEGRATED RANDOM OPENVINO DEMO
OpenVINO: 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Device  : CPU

Models:
Palm     : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/palm/palm_detection_full.tflite
Landmark : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/hand_landmark/hand_landmark_full.tflite
VideoMAE : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l22_openvino_export/model/l22_cislr_videomae_10class.xml

[PASS] All required artifacts found.

COMPILE OPENVINO MODELS

Palm input     : [1,192,192,3]
Landmark input : [1,224,224,3]
VideoMAE input : [1,16,3,224,224]
VideoMAE output: [1,10]

[PASS] All 3 models compiled with OpenVINO.

[PASS] Palm anchors: (2016, 4)

RANDOM VIDEO
Random index: 6
UID         : UlQ5cTjCqT8_1
Expected    : 6 -> tuesday
Video       : /home/dipshikha/Music/cao/CISLR_v1.5-a_videos/UlQ5cTjCqT8_1.mp4

Decoded frames: 75
Resolution: 300 x 300
FPS: 30.0

VideoMAE tens

In [8]:
# ==================================================================================================
# CISLR — L25
# LIVE WEBCAM ACQUISITION + OPENVINO PALM DETECTION + 21-POINT HAND LANDMARKS
# ==================================================================================================
#
# PURPOSE
# -------
# L25 validates the LIVE input side of the final sign-language interpreter.
#
# Pipeline:
#
#   Webcam
#      ↓
#   OpenCV capture
#      ↓
#   OpenVINO palm detector
#      ↓
#   Exact L1 palm decoding + weighted NMS
#      ↓
#   Exact L1 MediaPipe-style rotated hand ROI
#      ↓
#   OpenVINO 21-point hand landmark model
#      ↓
#   Exact inverse perspective projection
#      ↓
#   Live palm box + ROI + skeleton
#
# IMPORTANT
# ---------
# - NO VideoMAE classification here. That is L26.
# - NO training.
# - NO retraining.
# - NO validation/test evaluation.
# - NO locked test.
# - Live webcam only.
#
# Controls:
#   Q / ESC = quit
#   S       = save current frame
#
# ==================================================================================================

from pathlib import Path
import math
import time
import json
import hashlib
import platform

import cv2
import numpy as np
import pandas as pd
import openvino as ov


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

PALM_MODEL = (
    ROOT /
    "models/pretrained/palm/palm_detection_full.tflite"
)

HAND_MODEL = (
    ROOT /
    "models/pretrained/hand_landmark/hand_landmark_full.tflite"
)

L25_ROOT = (
    ROOT /
    "audit/l25_webcam_acquisition"
)

REPORT_DIR = (
    L25_ROOT /
    "reports"
)

SNAPSHOT_DIR = (
    L25_ROOT /
    "snapshots"
)

CONFIG_DIR = (
    L25_ROOT /
    "config"
)

for directory in [
    L25_ROOT,
    REPORT_DIR,
    SNAPSHOT_DIR,
    CONFIG_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 2. CONFIGURATION
# ==================================================================================================

DEVICE = "CPU"

CAMERA_ID = 0

# Request webcam resolution.
# The camera may return a different resolution.
REQUEST_WIDTH = 640
REQUEST_HEIGHT = 480

PALM_SIZE = 192
LANDMARK_SIZE = 224

PALM_THRESHOLD = 0.50
NMS_THRESHOLD = 0.30

MAX_HANDS = 2


# Exact L1 skeleton topology.

HAND_CONNECTIONS = [

    # Thumb
    (0,1), (1,2), (2,3), (3,4),

    # Index
    (0,5), (5,6), (6,7), (7,8),

    # Middle
    (5,9), (9,10), (10,11), (11,12),

    # Ring
    (9,13), (13,14), (14,15), (15,16),

    # Little finger
    (13,17), (17,18), (18,19), (19,20),

    # Palm
    (0,17)
]


print("=" * 110)
print("CISLR — L25 LIVE WEBCAM ACQUISITION")
print("=" * 110)

print()
print("OpenVINO version :", ov.__version__)
print("OpenCV version   :", cv2.__version__)
print("Device           :", DEVICE)
print("Camera ID        :", CAMERA_ID)

print()
print("Palm model:")
print(PALM_MODEL)

print()
print("Landmark model:")
print(HAND_MODEL)


# ==================================================================================================
# 3. VERIFY MODELS
# ==================================================================================================

for path in [
    PALM_MODEL,
    HAND_MODEL
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing model:\n{path}"
        )


print()
print("[PASS] Palm and landmark models found.")


# ==================================================================================================
# 4. OPENVINO INITIALIZATION
# ==================================================================================================

core = ov.Core()

print()
print("Available OpenVINO devices:")

for device in core.available_devices:
    print("  ", device)


if DEVICE not in core.available_devices:

    raise RuntimeError(
        f"{DEVICE} not available."
    )


# ==================================================================================================
# 5. LOAD PALM MODEL
# ==================================================================================================

palm_model = core.read_model(
    str(PALM_MODEL)
)

palm_net = core.compile_model(
    palm_model,
    DEVICE
)

palm_input = palm_net.input(0)


print()
print("=" * 110)
print("PALM MODEL")
print("=" * 110)

print(
    "Input shape:",
    list(
        palm_input.shape
    )
)

print("Outputs:")

for i, out in enumerate(
    palm_net.outputs
):

    print(
        f"  {i}:",
        list(
            out.shape
        )
    )


# ==================================================================================================
# 6. LOAD LANDMARK MODEL
# ==================================================================================================

hand_model = core.read_model(
    str(HAND_MODEL)
)

hand_net = core.compile_model(
    hand_model,
    DEVICE
)

hand_input = hand_net.input(0)


print()
print("=" * 110)
print("LANDMARK MODEL")
print("=" * 110)

print(
    "Input shape:",
    list(
        hand_input.shape
    )
)

print("Outputs:")

for i, out in enumerate(
    hand_net.outputs
):

    print(
        f"  {i}:",
        list(
            out.shape
        )
    )


print()
print("[PASS] OpenVINO palm + landmark models compiled.")


# ==================================================================================================
# 7. EXACT L1 PALM ANCHORS
# ==================================================================================================

def generate_anchors():

    num_layers = 4

    strides = [
        8,
        16,
        16,
        16
    ]

    anchors = []

    layer = 0

    while layer < num_layers:

        same = layer

        anchors_per_cell = 0

        while (
            same < num_layers
            and
            strides[same] == strides[layer]
        ):

            # Exact L1:
            # two anchors for each occurrence
            # of the same stride layer.

            anchors_per_cell += 2

            same += 1


        stride = strides[layer]

        feature_h = int(
            math.ceil(
                PALM_SIZE /
                stride
            )
        )

        feature_w = int(
            math.ceil(
                PALM_SIZE /
                stride
            )
        )


        for y in range(
            feature_h
        ):

            for x in range(
                feature_w
            ):

                cx = (
                    x + 0.5
                ) / feature_w

                cy = (
                    y + 0.5
                ) / feature_h


                for _ in range(
                    anchors_per_cell
                ):

                    anchors.append(
                        [
                            cx,
                            cy,
                            1.0,
                            1.0
                        ]
                    )


        layer = same


    return np.asarray(
        anchors,
        dtype=np.float32
    )


ANCHORS = generate_anchors()


if ANCHORS.shape != (
    2016,
    4
):

    raise RuntimeError(
        f"Wrong anchor shape: {ANCHORS.shape}"
    )


print()
print(
    "[PASS] Palm anchors:",
    ANCHORS.shape
)


# ==================================================================================================
# 8. PALM PREPROCESSING
# ==================================================================================================

def preprocess_palm(
    frame
):

    h, w = frame.shape[:2]


    # Preserve aspect ratio.

    scale = min(
        PALM_SIZE / w,
        PALM_SIZE / h
    )


    new_w = int(
        round(
            w * scale
        )
    )

    new_h = int(
        round(
            h * scale
        )
    )


    resized = cv2.resize(
        frame,
        (
            new_w,
            new_h
        ),
        interpolation=cv2.INTER_LINEAR
    )


    # BGR -> RGB

    rgb = cv2.cvtColor(
        resized,
        cv2.COLOR_BGR2RGB
    )


    # Letterbox.

    left = (
        PALM_SIZE -
        new_w
    ) // 2

    top = (
        PALM_SIZE -
        new_h
    ) // 2

    right = (
        PALM_SIZE -
        new_w -
        left
    )

    bottom = (
        PALM_SIZE -
        new_h -
        top
    )


    padded = cv2.copyMakeBorder(
        rgb,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=(
            0,
            0,
            0
        )
    )


    # Exact L1 normalization:
    # [0,255] -> [0,1]

    tensor = (
        padded.astype(
            np.float32
        )
        /
        255.0
    )


    input_shape = list(
        palm_input.shape
    )


    if input_shape == [
        1,
        192,
        192,
        3
    ]:

        tensor = tensor[
            None
        ]


    elif input_shape == [
        1,
        3,
        192,
        192
    ]:

        tensor = np.transpose(
            tensor,
            (
                2,
                0,
                1
            )
        )[None]


    else:

        raise RuntimeError(
            "Unexpected palm model input "
            f"shape: {input_shape}"
        )


    info = {

        "scale":
            float(scale),

        "left":
            int(left),

        "top":
            int(top),

        "original_width":
            int(w),

        "original_height":
            int(h)
    }


    return (
        np.ascontiguousarray(
            tensor,
            dtype=np.float32
        ),
        info
    )


# ==================================================================================================
# 9. SIGMOID
# ==================================================================================================

def sigmoid(
    x
):

    x = np.clip(
        x,
        -100,
        100
    )

    return (
        1.0 /
        (
            1.0 +
            np.exp(
                -x
            )
        )
    )


# ==================================================================================================
# 10. PALM INFERENCE
# ==================================================================================================

def run_palm(
    frame
):

    tensor, info = (
        preprocess_palm(
            frame
        )
    )


    start = (
        time.perf_counter()
    )


    result = palm_net(
        [tensor]
    )


    latency_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    regression = None
    logits = None


    for output in palm_net.outputs:

        arr = np.asarray(
            result[
                output
            ]
        )


        if arr.size == (
            2016 * 18
        ):

            regression = (
                arr.reshape(
                    2016,
                    18
                )
            )


        elif arr.size == 2016:

            logits = (
                arr.reshape(
                    2016
                )
            )


    if regression is None:

        raise RuntimeError(
            "Palm regression output not found."
        )


    if logits is None:

        raise RuntimeError(
            "Palm score output not found."
        )


    return (
        regression,
        logits,
        info,
        latency_ms
    )


# ==================================================================================================
# 11. DECODE PALM OUTPUT
# ==================================================================================================

def decode_palms(
    regression,
    logits
):

    scores = sigmoid(
        logits
    )


    selected = np.where(
        scores >=
        PALM_THRESHOLD
    )[0]


    detections = []


    for index in selected:

        r = regression[
            index
        ]

        ax, ay, aw, ah = (
            ANCHORS[
                index
            ]
        )


        cx = (
            r[0] /
            PALM_SIZE *
            aw +
            ax
        )

        cy = (
            r[1] /
            PALM_SIZE *
            ah +
            ay
        )

        bw = (
            r[2] /
            PALM_SIZE *
            aw
        )

        bh = (
            r[3] /
            PALM_SIZE *
            ah
        )


        bbox = np.array(
            [
                cx -
                bw / 2.0,

                cy -
                bh / 2.0,

                cx +
                bw / 2.0,

                cy +
                bh / 2.0
            ],
            dtype=np.float32
        )


        keypoints = []


        for k in range(7):

            offset = (
                4 +
                2 * k
            )


            x = (
                r[offset] /
                PALM_SIZE *
                aw +
                ax
            )

            y = (
                r[offset + 1] /
                PALM_SIZE *
                ah +
                ay
            )


            keypoints.append(
                [
                    x,
                    y
                ]
            )


        detections.append(
            {

                "score":
                    float(
                        scores[
                            index
                        ]
                    ),

                "bbox":
                    bbox,

                "keypoints":
                    np.asarray(
                        keypoints,
                        dtype=np.float32
                    )
            }
        )


    detections.sort(
        key=lambda d:
            d["score"],
        reverse=True
    )


    return detections


# ==================================================================================================
# 12. IoU
# ==================================================================================================

def bbox_iou(
    a,
    b
):

    x1 = max(
        a[0],
        b[0]
    )

    y1 = max(
        a[1],
        b[1]
    )

    x2 = min(
        a[2],
        b[2]
    )

    y2 = min(
        a[3],
        b[3]
    )


    intersection = (
        max(
            0.0,
            x2 - x1
        )
        *
        max(
            0.0,
            y2 - y1
        )
    )


    area_a = (
        max(
            0.0,
            a[2] -
            a[0]
        )
        *
        max(
            0.0,
            a[3] -
            a[1]
        )
    )


    area_b = (
        max(
            0.0,
            b[2] -
            b[0]
        )
        *
        max(
            0.0,
            b[3] -
            b[1]
        )
    )


    union = (
        area_a +
        area_b -
        intersection
    )


    if union <= 0:

        return 0.0


    return (
        intersection /
        union
    )


# ==================================================================================================
# 13. EXACT WEIGHTED NMS
# ==================================================================================================

def weighted_nms(
    detections
):

    remaining = list(
        detections
    )

    output = []


    while remaining:

        reference = (
            remaining[0]
        )

        group = []
        rest = []


        for detection in remaining:

            if bbox_iou(
                reference["bbox"],
                detection["bbox"]
            ) > NMS_THRESHOLD:

                group.append(
                    detection
                )

            else:

                rest.append(
                    detection
                )


        weights = np.asarray(
            [
                d["score"]
                for d
                in group
            ],
            dtype=np.float32
        )


        total_weight = float(
            weights.sum()
        )


        if total_weight <= 0:

            remaining = rest
            continue


        boxes = np.stack(
            [
                d["bbox"]
                for d
                in group
            ]
        )


        keypoints = np.stack(
            [
                d["keypoints"]
                for d
                in group
            ]
        )


        merged_box = (
            boxes *
            weights[:, None]
        ).sum(
            axis=0
        ) / total_weight


        merged_keypoints = (
            keypoints *
            weights[
                :,
                None,
                None
            ]
        ).sum(
            axis=0
        ) / total_weight


        output.append(
            {

                "score":
                    float(
                        max(
                            d["score"]
                            for d
                            in group
                        )
                    ),

                "bbox":
                    merged_box.astype(
                        np.float32
                    ),

                "keypoints":
                    merged_keypoints.astype(
                        np.float32
                    )
            }
        )


        remaining = rest


    output.sort(
        key=lambda d:
            d["score"],
        reverse=True
    )


    return output[
        :MAX_HANDS
    ]


# ==================================================================================================
# 14. EXACT LETTERBOX REMOVAL
# ==================================================================================================

def point_to_original(
    point,
    info
):

    x = (
        point[0] *
        PALM_SIZE
        -
        info["left"]
    ) / info["scale"]


    y = (
        point[1] *
        PALM_SIZE
        -
        info["top"]
    ) / info["scale"]


    return np.array(
        [
            x,
            y
        ],
        dtype=np.float32
    )


def detection_to_original(
    detection,
    info
):

    bbox = (
        detection[
            "bbox"
        ]
    )


    top_left = (
        point_to_original(
            [
                bbox[0],
                bbox[1]
            ],
            info
        )
    )


    bottom_right = (
        point_to_original(
            [
                bbox[2],
                bbox[3]
            ],
            info
        )
    )


    keypoints = np.stack(
        [
            point_to_original(
                point,
                info
            )

            for point
            in detection[
                "keypoints"
            ]
        ]
    )


    return {

        "score":
            detection[
                "score"
            ],

        "bbox":
            np.array(
                [
                    top_left[0],
                    top_left[1],
                    bottom_right[0],
                    bottom_right[1]
                ],
                dtype=np.float32
            ),

        "keypoints":
            keypoints.astype(
                np.float32
            )
    }


# ==================================================================================================
# 15. ANGLE NORMALIZATION
# ==================================================================================================

def normalize_radians(
    angle
):

    return (
        angle
        -
        2.0 *
        math.pi *
        math.floor(
            (
                angle +
                math.pi
            )
            /
            (
                2.0 *
                math.pi
            )
        )
    )


# ==================================================================================================
# 16. EXACT L1 PALM -> ROTATED HAND ROI
# ==================================================================================================

def palm_to_hand_rect(
    detection
):

    x1, y1, x2, y2 = (
        detection[
            "bbox"
        ]
    )


    box_width = (
        x2 - x1
    )

    box_height = (
        y2 - y1
    )


    center_x = (
        x1 + x2
    ) / 2.0

    center_y = (
        y1 + y2
    ) / 2.0


    # square_long

    long_side = max(
        box_width,
        box_height
    )


    rect_width = (
        long_side
    )

    rect_height = (
        long_side
    )


    # Exact L1 keypoints:
    # wrist = 0
    # middle-finger MCP = 2

    wrist = (
        detection[
            "keypoints"
        ][0]
    )


    middle = (
        detection[
            "keypoints"
        ][2]
    )


    angle = math.atan2(

        -(
            middle[1] -
            wrist[1]
        ),

        middle[0] -
        wrist[0]
    )


    rotation = (
        normalize_radians(

            math.pi /
            2.0
            -
            angle
        )
    )


    # Exact MediaPipe-style shift.

    shift_y = -0.5


    x_shift = (
        -rect_height *
        shift_y *
        math.sin(
            rotation
        )
    )


    y_shift = (
        rect_height *
        shift_y *
        math.cos(
            rotation
        )
    )


    center_x += (
        x_shift
    )

    center_y += (
        y_shift
    )


    # Exact L1 scaling.

    rect_width *= 2.6

    rect_height *= 2.6


    return {

        "cx":
            float(
                center_x
            ),

        "cy":
            float(
                center_y
            ),

        "width":
            float(
                rect_width
            ),

        "height":
            float(
                rect_height
            ),

        "rotation":
            float(
                rotation
            )
    }


# ==================================================================================================
# 17. ROTATED RECT CORNERS
# ==================================================================================================

def rect_corners(
    rect
):

    cx = rect["cx"]
    cy = rect["cy"]

    width = rect[
        "width"
    ]

    height = rect[
        "height"
    ]

    rotation = rect[
        "rotation"
    ]


    cos_r = math.cos(
        rotation
    )

    sin_r = math.sin(
        rotation
    )


    local = np.array(
        [
            [
                -width / 2,
                -height / 2
            ],

            [
                width / 2,
                -height / 2
            ],

            [
                width / 2,
                height / 2
            ],

            [
                -width / 2,
                height / 2
            ]
        ],
        dtype=np.float32
    )


    rotation_matrix = np.array(
        [
            [
                cos_r,
                -sin_r
            ],

            [
                sin_r,
                cos_r
            ]
        ],
        dtype=np.float32
    )


    corners = (
        local @
        rotation_matrix.T
    )


    corners += np.array(
        [
            cx,
            cy
        ],
        dtype=np.float32
    )


    return corners.astype(
        np.float32
    )


# ==================================================================================================
# 18. EXACT PERSPECTIVE HAND CROP
# ==================================================================================================

def crop_hand_roi(
    frame,
    rect
):

    source = (
        rect_corners(
            rect
        )
    )


    destination = np.array(
        [
            [
                0,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                0
            ],

            [
                LANDMARK_SIZE - 1,
                LANDMARK_SIZE - 1
            ],

            [
                0,
                LANDMARK_SIZE - 1
            ]
        ],
        dtype=np.float32
    )


    matrix = (
        cv2.getPerspectiveTransform(
            source,
            destination
        )
    )


    inverse_matrix = (
        cv2.getPerspectiveTransform(
            destination,
            source
        )
    )


    crop = cv2.warpPerspective(
        frame,
        matrix,
        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(
            0,
            0,
            0
        )
    )


    return (
        crop,
        matrix,
        inverse_matrix,
        source
    )


# ==================================================================================================
# 19. LANDMARK PREPROCESSING
# ==================================================================================================

def preprocess_hand(
    crop
):

    rgb = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2RGB
    )


    tensor = (
        rgb.astype(
            np.float32
        )
        /
        255.0
    )


    input_shape = list(
        hand_input.shape
    )


    if input_shape == [
        1,
        224,
        224,
        3
    ]:

        tensor = tensor[
            None
        ]


    elif input_shape == [
        1,
        3,
        224,
        224
    ]:

        tensor = np.transpose(
            tensor,
            (
                2,
                0,
                1
            )
        )[None]


    else:

        raise RuntimeError(
            "Unexpected hand landmark input "
            f"shape: {input_shape}"
        )


    return np.ascontiguousarray(
        tensor,
        dtype=np.float32
    )


# ==================================================================================================
# 20. LANDMARK INFERENCE
# ==================================================================================================

def run_landmark(
    crop
):

    tensor = (
        preprocess_hand(
            crop
        )
    )


    start = (
        time.perf_counter()
    )


    result = hand_net(
        [tensor]
    )


    latency_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    screen_landmarks = None
    world_landmarks = None

    scalar_outputs = []


    for index, output in enumerate(
        hand_net.outputs
    ):

        array = np.asarray(
            result[
                output
            ]
        )


        flat = array.reshape(
            -1
        )


        if flat.size == 63:

            candidate = (
                flat.reshape(
                    21,
                    3
                )
                .astype(
                    np.float32
                )
            )


            maximum_xy = float(
                np.max(
                    np.abs(
                        candidate[
                            :,
                            :2
                        ]
                    )
                )
            )


            # Exact L1 identification.

            if maximum_xy > 2.0:

                screen_landmarks = (
                    candidate
                )

            else:

                world_landmarks = (
                    candidate
                )


        elif flat.size == 1:

            scalar_outputs.append(
                (
                    index,
                    float(
                        flat[0]
                    )
                )
            )


    if screen_landmarks is None:

        raise RuntimeError(
            "21-point screen landmark output "
            "could not be identified."
        )


    return (
        screen_landmarks,
        world_landmarks,
        scalar_outputs,
        latency_ms
    )


# ==================================================================================================
# 21. EXACT INVERSE PROJECTION
# ==================================================================================================

def project_landmarks_back(
    landmarks,
    inverse_matrix,
    rect
):

    xy = (
        landmarks[
            :,
            :2
        ]
        .astype(
            np.float32
        )
    )


    points = (
        xy.reshape(
            -1,
            1,
            2
        )
    )


    projected = (
        cv2.perspectiveTransform(
            points,
            inverse_matrix
        )
        .reshape(
            -1,
            2
        )
    )


    z = (
        landmarks[
            :,
            2
        ]
        /
        LANDMARK_SIZE
        *
        rect[
            "width"
        ]
    )


    return (
        np.column_stack(
            [
                projected,
                z
            ]
        )
        .astype(
            np.float32
        )
    )


# ==================================================================================================
# 22. COMPLETE HAND PIPELINE
# ==================================================================================================

def detect_hands(
    frame
):

    # ----------------------------------------------------------------------------------------------
    # Palm inference
    # ----------------------------------------------------------------------------------------------

    (
        regression,
        logits,
        info,
        palm_ms
    ) = run_palm(
        frame
    )


    # ----------------------------------------------------------------------------------------------
    # Decode
    # ----------------------------------------------------------------------------------------------

    detections = (
        decode_palms(
            regression,
            logits
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Weighted NMS
    # ----------------------------------------------------------------------------------------------

    detections = (
        weighted_nms(
            detections
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Remove letterbox
    # ----------------------------------------------------------------------------------------------

    detections = [

        detection_to_original(
            detection,
            info
        )

        for detection
        in detections
    ]


    results = []

    landmark_total_ms = 0.0


    # ----------------------------------------------------------------------------------------------
    # Landmark inference
    # ----------------------------------------------------------------------------------------------

    for palm in detections:

        try:

            rect = (
                palm_to_hand_rect(
                    palm
                )
            )


            (
                crop,
                transform,
                inverse_transform,
                corners
            ) = crop_hand_roi(
                frame,
                rect
            )


            (
                landmarks,
                world_landmarks,
                scalar_outputs,
                landmark_ms
            ) = run_landmark(
                crop
            )


            projected = (
                project_landmarks_back(
                    landmarks,
                    inverse_transform,
                    rect
                )
            )


            landmark_total_ms += (
                landmark_ms
            )


            results.append(
                {

                    "palm":
                        palm,

                    "rect":
                        rect,

                    "corners":
                        corners,

                    "landmarks":
                        projected,

                    "world_landmarks":
                        world_landmarks,

                    "scalar_outputs":
                        scalar_outputs
                }
            )


        except Exception:

            # One malformed ROI should not kill
            # the webcam stream.

            continue


    return (
        results,
        palm_ms,
        landmark_total_ms
    )


# ==================================================================================================
# 23. DRAW LIVE HAND
# ==================================================================================================

def draw_hand(
    frame,
    result
):

    height, width = (
        frame.shape[:2]
    )


    palm = result[
        "palm"
    ]

    landmarks = result[
        "landmarks"
    ]

    corners = np.round(
        result[
            "corners"
        ]
    ).astype(
        np.int32
    )


    # ----------------------------------------------------------------------------------------------
    # PALM BOX — YELLOW
    # ----------------------------------------------------------------------------------------------

    x1, y1, x2, y2 = (
        palm[
            "bbox"
        ]
    )


    x1 = int(
        round(x1)
    )

    y1 = int(
        round(y1)
    )

    x2 = int(
        round(x2)
    )

    y2 = int(
        round(y2)
    )


    cv2.rectangle(
        frame,
        (
            x1,
            y1
        ),
        (
            x2,
            y2
        ),
        (
            0,
            255,
            255
        ),
        2,
        cv2.LINE_AA
    )


    # ----------------------------------------------------------------------------------------------
    # ROTATED HAND ROI — CYAN
    # ----------------------------------------------------------------------------------------------

    cv2.polylines(
        frame,
        [
            corners
        ],
        True,
        (
            255,
            255,
            0
        ),
        1,
        cv2.LINE_AA
    )


    # ----------------------------------------------------------------------------------------------
    # SKELETON — GREEN
    # ----------------------------------------------------------------------------------------------

    for start_id, end_id in (
        HAND_CONNECTIONS
    ):

        x_start = int(
            round(
                landmarks[
                    start_id,
                    0
                ]
            )
        )

        y_start = int(
            round(
                landmarks[
                    start_id,
                    1
                ]
            )
        )


        x_end = int(
            round(
                landmarks[
                    end_id,
                    0
                ]
            )
        )

        y_end = int(
            round(
                landmarks[
                    end_id,
                    1
                ]
            )
        )


        valid_start = (
            -width <=
            x_start <=
            2 * width
            and
            -height <=
            y_start <=
            2 * height
        )


        valid_end = (
            -width <=
            x_end <=
            2 * width
            and
            -height <=
            y_end <=
            2 * height
        )


        if (
            valid_start
            and
            valid_end
        ):

            cv2.line(
                frame,
                (
                    x_start,
                    y_start
                ),
                (
                    x_end,
                    y_end
                ),
                (
                    0,
                    255,
                    0
                ),
                2,
                cv2.LINE_AA
            )


    # ----------------------------------------------------------------------------------------------
    # LANDMARKS — RED
    # ----------------------------------------------------------------------------------------------

    for landmark_id in range(
        21
    ):

        x = int(
            round(
                landmarks[
                    landmark_id,
                    0
                ]
            )
        )

        y = int(
            round(
                landmarks[
                    landmark_id,
                    1
                ]
            )
        )


        if (
            0 <= x < width
            and
            0 <= y < height
        ):

            radius = (
                5
                if landmark_id == 0
                else 3
            )


            # White outline

            cv2.circle(
                frame,
                (
                    x,
                    y
                ),
                radius + 1,
                (
                    255,
                    255,
                    255
                ),
                -1,
                cv2.LINE_AA
            )


            # Red landmark

            cv2.circle(
                frame,
                (
                    x,
                    y
                ),
                radius,
                (
                    0,
                    0,
                    255
                ),
                -1,
                cv2.LINE_AA
            )


    return frame


# ==================================================================================================
# 24. OPEN WEBCAM
# ==================================================================================================

print()
print("=" * 110)
print("OPENING WEBCAM")
print("=" * 110)


camera = cv2.VideoCapture(
    CAMERA_ID
)


if not camera.isOpened():

    raise RuntimeError(
        "\nCould not open webcam.\n\n"
        "Try CAMERA_ID = 1 if your webcam is not /dev/video0."
    )


camera.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    REQUEST_WIDTH
)

camera.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    REQUEST_HEIGHT
)


# Read first frame.

ok, test_frame = (
    camera.read()
)


if not ok:

    camera.release()

    raise RuntimeError(
        "Webcam opened but no frame could be read."
    )


actual_height, actual_width = (
    test_frame.shape[:2]
)


camera_fps_reported = float(
    camera.get(
        cv2.CAP_PROP_FPS
    )
)


print()
print("[PASS] Webcam opened.")

print(
    "Requested resolution:",
    REQUEST_WIDTH,
    "x",
    REQUEST_HEIGHT
)

print(
    "Actual resolution   :",
    actual_width,
    "x",
    actual_height
)

print(
    "Camera-reported FPS :",
    camera_fps_reported
)

print()
print("Controls:")
print("  Q   -> quit")
print("  ESC -> quit")
print("  S   -> save current frame")


# ==================================================================================================
# 25. LIVE METRICS
# ==================================================================================================

frame_number = 0

frames_with_hand = 0

total_hand_instances = 0

frame_times = []

palm_times = []

landmark_times = []

capture_start = (
    time.perf_counter()
)

last_fps_time = (
    time.perf_counter()
)

fps_counter = 0

display_fps = 0.0


# Store lightweight frame-level report.

frame_records = []


# ==================================================================================================
# 26. LIVE LOOP
# ==================================================================================================

print()
print("=" * 110)
print("L25 LIVE STREAM STARTED")
print("=" * 110)

print()
print("Show your hand to the webcam.")
print("Press Q or ESC inside the webcam window to stop.")
print()


try:

    while True:

        frame_start = (
            time.perf_counter()
        )


        ok, frame = (
            camera.read()
        )


        if not ok:

            print(
                "\n[WARN] Webcam frame read failed."
            )

            break


        frame_number += 1


        # ------------------------------------------------------------------------------------------
        # IMPORTANT:
        #
        # Do NOT horizontally flip before inference.
        #
        # Training/deployment pipeline did not use horizontal
        # flipping as a semantic-preserving transformation.
        # ------------------------------------------------------------------------------------------


        try:

            (
                hand_results,
                palm_ms,
                landmark_ms
            ) = detect_hands(
                frame
            )


        except Exception as error:

            hand_results = []

            palm_ms = np.nan

            landmark_ms = np.nan


        hand_count = len(
            hand_results
        )


        if hand_count > 0:

            frames_with_hand += 1


        total_hand_instances += (
            hand_count
        )


        if np.isfinite(
            palm_ms
        ):

            palm_times.append(
                palm_ms
            )


        if np.isfinite(
            landmark_ms
        ):

            landmark_times.append(
                landmark_ms
            )


        # ------------------------------------------------------------------------------------------
        # Draw palm / skeleton
        # ------------------------------------------------------------------------------------------

        annotated = (
            frame.copy()
        )


        for hand_result in (
            hand_results
        ):

            annotated = draw_hand(
                annotated,
                hand_result
            )


        # ------------------------------------------------------------------------------------------
        # FPS calculation
        # ------------------------------------------------------------------------------------------

        fps_counter += 1


        now = (
            time.perf_counter()
        )


        elapsed_fps = (
            now -
            last_fps_time
        )


        if elapsed_fps >= 0.5:

            display_fps = (
                fps_counter /
                elapsed_fps
            )

            fps_counter = 0

            last_fps_time = (
                now
            )


        # ------------------------------------------------------------------------------------------
        # Top information bar
        # ------------------------------------------------------------------------------------------

        height, width = (
            annotated.shape[:2]
        )


        overlay = (
            annotated.copy()
        )


        cv2.rectangle(
            overlay,
            (
                0,
                0
            ),
            (
                width,
                42
            ),
            (
                0,
                0,
                0
            ),
            -1
        )


        cv2.addWeighted(
            overlay,
            0.60,
            annotated,
            0.40,
            0,
            annotated
        )


        info_text = (
            f"L25 LIVE | "
            f"Hands: {hand_count} | "
            f"FPS: {display_fps:.1f} | "
            f"OpenVINO {DEVICE}"
        )


        cv2.putText(
            annotated,
            info_text,
            (
                8,
                26
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.50,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        # Bottom instructions.

        cv2.putText(
            annotated,
            "Q/ESC: quit | S: snapshot",
            (
                8,
                height - 12
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (
                255,
                255,
                255
            ),
            1,
            cv2.LINE_AA
        )


        # ------------------------------------------------------------------------------------------
        # Display
        # ------------------------------------------------------------------------------------------

        cv2.imshow(
            "CISLR L25 - Live OpenVINO Hand Acquisition",
            annotated
        )


        # ------------------------------------------------------------------------------------------
        # Frame timing
        # ------------------------------------------------------------------------------------------

        frame_ms = (
            time.perf_counter()
            -
            frame_start
        ) * 1000.0


        frame_times.append(
            frame_ms
        )


        frame_records.append(
            {

                "frame":
                    frame_number,

                "timestamp_sec":
                    float(
                        time.perf_counter()
                        -
                        capture_start
                    ),

                "hand_count":
                    hand_count,

                "palm_ms":
                    (
                        float(
                            palm_ms
                        )
                        if np.isfinite(
                            palm_ms
                        )
                        else np.nan
                    ),

                "landmark_ms":
                    (
                        float(
                            landmark_ms
                        )
                        if np.isfinite(
                            landmark_ms
                        )
                        else np.nan
                    ),

                "complete_frame_ms":
                    float(
                        frame_ms
                    )
            }
        )


        # ------------------------------------------------------------------------------------------
        # Keyboard
        # ------------------------------------------------------------------------------------------

        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        # Q

        if key == ord(
            "q"
        ):

            print(
                "\nQ pressed."
            )

            break


        # ESC

        if key == 27:

            print(
                "\nESC pressed."
            )

            break


        # S = save snapshot

        if key == ord(
            "s"
        ):

            snapshot_path = (
                SNAPSHOT_DIR /
                (
                    f"l25_snapshot_"
                    f"{int(time.time())}.jpg"
                )
            )


            cv2.imwrite(
                str(
                    snapshot_path
                ),
                annotated
            )


            print(
                "\n[SNAPSHOT]",
                snapshot_path
            )


finally:

    camera.release()

    cv2.destroyAllWindows()


# ==================================================================================================
# 27. FINAL METRICS
# ==================================================================================================

capture_duration = (
    time.perf_counter()
    -
    capture_start
)


if frame_number == 0:

    raise RuntimeError(
        "No webcam frames processed."
    )


overall_fps = (
    frame_number /
    capture_duration
)


hand_detection_rate = (
    frames_with_hand /
    frame_number
)


mean_frame_ms = (
    float(
        np.mean(
            frame_times
        )
    )
    if frame_times
    else np.nan
)


median_frame_ms = (
    float(
        np.median(
            frame_times
        )
    )
    if frame_times
    else np.nan
)


mean_palm_ms = (
    float(
        np.mean(
            palm_times
        )
    )
    if palm_times
    else np.nan
)


median_palm_ms = (
    float(
        np.median(
            palm_times
        )
    )
    if palm_times
    else np.nan
)


mean_landmark_ms = (
    float(
        np.mean(
            landmark_times
        )
    )
    if landmark_times
    else np.nan
)


median_landmark_ms = (
    float(
        np.median(
            landmark_times
        )
    )
    if landmark_times
    else np.nan
)


# ==================================================================================================
# 28. SAVE FRAME REPORT
# ==================================================================================================

frame_report_path = (
    REPORT_DIR /
    "l25_frame_metrics.csv"
)


frame_df = pd.DataFrame(
    frame_records
)


frame_df.to_csv(
    frame_report_path,
    index=False
)


# ==================================================================================================
# 29. SAVE CONFIG
# ==================================================================================================

config = {

    "stage":
        "L25",

    "name":
        "Live webcam acquisition with OpenVINO palm and hand landmarks",

    "device":
        DEVICE,

    "camera_id":
        CAMERA_ID,

    "requested_resolution":
        [
            REQUEST_WIDTH,
            REQUEST_HEIGHT
        ],

    "actual_resolution":
        [
            actual_width,
            actual_height
        ],

    "palm_input_size":
        PALM_SIZE,

    "landmark_input_size":
        LANDMARK_SIZE,

    "palm_threshold":
        PALM_THRESHOLD,

    "nms_threshold":
        NMS_THRESHOLD,

    "max_hands":
        MAX_HANDS,

    "horizontal_flip":
        False,

    "videomae_enabled":
        False,

    "training":
        False,

    "locked_test_used":
        False
}


config_path = (
    CONFIG_DIR /
    "l25_config.json"
)


with open(
    config_path,
    "w"
) as file:

    json.dump(
        config,
        file,
        indent=2
    )


# ==================================================================================================
# 30. SAVE SUMMARY
# ==================================================================================================

summary = {

    "frames_processed":
        int(
            frame_number
        ),

    "capture_duration_sec":
        float(
            capture_duration
        ),

    "overall_pipeline_fps":
        float(
            overall_fps
        ),

    "frames_with_hand":
        int(
            frames_with_hand
        ),

    "hand_detection_rate":
        float(
            hand_detection_rate
        ),

    "total_hand_instances":
        int(
            total_hand_instances
        ),

    "mean_complete_frame_ms":
        float(
            mean_frame_ms
        ),

    "median_complete_frame_ms":
        float(
            median_frame_ms
        ),

    "mean_palm_inference_ms":
        float(
            mean_palm_ms
        ),

    "median_palm_inference_ms":
        float(
            median_palm_ms
        ),

    "mean_landmark_inference_ms":
        float(
            mean_landmark_ms
        ),

    "median_landmark_inference_ms":
        float(
            median_landmark_ms
        ),

    "videomae_used":
        False,

    "locked_test_used":
        False
}


summary_path = (
    REPORT_DIR /
    "l25_summary.json"
)


with open(
    summary_path,
    "w"
) as file:

    json.dump(
        summary,
        file,
        indent=2
    )


# ==================================================================================================
# 31. SHA256 HELPER
# ==================================================================================================

def sha256_file(
    path
):

    hasher = (
        hashlib.sha256()
    )


    with open(
        path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            hasher.update(
                chunk
            )


    return (
        hasher.hexdigest()
    )


# ==================================================================================================
# 32. L25 MASTER FREEZE
# ==================================================================================================

master = {

    "stage":
        "L25",

    "status":
        "COMPLETE",

    "experiment":
        "Live webcam acquisition",

    "pipeline":
        [
            "webcam",
            "OpenVINO palm detector",
            "weighted NMS",
            "rotated hand ROI",
            "OpenVINO 21-point hand landmark detector",
            "inverse perspective projection",
            "live visualization"
        ],

    "models":
        {

            "palm":
                str(
                    PALM_MODEL
                ),

            "palm_sha256":
                sha256_file(
                    PALM_MODEL
                ),

            "landmark":
                str(
                    HAND_MODEL
                ),

            "landmark_sha256":
                sha256_file(
                    HAND_MODEL
                )
        },

    "results":
        summary,

    "artifacts":
        {

            "frame_metrics":
                str(
                    frame_report_path
                ),

            "frame_metrics_sha256":
                sha256_file(
                    frame_report_path
                ),

            "config":
                str(
                    config_path
                ),

            "config_sha256":
                sha256_file(
                    config_path
                ),

            "summary":
                str(
                    summary_path
                ),

            "summary_sha256":
                sha256_file(
                    summary_path
                )
        },

    "scientific_integrity":
        {

            "training_performed":
                False,

            "retraining_performed":
                False,

            "videomae_classification":
                False,

            "validation_evaluation":
                False,

            "test_evaluation":
                False,

            "locked_test_loaded":
                False,

            "locked_test_evaluated":
                False,

            "formal_performance_benchmark":
                False
        },

    "note":
        (
            "L25 validates live webcam acquisition and the "
            "OpenVINO palm/21-point landmark perception pipeline. "
            "Displayed timing is diagnostic only. Formal end-to-end "
            "benchmarking remains L32."
        )
}


master_path = (
    L25_ROOT /
    "L25_WEBCAM_ACQUISITION_MASTER_FREEZE.json"
)


with open(
    master_path,
    "w"
) as file:

    json.dump(
        master,
        file,
        indent=2
    )


master_sha = (
    sha256_file(
        master_path
    )
)


# ==================================================================================================
# 33. FINAL OUTPUT
# ==================================================================================================

print()
print("=" * 110)
print("L25 — WEBCAM ACQUISITION COMPLETE")
print("=" * 110)

print()
print("Frames processed       :", frame_number)

print(
    "Duration               :",
    f"{capture_duration:.2f} sec"
)

print(
    "Overall pipeline FPS   :",
    f"{overall_fps:.2f}"
)

print()
print(
    "Frames with hand       :",
    frames_with_hand
)

print(
    "Hand detection rate    :",
    f"{hand_detection_rate * 100:.2f}%"
)

print(
    "Hand instances         :",
    total_hand_instances
)

print()
print(
    "Mean complete frame    :",
    f"{mean_frame_ms:.3f} ms"
)

print(
    "Median complete frame  :",
    f"{median_frame_ms:.3f} ms"
)

print()
print(
    "Mean palm inference    :",
    f"{mean_palm_ms:.3f} ms"
)

print(
    "Median palm inference  :",
    f"{median_palm_ms:.3f} ms"
)

print()
print(
    "Mean landmark inference:",
    f"{mean_landmark_ms:.3f} ms"
)

print(
    "Median landmark infer. :",
    f"{median_landmark_ms:.3f} ms"
)


print()
print("-" * 110)

print("ARTIFACTS")

print("-" * 110)

print(
    "Frame metrics:"
)

print(
    frame_report_path
)

print()
print(
    "Summary:"
)

print(
    summary_path
)

print()
print(
    "Config:"
)

print(
    config_path
)

print()
print(
    "Master freeze:"
)

print(
    master_path
)

print()
print(
    "Master SHA256:"
)

print(
    master_sha
)


print()
print("-" * 110)

print("INTEGRITY")

print("-" * 110)

print(
    "Live webcam               : YES"
)

print(
    "OpenVINO palm detector    : YES"
)

print(
    "OpenVINO landmarks        : YES"
)

print(
    "21-point skeleton         : YES"
)

print(
    "VideoMAE classification   : NO — L26"
)

print(
    "Training                  : NO"
)

print(
    "Retraining                : NO"
)

print(
    "Locked test loaded        : NO"
)

print(
    "Locked test evaluated     : NO"
)

print()
print(
    "[PASS] L25 live acquisition pipeline completed."
)

CISLR — L25 LIVE WEBCAM ACQUISITION

OpenVINO version : 2026.3.1-22476-759c5a6ab8c-releases/2026/3
OpenCV version   : 5.0.0
Device           : CPU
Camera ID        : 0

Palm model:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/palm/palm_detection_full.tflite

Landmark model:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/hand_landmark/hand_landmark_full.tflite

[PASS] Palm and landmark models found.

Available OpenVINO devices:
   CPU
   GPU

PALM MODEL
Input shape: [1, 192, 192, 3]
Outputs:
  0: [1, 2016, 18]
  1: [1, 2016, 1]

LANDMARK MODEL
Input shape: [1, 224, 224, 3]
Outputs:
  0: [1, 63]
  1: [1, 1]
  2: [1, 1]
  3: [1, 63]

[PASS] OpenVINO palm + landmark models compiled.

[PASS] Palm anchors: (2016, 4)

OPENING WEBCAM

[PASS] Webcam opened.
Requested resolution: 640 x 480
Actual resolution   : 640 x 480
Camera-reported FPS : 30.0

Controls:
  Q   -> quit
  ESC -> quit
  S   -> save current frame

L25 LIVE STREA

QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or sw


Q pressed.

L25 — WEBCAM ACQUISITION COMPLETE

Frames processed       : 806
Duration               : 30.95 sec
Overall pipeline FPS   : 26.05

Frames with hand       : 723
Hand detection rate    : 89.70%
Hand instances         : 988

Mean complete frame    : 34.952 ms
Median complete frame  : 30.662 ms

Mean palm inference    : 4.939 ms
Median palm inference  : 4.778 ms

Mean landmark inference: 5.176 ms
Median landmark infer. : 4.351 ms

--------------------------------------------------------------------------------------------------------------
ARTIFACTS
--------------------------------------------------------------------------------------------------------------
Frame metrics:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l25_webcam_acquisition/reports/l25_frame_metrics.csv

Summary:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l25_webcam_acquisition/reports/l25_summary.json

Config:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMA

In [13]:
# ==================================================================================================
# CISLR — L26C FIX
# REPLACE SECTION 16 ONWARD WITH THIS CELL
# ==================================================================================================
#
# Fixes:
#   RuntimeError:
#   "to_shape was called on a dynamic shape"
#
# Assumes these already exist from the previous L26C cell:
#
#   model
#   val_dataset
#   CLASS_NAMES
#   SEQUENCE_LENGTH
#   FEATURE_DIM
#   best_epoch
#   best_val_accuracy
#   train_dataset
#   MODEL_ROOT
#   REPORT_ROOT
#   history
#
# NO retraining.
# NO landmark re-extraction.
# NO locked-test access.
#
# ==================================================================================================

import copy
import json
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import openvino as ov


print("=" * 110)
print("L26C — CORRECTED STATIC OPENVINO EXPORT")
print("=" * 110)


# ==================================================================================================
# 1. SANITY CHECK REQUIRED VARIABLES
# ==================================================================================================

required_variables = [
    "model",
    "val_dataset",
    "CLASS_NAMES",
    "SEQUENCE_LENGTH",
    "FEATURE_DIM",
    "best_epoch",
    "best_val_accuracy",
    "train_dataset",
    "MODEL_ROOT",
    "REPORT_ROOT"
]


missing = [
    name
    for name in required_variables
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "Missing variables from the previous L26C training cell:\n"
        + "\n".join(missing)
    )


print()
print("[PASS] Previous L26C training state found.")

print(
    "Sequence length :",
    SEQUENCE_LENGTH
)

print(
    "Feature dim     :",
    FEATURE_DIM
)

print(
    "Classes         :",
    len(CLASS_NAMES)
)

print(
    "Best epoch      :",
    best_epoch
)

print(
    "Best val Top-1  :",
    f"{best_val_accuracy*100:.2f}%"
)


# ==================================================================================================
# 2. PREPARE PYTORCH MODEL
# ==================================================================================================

model_cpu = copy.deepcopy(
    model
).cpu().eval()


example_input = torch.zeros(
    (
        1,
        SEQUENCE_LENGTH,
        FEATURE_DIM
    ),
    dtype=torch.float32
)


print()
print(
    "Example input shape:",
    tuple(
        example_input.shape
    )
)


# ==================================================================================================
# 3. CONVERT PYTORCH -> OPENVINO
# ==================================================================================================

print()
print("Converting BiGRU to OpenVINO...")


ov_model = ov.convert_model(
    model_cpu,
    example_input=example_input
)


print("[PASS] Initial conversion complete.")


# ==================================================================================================
# 4. INSPECT PARTIAL SHAPE SAFELY
#
# IMPORTANT:
# Do NOT call:
#
#     ov_model.input(0).shape
#
# when the shape may still be dynamic.
# ==================================================================================================

print()
print(
    "Converted partial shape:",
    ov_model.input(0).partial_shape
)


# ==================================================================================================
# 5. FORCE STATIC DEPLOYMENT SHAPE
#
# THIS IS THE MAIN FIX.
# ==================================================================================================

STATIC_INPUT_SHAPE = [
    1,
    int(SEQUENCE_LENGTH),
    int(FEATURE_DIM)
]


print()
print(
    "Freezing deployment input to:",
    STATIC_INPUT_SHAPE
)


ov_model.reshape(
    {
        ov_model.input(0):
            STATIC_INPUT_SHAPE
    }
)


print(
    "After reshape partial shape:",
    ov_model.input(0).partial_shape
)


# ==================================================================================================
# 6. VERIFY MODEL IS STATIC
# ==================================================================================================

input_partial_shape = (
    ov_model.input(0).partial_shape
)


if input_partial_shape.is_dynamic:

    raise RuntimeError(
        "OpenVINO input is still dynamic after reshape:\n"
        f"{input_partial_shape}"
    )


print()
print(
    "[PASS] OpenVINO deployment input is static."
)


# ==================================================================================================
# 7. SAVE STATIC OPENVINO MODEL
# ==================================================================================================

xml_path = (
    MODEL_ROOT /
    "l26c_fast_landmark_bigru.xml"
)


ov.save_model(
    ov_model,
    str(xml_path)
)


bin_path = xml_path.with_suffix(
    ".bin"
)


if not xml_path.exists():

    raise FileNotFoundError(
        xml_path
    )


if not bin_path.exists():

    raise FileNotFoundError(
        bin_path
    )


print()
print("[PASS] Static OpenVINO model saved.")

print("XML:")
print(xml_path)

print()
print("BIN:")
print(bin_path)


# ==================================================================================================
# 8. CREATE FRESH OPENVINO CORE
# ==================================================================================================

ov_core = ov.Core()


print()
print(
    "Available devices:",
    ov_core.available_devices
)


# ==================================================================================================
# 9. READ SAVED MODEL AGAIN
#
# This verifies that static shape survived serialization.
# ==================================================================================================

saved_model = ov_core.read_model(
    str(xml_path)
)


saved_input = saved_model.input(0)


print()
print(
    "Saved model partial shape:",
    saved_input.partial_shape
)


if saved_input.partial_shape.is_dynamic:

    print(
        "[WARN] Saved model still reports a dynamic input."
    )

    print(
        "Applying static reshape again before compilation."
    )


    saved_model.reshape(
        {
            saved_model.input(0):
                STATIC_INPUT_SHAPE
        }
    )


# ==================================================================================================
# 10. COMPILE ON CPU
# ==================================================================================================

print()
print("Compiling lightweight classifier on CPU...")


compiled = ov_core.compile_model(
    saved_model,
    "CPU",
    {
        "PERFORMANCE_HINT":
            "LATENCY"
    }
)


ov_input = compiled.input(0)
ov_output = compiled.output(0)


# ==================================================================================================
# 11. SAFE SHAPE REPORTING
#
# partial_shape is safe even when dimensions are dynamic.
# ==================================================================================================

print()
print("=" * 110)
print("OPENVINO MODEL")
print("=" * 110)

print()
print(
    "Input partial shape :",
    ov_input.partial_shape
)

print(
    "Output partial shape:",
    ov_output.partial_shape
)


# Only call .shape when static.

if not ov_input.partial_shape.is_dynamic:

    print(
        "Static input shape  :",
        list(
            ov_input.shape
        )
    )


if not ov_output.partial_shape.is_dynamic:

    print(
        "Static output shape :",
        list(
            ov_output.shape
        )
    )


# ==================================================================================================
# 12. VERIFY EXPECTED INPUT
# ==================================================================================================

expected_input_shape = (
    1,
    SEQUENCE_LENGTH,
    FEATURE_DIM
)


actual_input_shape = tuple(
    int(x)
    for x in ov_input.shape
)


if actual_input_shape != expected_input_shape:

    raise RuntimeError(
        "Unexpected OpenVINO input shape.\n"
        f"Expected: {expected_input_shape}\n"
        f"Actual:   {actual_input_shape}"
    )


print()
print(
    "[PASS] Input shape verified:",
    actual_input_shape
)


# ==================================================================================================
# 13. GET VALIDATION SAMPLE
# ==================================================================================================

sample_x, sample_y = (
    val_dataset[0]
)


sample_np = (
    sample_x
    .numpy()[
        None
    ]
    .astype(
        np.float32
    )
)


print()
print(
    "Validation tensor:",
    sample_np.shape
)

print(
    "Ground truth:",
    CLASS_NAMES[
        int(sample_y)
    ]
)


if sample_np.shape != (
    1,
    SEQUENCE_LENGTH,
    FEATURE_DIM
):

    raise RuntimeError(
        f"Wrong validation tensor shape: {sample_np.shape}"
    )


# ==================================================================================================
# 14. PYTORCH INFERENCE
# ==================================================================================================

with torch.no_grad():

    pt_logits = (
        model_cpu(
            torch.from_numpy(
                sample_np
            )
        )
        .detach()
        .cpu()
        .numpy()
    )


# ==================================================================================================
# 15. OPENVINO INFERENCE
# ==================================================================================================

ov_result = compiled(
    {
        ov_input:
            sample_np
    }
)


ov_logits = np.asarray(
    ov_result[
        ov_output
    ]
)


print()
print(
    "PyTorch logits shape :",
    pt_logits.shape
)

print(
    "OpenVINO logits shape:",
    ov_logits.shape
)


# ==================================================================================================
# 16. NUMERICAL COMPARISON
# ==================================================================================================

if pt_logits.shape != ov_logits.shape:

    raise RuntimeError(
        "PyTorch/OpenVINO output shape mismatch."
    )


max_error = float(
    np.max(
        np.abs(
            pt_logits -
            ov_logits
        )
    )
)


mean_error = float(
    np.mean(
        np.abs(
            pt_logits -
            ov_logits
        )
    )
)


pt_prediction = int(
    np.argmax(
        pt_logits,
        axis=1
    )[0]
)


ov_prediction = int(
    np.argmax(
        ov_logits,
        axis=1
    )[0]
)


print()
print("=" * 110)
print("PYTORCH ↔ OPENVINO VERIFICATION")
print("=" * 110)

print()
print(
    "Ground truth        :",
    CLASS_NAMES[
        int(sample_y)
    ]
)

print(
    "PyTorch prediction  :",
    CLASS_NAMES[
        pt_prediction
    ]
)

print(
    "OpenVINO prediction :",
    CLASS_NAMES[
        ov_prediction
    ]
)

print()
print(
    "Max abs logit error :",
    f"{max_error:.10f}"
)

print(
    "Mean abs error      :",
    f"{mean_error:.10f}"
)


if pt_prediction != ov_prediction:

    raise RuntimeError(
        "PyTorch/OpenVINO Top-1 mismatch."
    )


print()
print(
    "[PASS] PyTorch/OpenVINO Top-1 predictions match."
)


# ==================================================================================================
# 17. VERIFY ENTIRE VALIDATION SET
#
# Since validation is small, compare PT and OV over all validation samples.
# This does NOT touch locked test.
# ==================================================================================================

print()
print("=" * 110)
print("FULL VALIDATION PT ↔ OV CHECK")
print("=" * 110)


pt_correct = 0
ov_correct = 0
agreement = 0

validation_max_error = 0.0

validation_records = []


for index in range(
    len(
        val_dataset
    )
):

    x,y = val_dataset[
        index
    ]


    x_np = (
        x.numpy()[
            None
        ]
        .astype(
            np.float32
        )
    )


    with torch.no_grad():

        pt_out = (
            model_cpu(
                torch.from_numpy(
                    x_np
                )
            )
            .detach()
            .cpu()
            .numpy()
        )


    ov_out = np.asarray(
        compiled(
            {
                ov_input:
                    x_np
            }
        )[
            ov_output
        ]
    )


    pt_id = int(
        np.argmax(
            pt_out,
            axis=1
        )[0]
    )


    ov_id = int(
        np.argmax(
            ov_out,
            axis=1
        )[0]
    )


    gt_id = int(
        y
    )


    current_error = float(
        np.max(
            np.abs(
                pt_out -
                ov_out
            )
        )
    )


    validation_max_error = max(
        validation_max_error,
        current_error
    )


    if pt_id == gt_id:

        pt_correct += 1


    if ov_id == gt_id:

        ov_correct += 1


    if pt_id == ov_id:

        agreement += 1


    validation_records.append(
        {
            "index":
                index,

            "ground_truth":
                CLASS_NAMES[
                    gt_id
                ],

            "pytorch_prediction":
                CLASS_NAMES[
                    pt_id
                ],

            "openvino_prediction":
                CLASS_NAMES[
                    ov_id
                ],

            "pt_ov_match":
                bool(
                    pt_id ==
                    ov_id
                ),

            "max_abs_logit_error":
                current_error
        }
    )


validation_n = len(
    val_dataset
)


pt_val_accuracy = (
    pt_correct /
    validation_n
)


ov_val_accuracy = (
    ov_correct /
    validation_n
)


agreement_rate = (
    agreement /
    validation_n
)


print()
print(
    "Validation N            :",
    validation_n
)

print(
    "PyTorch Top-1           :",
    f"{pt_val_accuracy*100:.2f}% "
    f"({pt_correct}/{validation_n})"
)

print(
    "OpenVINO Top-1          :",
    f"{ov_val_accuracy*100:.2f}% "
    f"({ov_correct}/{validation_n})"
)

print(
    "PT ↔ OV Top-1 agreement :",
    f"{agreement_rate*100:.2f}% "
    f"({agreement}/{validation_n})"
)

print(
    "Global max logit error  :",
    f"{validation_max_error:.10f}"
)


# ==================================================================================================
# 18. SAVE VALIDATION VERIFICATION
# ==================================================================================================

verification_path = (
    REPORT_ROOT /
    "l26c_openvino_validation_verification.csv"
)


pd.DataFrame(
    validation_records
).to_csv(
    verification_path,
    index=False
)


# ==================================================================================================
# 19. OPENVINO LATENCY BENCHMARK
#
# This measures ONLY the tiny temporal classifier.
# Palm/landmark/end-to-end latency is measured separately in live deployment.
# ==================================================================================================

print()
print("=" * 110)
print("OPENVINO TEMPORAL CLASSIFIER SPEED")
print("=" * 110)


# Warm-up

for _ in range(20):

    _ = compiled(
        {
            ov_input:
                sample_np
        }
    )


benchmark_runs = 200

latencies = []


for _ in range(
    benchmark_runs
):

    start = time.perf_counter()


    _ = compiled(
        {
            ov_input:
                sample_np
        }
    )


    latency_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    latencies.append(
        latency_ms
    )


latencies = np.asarray(
    latencies,
    dtype=np.float64
)


mean_latency = float(
    np.mean(
        latencies
    )
)


median_latency = float(
    np.median(
        latencies
    )
)


p95_latency = float(
    np.percentile(
        latencies,
        95
    )
)


minimum_latency = float(
    np.min(
        latencies
    )
)


maximum_latency = float(
    np.max(
        latencies
    )
)


print()
print(
    "Runs   :",
    benchmark_runs
)

print(
    "Mean   :",
    f"{mean_latency:.3f} ms"
)

print(
    "Median :",
    f"{median_latency:.3f} ms"
)

print(
    "P95    :",
    f"{p95_latency:.3f} ms"
)

print(
    "Min    :",
    f"{minimum_latency:.3f} ms"
)

print(
    "Max    :",
    f"{maximum_latency:.3f} ms"
)


# ==================================================================================================
# 20. SAVE LATENCY VALUES
# ==================================================================================================

latency_path = (
    REPORT_ROOT /
    "l26c_openvino_latency.csv"
)


pd.DataFrame(
    {
        "run":
            np.arange(
                1,
                benchmark_runs + 1
            ),

        "latency_ms":
            latencies
    }
).to_csv(
    latency_path,
    index=False
)


# ==================================================================================================
# 21. SAVE TRAINING HISTORY IF AVAILABLE
# ==================================================================================================

history_path = (
    REPORT_ROOT /
    "training_history.csv"
)


if "history" in globals():

    pd.DataFrame(
        history
    ).to_csv(
        history_path,
        index=False
    )


# ==================================================================================================
# 22. SHA256
# ==================================================================================================

def sha256_file(
    path
):

    h = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:
                break


            h.update(
                block
            )


    return h.hexdigest()


xml_sha = sha256_file(
    xml_path
)

bin_sha = sha256_file(
    bin_path
)


# ==================================================================================================
# 23. SAVE SUMMARY
# ==================================================================================================

summary = {

    "stage":
        "L26C",

    "experiment":
        "10-class lightweight landmark BiGRU real-time deployment classifier",

    "status":
        "OPENVINO_EXPORT_VERIFIED",

    "classes":
        list(
            CLASS_NAMES
        ),

    "train_samples":
        int(
            len(
                train_dataset
            )
        ),

    "validation_samples":
        int(
            len(
                val_dataset
            )
        ),

    "sequence_length":
        int(
            SEQUENCE_LENGTH
        ),

    "feature_dim":
        int(
            FEATURE_DIM
        ),

    "openvino_input_shape":
        [
            1,
            int(
                SEQUENCE_LENGTH
            ),
            int(
                FEATURE_DIM
            )
        ],

    "best_epoch":
        int(
            best_epoch
        ),

    "training_best_validation_top1":
        float(
            best_val_accuracy
        ),

    "verified_pytorch_validation_top1":
        float(
            pt_val_accuracy
        ),

    "verified_openvino_validation_top1":
        float(
            ov_val_accuracy
        ),

    "pytorch_openvino_top1_agreement":
        float(
            agreement_rate
        ),

    "pytorch_openvino_global_max_abs_error":
        float(
            validation_max_error
        ),

    "openvino_device":
        "CPU",

    "classifier_latency_ms": {

        "mean":
            mean_latency,

        "median":
            median_latency,

        "p95":
            p95_latency,

        "minimum":
            minimum_latency,

        "maximum":
            maximum_latency
    },

    "artifacts": {

        "xml":
            str(
                xml_path
            ),

        "xml_sha256":
            xml_sha,

        "bin":
            str(
                bin_path
            ),

        "bin_sha256":
            bin_sha,

        "validation_verification":
            str(
                verification_path
            ),

        "latency_report":
            str(
                latency_path
            )
    },

    "scientific_integrity": {

        "locked_test_loaded":
            False,

        "locked_test_evaluated":
            False,

        "l22_videomae_modified":
            False,

        "l19_model_modified":
            False,

        "retraining_in_this_fix_cell":
            False,

        "validation_used_for_model_selection":
            True,

        "formal_end_to_end_latency":
            False
    }
}


summary_path = (
    REPORT_ROOT /
    "l26c_summary.json"
)


with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ==================================================================================================
# 24. MASTER FREEZE
# ==================================================================================================

master = {

    "stage":
        "L26C",

    "status":
        "COMPLETE",

    "purpose":
        (
            "Develop and export a lightweight 10-class "
            "landmark temporal classifier for low-latency "
            "real-time deployment."
        ),

    "model":
        "2-layer bidirectional GRU",

    "input":
        {
            "shape":
                [
                    1,
                    int(
                        SEQUENCE_LENGTH
                    ),
                    int(
                        FEATURE_DIM
                    )
                ],

            "sequence_length":
                int(
                    SEQUENCE_LENGTH
                ),

            "feature_dim":
                int(
                    FEATURE_DIM
                )
        },

    "validation":
        {

            "n":
                int(
                    validation_n
                ),

            "pytorch_top1":
                float(
                    pt_val_accuracy
                ),

            "openvino_top1":
                float(
                    ov_val_accuracy
                ),

            "pt_ov_agreement":
                float(
                    agreement_rate
                )
        },

    "performance":
        {

            "device":
                "CPU",

            "classifier_only_mean_ms":
                mean_latency,

            "classifier_only_median_ms":
                median_latency,

            "classifier_only_p95_ms":
                p95_latency
        },

    "openvino":
        {

            "xml":
                str(
                    xml_path
                ),

            "xml_sha256":
                xml_sha,

            "bin":
                str(
                    bin_path
                ),

            "bin_sha256":
                bin_sha
        },

    "integrity":
        {

            "locked_test_used":
                False,

            "l22_videomae_changed":
                False,

            "historical_frozen_artifacts_overwritten":
                False
        }
}


master_path = (
    OUTPUT_ROOT /
    "L26C_REALTIME_LANDMARK_OPENVINO_MASTER_FREEZE.json"
)


with open(
    master_path,
    "w"
) as f:

    json.dump(
        master,
        f,
        indent=2
    )


master_sha = sha256_file(
    master_path
)


# ==================================================================================================
# 25. FINAL REPORT
# ==================================================================================================

print()
print("=" * 110)
print("L26C — CORRECTED EXPORT COMPLETE")
print("=" * 110)

print()

print(
    "Model input              :",
    f"[1, {SEQUENCE_LENGTH}, {FEATURE_DIM}]"
)

print(
    "Model output             :",
    f"[1, {len(CLASS_NAMES)}]"
)

print()

print(
    "Best training-run val    :",
    f"{best_val_accuracy*100:.2f}%"
)

print(
    "Verified PyTorch val     :",
    f"{pt_val_accuracy*100:.2f}% "
    f"({pt_correct}/{validation_n})"
)

print(
    "Verified OpenVINO val    :",
    f"{ov_val_accuracy*100:.2f}% "
    f"({ov_correct}/{validation_n})"
)

print(
    "PT/OV agreement          :",
    f"{agreement_rate*100:.2f}%"
)

print()

print(
    "PT/OV max logit error    :",
    f"{validation_max_error:.10f}"
)

print()

print(
    "OV classifier mean       :",
    f"{mean_latency:.3f} ms"
)

print(
    "OV classifier median     :",
    f"{median_latency:.3f} ms"
)

print(
    "OV classifier P95        :",
    f"{p95_latency:.3f} ms"
)

print()
print("-" * 110)

print("OpenVINO XML:")
print(xml_path)

print()
print("OpenVINO BIN:")
print(bin_path)

print()
print("Summary:")
print(summary_path)

print()
print("Master:")
print(master_path)

print()
print("Master SHA256:")
print(master_sha)

print()
print("-" * 110)

print(
    "Locked test used         : NO"
)

print(
    "L22 VideoMAE changed     : NO"
)

print(
    "Retraining in fix cell   : NO"
)

print()
print(
    "[PASS] L26C static OpenVINO export and verification complete."
)

L26C — CORRECTED STATIC OPENVINO EXPORT

[PASS] Previous L26C training state found.
Sequence length : 32
Feature dim     : 63
Classes         : 10
Best epoch      : 20
Best val Top-1  : 60.00%

Example input shape: (1, 32, 63)

Converting BiGRU to OpenVINO...
[PASS] Initial conversion complete.

Converted partial shape: [?,?,?]

Freezing deployment input to: [1, 32, 63]
After reshape partial shape: [1,32,63]

[PASS] OpenVINO deployment input is static.

[PASS] Static OpenVINO model saved.
XML:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l26c_realtime_landmark_classifier/models/l26c_fast_landmark_bigru.xml

BIN:
/home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l26c_realtime_landmark_classifier/models/l26c_fast_landmark_bigru.bin

Available devices: ['CPU', 'GPU']

Saved model partial shape: [1,32,63]

Compiling lightweight classifier on CPU...

OPENVINO MODEL

Input partial shape : [1,32,63]
Output partial shape: [1,10]
Static input shape  : [1

In [14]:
# ==================================================================================================
# CISLR — L26D
# LOW-LATENCY REAL-TIME LANDMARK SIGN RECOGNITION
# ==================================================================================================
#
# Webcam
#   ↓
# OpenVINO Palm Detector
#   ↓
# OpenVINO 21-point Landmark Detector
#   ↓
# normalized 63-D landmark vector
#   ↓
# rolling 32-frame sequence
#   ↓
# OpenVINO BiGRU
#   ↓
# LIVE 10-class prediction
#
# NO VIDEOMAE.
# NO classifier background worker required.
#
# BiGRU measured median inference ≈ 0.697 ms.
#
# NOTE:
#   Current development validation accuracy = 60% (6/10).
#   This cell tests REAL-TIME responsiveness.
#
# ==================================================================================================

from pathlib import Path
from collections import deque

import cv2
import time
import numpy as np
import openvino as ov


# ==================================================================================================
# 1. CONFIGURATION
# ==================================================================================================

CAMERA_ID = 0

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480

SEQUENCE_LENGTH = 32
FEATURE_DIM = 63

CLASS_NAMES = [
    "monday",
    "interest",
    "thursday",
    "christmas",
    "pencil",
    "mother",
    "tuesday",
    "friday",
    "rubber",
    "wash"
]


MODEL_PATH = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL/audit/"
    "l26c_realtime_landmark_classifier/models/"
    "l26c_fast_landmark_bigru.xml"
)


# Run hand detection every frame initially.
#
# If hand detection itself becomes slow, change this to 2.
#
HAND_DETECT_EVERY = 1


# We can classify every frame because classifier itself is <1 ms.

CLASSIFY_EVERY = 1


# ==================================================================================================
# 2. CHECK REQUIRED FUNCTIONS
# ==================================================================================================

if "detect_hands" not in globals():

    raise RuntimeError(
        "\n"
        "detect_hands() is not currently defined.\n"
        "Run your working L25/L26 OpenVINO palm + landmark cell first."
    )


print("=" * 120)
print("CISLR — L26D LOW-LATENCY LIVE SIGN RECOGNITION")
print("=" * 120)


# ==================================================================================================
# 3. LOAD LIGHTWEIGHT OPENVINO CLASSIFIER
# ==================================================================================================

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        MODEL_PATH
    )


core_live = ov.Core()


print()
print("Available OpenVINO devices:")
print(core_live.available_devices)


landmark_temporal_model = (
    core_live.read_model(
        str(MODEL_PATH)
    )
)


landmark_temporal_net = (
    core_live.compile_model(
        landmark_temporal_model,
        "CPU",
        {
            "PERFORMANCE_HINT": "LATENCY"
        }
    )
)


temporal_input = (
    landmark_temporal_net.input(0)
)

temporal_output = (
    landmark_temporal_net.output(0)
)


print()
print("[PASS] Real-time classifier loaded.")

print(
    "Input :",
    temporal_input.partial_shape
)

print(
    "Output:",
    temporal_output.partial_shape
)


# ==================================================================================================
# 4. SOFTMAX
# ==================================================================================================

def live_softmax(x):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    x = x - np.max(x)

    e = np.exp(x)

    return (
        e /
        np.sum(e)
    )


# ==================================================================================================
# 5. LANDMARK NORMALIZATION
#
# MUST MATCH L26C TRAINING.
# ==================================================================================================

def live_normalize_landmarks(
    landmarks
):

    landmarks = np.asarray(
        landmarks,
        dtype=np.float32
    )


    if landmarks.shape != (21,3):

        raise ValueError(
            f"Expected (21,3), got {landmarks.shape}"
        )


    xyz = landmarks.copy()


    # Wrist-centred representation.

    xyz -= xyz[0:1]


    # Same scale definition used during L26C.

    scale = np.linalg.norm(
        xyz[9]
    )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = np.max(
            np.linalg.norm(
                xyz,
                axis=1
            )
        )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = 1.0


    xyz /= scale


    feature = xyz.reshape(
        -1
    ).astype(
        np.float32
    )


    return feature


# ==================================================================================================
# 6. EXTRACT LANDMARK ARRAY FROM detect_hands()
#
# Your previous pipeline returned result["landmarks"].
# ==================================================================================================

def get_landmarks_from_result(result):

    if result is None:
        return None


    if isinstance(result, dict):

        for key in [
            "landmarks",
            "screen_landmarks",
            "points"
        ]:

            if key in result:

                arr = np.asarray(
                    result[key],
                    dtype=np.float32
                )


                if arr.shape == (21,3):

                    return arr


    return None


# ==================================================================================================
# 7. CAMERA
#
# CAP_PROP_BUFFERSIZE=1 helps prevent old webcam frames from accumulating.
# ==================================================================================================

cap = cv2.VideoCapture(
    CAMERA_ID
)


if not cap.isOpened():

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    CAMERA_WIDTH
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    CAMERA_HEIGHT
)


try:

    cap.set(
        cv2.CAP_PROP_BUFFERSIZE,
        1
    )

except Exception:

    pass


actual_width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

actual_height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

camera_fps = cap.get(
    cv2.CAP_PROP_FPS
)


print()
print("=" * 120)
print("WEBCAM")
print("=" * 120)

print(
    "Resolution :",
    actual_width,
    "x",
    actual_height
)

print(
    "Camera FPS :",
    camera_fps
)


# ==================================================================================================
# 8. TEMPORAL STATE
# ==================================================================================================

landmark_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


# Last valid feature is useful for occasional missed detections.

last_valid_feature = np.zeros(
    FEATURE_DIM,
    dtype=np.float32
)


latest_hand_results = []

latest_label = "COLLECTING"

latest_confidence = 0.0

latest_top3 = []


# ==================================================================================================
# 9. METRICS
# ==================================================================================================

frame_counter = 0

hand_frames = 0

prediction_count = 0


frame_times = []

palm_times = []

landmark_times = []

classifier_times = []


fps_counter = 0

fps_start = time.perf_counter()

display_fps = 0.0


start_time = time.perf_counter()


print()
print("=" * 120)
print("LIVE RECOGNITION STARTED")
print("=" * 120)

print()
print("Palm detector       : OpenVINO")
print("Landmark detector   : OpenVINO")
print("Temporal classifier : OpenVINO BiGRU CPU")
print("VideoMAE            : DISABLED")
print()
print("Q / ESC = quit")
print("R       = reset sequence")
print()


# ==================================================================================================
# 10. LIVE LOOP
# ==================================================================================================

try:

    while True:

        frame_start = time.perf_counter()


        # ==========================================================================================
        # READ NEW CAMERA FRAME
        # ==========================================================================================

        ok, frame = cap.read()


        if not ok:

            print(
                "\n[WARN] Camera frame read failed."
            )

            continue


        frame_counter += 1


        # ==========================================================================================
        # HAND DETECTION
        # ==========================================================================================

        should_detect = (

            frame_counter == 1

            or

            frame_counter %
            HAND_DETECT_EVERY
            ==
            0
        )


        hand_found = False


        if should_detect:

            try:

                (
                    latest_hand_results,
                    palm_ms,
                    landmark_ms
                ) = detect_hands(
                    frame
                )


                palm_times.append(
                    float(
                        palm_ms
                    )
                )


                landmark_times.append(
                    float(
                        landmark_ms
                    )
                )


            except Exception as e:

                latest_hand_results = []

                palm_ms = np.nan

                landmark_ms = np.nan


        # ==========================================================================================
        # LANDMARK -> FEATURE
        # ==========================================================================================

        if len(latest_hand_results) > 0:

            landmarks = (
                get_landmarks_from_result(
                    latest_hand_results[0]
                )
            )


            if landmarks is not None:

                try:

                    feature = (
                        live_normalize_landmarks(
                            landmarks
                        )
                    )


                    if np.all(
                        np.isfinite(
                            feature
                        )
                    ):

                        last_valid_feature = (
                            feature.copy()
                        )

                        hand_found = True

                        hand_frames += 1


                except Exception:

                    hand_found = False


        # ==========================================================================================
        # TEMPORAL BUFFER
        #
        # IMPORTANT:
        #
        # When hand is detected:
        #     append current landmarks
        #
        # When a detector misses a frame:
        #     reuse most recent valid landmarks
        #
        # This avoids destroying the temporal sequence because of one missed detection.
        # ==========================================================================================

        if hand_found:

            landmark_buffer.append(
                last_valid_feature.copy()
            )


        elif len(landmark_buffer) > 0:

            landmark_buffer.append(
                last_valid_feature.copy()
            )


        # ==========================================================================================
        # TEMPORAL CLASSIFICATION
        #
        # After first 32 observations:
        #
        #   1..32 -> prediction
        #   2..33 -> prediction
        #   3..34 -> prediction
        #
        # No waiting for another complete clip.
        # ==========================================================================================

        if (
            len(landmark_buffer)
            ==
            SEQUENCE_LENGTH

            and

            frame_counter %
            CLASSIFY_EVERY
            ==
            0
        ):

            sequence = np.stack(
                landmark_buffer,
                axis=0
            ).astype(
                np.float32
            )


            sequence = sequence[
                None,
                :,
                :
            ]


            classifier_start = (
                time.perf_counter()
            )


            result = (
                landmark_temporal_net(
                    {
                        temporal_input:
                            sequence
                    }
                )
            )


            classifier_ms = (
                time.perf_counter()
                -
                classifier_start
            ) * 1000.0


            classifier_times.append(
                classifier_ms
            )


            logits = np.asarray(
                result[
                    temporal_output
                ]
            ).reshape(
                -1
            )


            probabilities = (
                live_softmax(
                    logits
                )
            )


            ranking = np.argsort(
                probabilities
            )[::-1]


            class_id = int(
                ranking[0]
            )


            latest_label = (
                CLASS_NAMES[
                    class_id
                ]
            )


            latest_confidence = float(
                probabilities[
                    class_id
                ]
            )


            latest_top3 = [

                (
                    CLASS_NAMES[
                        int(i)
                    ],

                    float(
                        probabilities[
                            int(i)
                        ]
                    )
                )

                for i in ranking[:3]
            ]


            prediction_count += 1


        # ==========================================================================================
        # DRAW SKELETON
        # ==========================================================================================

        display = frame.copy()


        if "draw_hand" in globals():

            for result in latest_hand_results:

                try:

                    draw_hand(
                        display,
                        result
                    )

                except Exception:

                    pass


        # ==========================================================================================
        # FPS
        # ==========================================================================================

        fps_counter += 1


        now = time.perf_counter()


        fps_elapsed = (
            now -
            fps_start
        )


        if fps_elapsed >= 0.5:

            display_fps = (
                fps_counter /
                fps_elapsed
            )


            fps_counter = 0

            fps_start = now


        # ==========================================================================================
        # UI BACKGROUND
        # ==========================================================================================

        height,width = (
            display.shape[:2]
        )


        overlay = display.copy()


        cv2.rectangle(
            overlay,
            (0,0),
            (width,150),
            (0,0,0),
            -1
        )


        cv2.addWeighted(
            overlay,
            0.65,
            display,
            0.35,
            0,
            display
        )


        # ==========================================================================================
        # FPS
        # ==========================================================================================

        cv2.putText(
            display,
            f"L26D REAL-TIME | FPS {display_fps:.1f}",
            (10,24),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # BUFFER STATUS
        # ==========================================================================================

        cv2.putText(
            display,
            (
                f"Landmarks: "
                f"{len(landmark_buffer)}/"
                f"{SEQUENCE_LENGTH}"
            ),
            (10,49),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.47,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        # ==========================================================================================
        # PREDICTION
        # ==========================================================================================

        if (
            len(landmark_buffer)
            <
            SEQUENCE_LENGTH
        ):

            cv2.putText(
                display,
                (
                    "Collecting motion..."
                ),
                (10,82),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0,255,255),
                2,
                cv2.LINE_AA
            )


        else:

            cv2.putText(
                display,
                (
                    f"SIGN: "
                    f"{latest_label.upper()}"
                ),
                (10,82),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (0,255,0),
                2,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                (
                    f"Confidence: "
                    f"{latest_confidence*100:.1f}%"
                ),
                (10,108),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.50,
                (255,255,255),
                1,
                cv2.LINE_AA
            )


            if classifier_times:

                cv2.putText(
                    display,
                    (
                        f"Classifier: "
                        f"{classifier_times[-1]:.2f} ms"
                    ),
                    (10,132),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (255,255,255),
                    1,
                    cv2.LINE_AA
                )


        # ==========================================================================================
        # SHOW
        # ==========================================================================================

        cv2.imshow(
            "CISLR - Immediate Real-Time Sign Recognition",
            display
        )


        # ==========================================================================================
        # KEYBOARD
        # ==========================================================================================

        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        if (
            key == ord("q")
            or
            key == 27
        ):

            break


        elif key == ord("r"):

            landmark_buffer.clear()

            last_valid_feature[:] = 0

            latest_label = "COLLECTING"

            latest_confidence = 0.0

            latest_top3 = []


            print(
                "\n[RESET] Landmark sequence cleared."
            )


        # ==========================================================================================
        # FRAME TIME
        # ==========================================================================================

        frame_ms = (
            time.perf_counter()
            -
            frame_start
        ) * 1000.0


        frame_times.append(
            frame_ms
        )


finally:

    cap.release()

    cv2.destroyAllWindows()


# ==================================================================================================
# 11. FINAL METRICS
# ==================================================================================================

duration = (
    time.perf_counter()
    -
    start_time
)


overall_fps = (

    frame_counter /
    duration

    if duration > 0

    else 0.0
)


def safe_mean(values):

    if len(values) == 0:
        return float("nan")

    return float(
        np.mean(
            values
        )
    )


def safe_median(values):

    if len(values) == 0:
        return float("nan")

    return float(
        np.median(
            values
        )
    )


print()
print("=" * 120)
print("L26D — REAL-TIME RESULTS")
print("=" * 120)

print()

print(
    "Frames processed          :",
    frame_counter
)

print(
    "Duration                  :",
    f"{duration:.2f} sec"
)

print(
    "Overall FPS               :",
    f"{overall_fps:.2f}"
)

print()

print(
    "Frames with valid hand    :",
    hand_frames
)

print(
    "Predictions generated     :",
    prediction_count
)

print()

print(
    "Mean frame time           :",
    f"{safe_mean(frame_times):.3f} ms"
)

print(
    "Median frame time         :",
    f"{safe_median(frame_times):.3f} ms"
)

print()

print(
    "Mean palm inference       :",
    f"{safe_mean(palm_times):.3f} ms"
)

print(
    "Mean landmark inference   :",
    f"{safe_mean(landmark_times):.3f} ms"
)

print()

print(
    "Mean BiGRU inference      :",
    f"{safe_mean(classifier_times):.3f} ms"
)

print(
    "Median BiGRU inference    :",
    f"{safe_median(classifier_times):.3f} ms"
)

print()

print(
    "VideoMAE used             : NO"
)

print(
    "OpenVINO temporal model   : YES"
)

print(
    "Locked test used          : NO"
)

print(
    "L27 rejection             : NO"
)

print(
    "L28 stabilization         : NO"
)

print()

print(
    "[DONE] L26D real-time responsiveness test finished."
)

CISLR — L26D LOW-LATENCY LIVE SIGN RECOGNITION

Available OpenVINO devices:
['CPU', 'GPU']

[PASS] Real-time classifier loaded.
Input : [1,32,63]
Output: [1,10]

WEBCAM
Resolution : 640 x 480
Camera FPS : 30.0

LIVE RECOGNITION STARTED

Palm detector       : OpenVINO
Landmark detector   : OpenVINO
Temporal classifier : OpenVINO BiGRU CPU
VideoMAE            : DISABLED

Q / ESC = quit
R       = reset sequence


L26D — REAL-TIME RESULTS

Frames processed          : 5239
Duration                  : 282.15 sec
Overall FPS               : 18.57

Frames with valid hand    : 3939
Predictions generated     : 5208

Mean frame time           : 53.826 ms
Median frame time         : 63.050 ms

Mean palm inference       : 3.640 ms
Mean landmark inference   : 3.557 ms

Mean BiGRU inference      : 0.866 ms
Median BiGRU inference    : 0.846 ms

VideoMAE used             : NO
OpenVINO temporal model   : YES
Locked test used          : NO
L27 rejection             : NO
L28 stabilization         : NO

[D

In [21]:
# ==================================================================================================
# CISLR — L27A
# DEVELOPMENT CALIBRATION FOR NO_SIGN / UNKNOWN REJECTION
# ==================================================================================================
#
# Uses:
#   OpenVINO palm + landmarks
#   frozen L26C 10-class BiGRU
#
# Does NOT use:
#   locked test set
#   VideoMAE
#   L28 temporal stabilization
#
# Keyboard:
#   V = collect one VALID_SIGN temporal sample
#   N = collect one NO_SIGN / UNKNOWN temporal sample
#   C = calculate and save rejection thresholds
#   Q / ESC = quit
#
# IMPORTANT:
#   VALID_SIGN:
#       perform one of the 10 known signs naturally.
#
#   NO_SIGN:
#       neutral hand
#       stationary hand
#       random movement
#       entering/leaving frame
#       non-vocabulary gestures
#
# ==================================================================================================

from pathlib import Path
from collections import deque

import cv2
import json
import time
import numpy as np
import pandas as pd
import openvino as ov


# ==================================================================================================
# 1. CONFIG
# ==================================================================================================

CAMERA_ID = 0

WIDTH = 640
HEIGHT = 480

SEQUENCE_LENGTH = 32
FEATURE_DIM = 63

CLASS_NAMES = [
    "monday",
    "interest",
    "thursday",
    "christmas",
    "pencil",
    "mother",
    "tuesday",
    "friday",
    "rubber",
    "wash"
]


MODEL_PATH = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL/audit/"
    "l26c_realtime_landmark_classifier/models/"
    "l26c_fast_landmark_bigru.xml"
)


OUTPUT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH/"
    "CISLR_LANDMARK_MODEL/audit/"
    "l27_nosign_rejection"
)


REPORT_ROOT = OUTPUT_ROOT / "reports"
CONFIG_ROOT = OUTPUT_ROOT / "config"


REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ==================================================================================================
# 2. CHECK HAND PIPELINE
# ==================================================================================================

if "detect_hands" not in globals():

    raise RuntimeError(
        "detect_hands() not found. "
        "Run the working OpenVINO palm/landmark cell first."
    )


# ==================================================================================================
# 3. LOAD L26C MODEL
# ==================================================================================================

core_l27 = ov.Core()


model = core_l27.read_model(
    str(MODEL_PATH)
)


net = core_l27.compile_model(
    model,
    "CPU",
    {
        "PERFORMANCE_HINT": "LATENCY"
    }
)


model_input = net.input(0)
model_output = net.output(0)


print("=" * 110)
print("CISLR — L27A NO_SIGN CALIBRATION")
print("=" * 110)

print()
print("Model :", MODEL_PATH)
print("Input :", model_input.partial_shape)
print("Output:", model_output.partial_shape)


# ==================================================================================================
# 4. FUNCTIONS
# ==================================================================================================

def softmax(x):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    x = x - np.max(x)

    e = np.exp(x)

    return e / np.sum(e)


def normalize_landmarks(
    landmarks
):

    xyz = np.asarray(
        landmarks,
        dtype=np.float32
    ).copy()


    if xyz.shape != (21,3):

        raise ValueError(
            f"Expected (21,3), got {xyz.shape}"
        )


    # SAME preprocessing as L26C.

    xyz -= xyz[0:1]


    scale = np.linalg.norm(
        xyz[9]
    )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = np.max(
            np.linalg.norm(
                xyz,
                axis=1
            )
        )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = 1.0


    xyz /= scale


    return xyz.reshape(
        63
    ).astype(
        np.float32
    )


def extract_landmarks(
    result
):

    if not isinstance(
        result,
        dict
    ):

        return None


    for key in [
        "landmarks",
        "screen_landmarks",
        "points"
    ]:

        if key in result:

            arr = np.asarray(
                result[key],
                dtype=np.float32
            )


            if arr.shape == (21,3):

                return arr


    return None


def calculate_metrics(
    sequence
):

    x = np.asarray(
        sequence,
        dtype=np.float32
    )[None,:,:]


    start = time.perf_counter()


    result = net(
        {
            model_input:
                x
        }
    )


    inference_ms = (
        time.perf_counter()
        -
        start
    ) * 1000.0


    logits = np.asarray(
        result[
            model_output
        ]
    ).reshape(-1)


    probabilities = softmax(
        logits
    )


    order = np.argsort(
        probabilities
    )[::-1]


    top1_id = int(
        order[0]
    )

    top2_id = int(
        order[1]
    )


    top1 = float(
        probabilities[
            top1_id
        ]
    )

    top2 = float(
        probabilities[
            top2_id
        ]
    )


    margin = (
        top1 -
        top2
    )


    # Normalized entropy:
    #
    # 0 -> extremely certain
    # 1 -> maximally uncertain

    eps = 1e-12


    entropy = float(
        -np.sum(
            probabilities
            *
            np.log(
                probabilities + eps
            )
        )
        /
        np.log(
            len(
                probabilities
            )
        )
    )


    return {

        "prediction":
            CLASS_NAMES[
                top1_id
            ],

        "top1":
            top1,

        "top2":
            top2,

        "margin":
            margin,

        "entropy":
            entropy,

        "inference_ms":
            inference_ms
    }


# ==================================================================================================
# 5. CAMERA
# ==================================================================================================

cap = cv2.VideoCapture(
    CAMERA_ID
)


if not cap.isOpened():

    raise RuntimeError(
        "Could not open webcam."
    )


cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    WIDTH
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    HEIGHT
)


try:

    cap.set(
        cv2.CAP_PROP_BUFFERSIZE,
        1
    )

except Exception:

    pass


# ==================================================================================================
# 6. STATE
# ==================================================================================================

landmark_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


hand_buffer = deque(
    maxlen=SEQUENCE_LENGTH
)


last_valid_feature = np.zeros(
    FEATURE_DIM,
    dtype=np.float32
)


latest_results = []


records = []


valid_count = 0
nosign_count = 0


print()
print("Controls:")
print(" V = save current 32-frame window as VALID_SIGN")
print(" N = save current 32-frame window as NO_SIGN")
print(" C = calculate thresholds")
print(" Q = quit")

print()
print("Recommendation:")
print(" collect >=30 VALID and >=30 NO_SIGN windows")
print()


# ==================================================================================================
# 7. LIVE CALIBRATION LOOP
# ==================================================================================================

try:

    while True:

        ok, frame = cap.read()


        if not ok:
            continue


        # ------------------------------------------------------------------------------------------
        # HAND DETECTION
        # ------------------------------------------------------------------------------------------

        try:

            latest_results, _, _ = (
                detect_hands(
                    frame
                )
            )

        except Exception:

            latest_results = []


        current_feature = None

        hand_present = False


        if len(latest_results) > 0:

            landmarks = extract_landmarks(
                latest_results[0]
            )


            if landmarks is not None:

                try:

                    current_feature = (
                        normalize_landmarks(
                            landmarks
                        )
                    )


                    if np.all(
                        np.isfinite(
                            current_feature
                        )
                    ):

                        last_valid_feature = (
                            current_feature.copy()
                        )

                        hand_present = True


                except Exception:

                    current_feature = None


        # ------------------------------------------------------------------------------------------
        # TEMPORAL BUFFER
        # ------------------------------------------------------------------------------------------

        if hand_present:

            landmark_buffer.append(
                current_feature.copy()
            )


        elif len(landmark_buffer) > 0:

            # Preserve continuity across a brief missed detection.

            landmark_buffer.append(
                last_valid_feature.copy()
            )


        hand_buffer.append(
            bool(
                hand_present
            )
        )


        # ------------------------------------------------------------------------------------------
        # CURRENT MODEL METRICS
        # ------------------------------------------------------------------------------------------

        current_metrics = None


        if (
            len(
                landmark_buffer
            )
            ==
            SEQUENCE_LENGTH
        ):

            current_metrics = (
                calculate_metrics(
                    np.stack(
                        landmark_buffer
                    )
                )
            )


        # ------------------------------------------------------------------------------------------
        # DRAW
        # ------------------------------------------------------------------------------------------

        display = frame.copy()


        if "draw_hand" in globals():

            for result in latest_results:

                try:

                    draw_hand(
                        display,
                        result
                    )

                except Exception:

                    pass


        h,w = display.shape[:2]


        overlay = display.copy()


        cv2.rectangle(
            overlay,
            (0,0),
            (w,185),
            (0,0,0),
            -1
        )


        cv2.addWeighted(
            overlay,
            0.65,
            display,
            0.35,
            0,
            display
        )


        cv2.putText(
            display,
            "L27A CALIBRATION",
            (10,24),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,255),
            2,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Buffer: "
                f"{len(landmark_buffer)}/32"
            ),
            (10,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.putText(
            display,
            (
                f"Saved VALID={valid_count} "
                f"NO_SIGN={nosign_count}"
            ),
            (10,75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.48,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        if current_metrics is not None:

            cv2.putText(
                display,
                (
                    f"Raw: "
                    f"{current_metrics['prediction'].upper()} "
                    f"{current_metrics['top1']*100:.1f}%"
                ),
                (10,105),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0,255,255),
                2,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                (
                    f"Margin "
                    f"{current_metrics['margin']:.3f} | "
                    f"Entropy "
                    f"{current_metrics['entropy']:.3f}"
                ),
                (10,132),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (255,255,255),
                1,
                cv2.LINE_AA
            )


        cv2.putText(
            display,
            "V=VALID   N=NO_SIGN   C=CALCULATE   Q=QUIT",
            (10,165),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (255,255,255),
            1,
            cv2.LINE_AA
        )


        cv2.imshow(
            "CISLR L27A - Rejection Calibration",
            display
        )


        # ------------------------------------------------------------------------------------------
        # KEYBOARD
        # ------------------------------------------------------------------------------------------

        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        if (
            key == ord("q")
            or
            key == 27
        ):

            break


        # ------------------------------------------------------------------------------------------
        # SAVE VALID / NO_SIGN WINDOW
        # ------------------------------------------------------------------------------------------

        if key in [
            ord("v"),
            ord("n")
        ]:

            if current_metrics is None:

                print(
                    "[WAIT] Need a complete 32-frame landmark sequence."
                )

                continue


            calibration_label = (

                "VALID_SIGN"

                if key == ord("v")

                else "NO_SIGN"
            )


            hand_ratio = float(
                np.mean(
                    np.asarray(
                        hand_buffer,
                        dtype=np.float32
                    )
                )
            )


            record = {

                "calibration_label":
                    calibration_label,

                "raw_prediction":
                    current_metrics[
                        "prediction"
                    ],

                "top1":
                    current_metrics[
                        "top1"
                    ],

                "top2":
                    current_metrics[
                        "top2"
                    ],

                "margin":
                    current_metrics[
                        "margin"
                    ],

                "entropy":
                    current_metrics[
                        "entropy"
                    ],

                "hand_ratio":
                    hand_ratio,

                "inference_ms":
                    current_metrics[
                        "inference_ms"
                    ]
            }


            records.append(
                record
            )


            if calibration_label == "VALID_SIGN":

                valid_count += 1

            else:

                nosign_count += 1


            print(
                f"[SAVE] {calibration_label:<10} | "
                f"raw={record['raw_prediction']:<10} | "
                f"conf={record['top1']:.3f} | "
                f"margin={record['margin']:.3f} | "
                f"entropy={record['entropy']:.3f} | "
                f"hand={record['hand_ratio']:.2f}"
            )


        # ------------------------------------------------------------------------------------------
        # CALCULATE THRESHOLDS
        # ------------------------------------------------------------------------------------------

        if key == ord("c"):

            if (
                valid_count < 10
                or
                nosign_count < 10
            ):

                print()
                print(
                    "[WAIT] Collect at least 10 VALID "
                    "and 10 NO_SIGN samples first."
                )

                continue


            calibration_df = pd.DataFrame(
                records
            )


            valid_df = calibration_df[
                calibration_df[
                    "calibration_label"
                ]
                ==
                "VALID_SIGN"
            ]


            nosign_df = calibration_df[
                calibration_df[
                    "calibration_label"
                ]
                ==
                "NO_SIGN"
            ]


            # ======================================================================================
            # DATA-DRIVEN GRID SEARCH
            #
            # Decision:
            #
            # ACCEPT if:
            #
            #   top1    >= confidence threshold
            #   margin  >= margin threshold
            #   entropy <= entropy threshold
            #   hand_ratio >= hand threshold
            #
            # Select thresholds by balanced accuracy.
            #
            # This is development calibration only.
            # ======================================================================================

            conf_candidates = np.linspace(
                0.10,
                0.95,
                18
            )


            margin_candidates = np.linspace(
                0.00,
                0.80,
                17
            )


            entropy_candidates = np.linspace(
                0.20,
                1.00,
                17
            )


            hand_candidates = [
                0.40,
                0.50,
                0.60,
                0.70,
                0.80
            ]


            best = None


            for conf_thr in conf_candidates:

                for margin_thr in margin_candidates:

                    for entropy_thr in entropy_candidates:

                        for hand_thr in hand_candidates:

                            accepted = (

                                (
                                    calibration_df[
                                        "top1"
                                    ]
                                    >=
                                    conf_thr
                                )

                                &

                                (
                                    calibration_df[
                                        "margin"
                                    ]
                                    >=
                                    margin_thr
                                )

                                &

                                (
                                    calibration_df[
                                        "entropy"
                                    ]
                                    <=
                                    entropy_thr
                                )

                                &

                                (
                                    calibration_df[
                                        "hand_ratio"
                                    ]
                                    >=
                                    hand_thr
                                )
                            )


                            truth_valid = (

                                calibration_df[
                                    "calibration_label"
                                ]
                                ==
                                "VALID_SIGN"
                            )


                            tp = int(
                                (
                                    accepted
                                    &
                                    truth_valid
                                ).sum()
                            )


                            fn = int(
                                (
                                    (~accepted)
                                    &
                                    truth_valid
                                ).sum()
                            )


                            tn = int(
                                (
                                    (~accepted)
                                    &
                                    (~truth_valid)
                                ).sum()
                            )


                            fp = int(
                                (
                                    accepted
                                    &
                                    (~truth_valid)
                                ).sum()
                            )


                            tpr = (
                                tp /
                                (tp + fn)
                                if (tp + fn)
                                else 0
                            )


                            tnr = (
                                tn /
                                (tn + fp)
                                if (tn + fp)
                                else 0
                            )


                            balanced_accuracy = (
                                tpr + tnr
                            ) / 2.0


                            # Prefer lower false-accept rate on ties.

                            candidate = {

                                "confidence_threshold":
                                    float(
                                        conf_thr
                                    ),

                                "margin_threshold":
                                    float(
                                        margin_thr
                                    ),

                                "entropy_threshold":
                                    float(
                                        entropy_thr
                                    ),

                                "hand_ratio_threshold":
                                    float(
                                        hand_thr
                                    ),

                                "balanced_accuracy":
                                    float(
                                        balanced_accuracy
                                    ),

                                "valid_accept_rate":
                                    float(
                                        tpr
                                    ),

                                "nosign_reject_rate":
                                    float(
                                        tnr
                                    ),

                                "TP":
                                    tp,

                                "FN":
                                    fn,

                                "TN":
                                    tn,

                                "FP":
                                    fp
                            }


                            if best is None:

                                best = candidate


                            else:

                                better = (

                                    candidate[
                                        "balanced_accuracy"
                                    ]
                                    >
                                    best[
                                        "balanced_accuracy"
                                    ]
                                )


                                tied_safer = (

                                    candidate[
                                        "balanced_accuracy"
                                    ]
                                    ==
                                    best[
                                        "balanced_accuracy"
                                    ]

                                    and

                                    candidate[
                                        "FP"
                                    ]
                                    <
                                    best[
                                        "FP"
                                    ]
                                )


                                if better or tied_safer:

                                    best = candidate


            # --------------------------------------------------------------------------------------
            # SAVE
            # --------------------------------------------------------------------------------------

            csv_path = (
                REPORT_ROOT /
                "l27_calibration_samples.csv"
            )


            calibration_df.to_csv(
                csv_path,
                index=False
            )


            threshold_path = (
                CONFIG_ROOT /
                "l27_rejection_thresholds.json"
            )


            threshold_config = {

                "stage":
                    "L27",

                "method":
                    (
                        "development webcam calibration; "
                        "confidence + top1-top2 margin + "
                        "normalized entropy + hand ratio"
                    ),

                "valid_samples":
                    int(
                        valid_count
                    ),

                "nosign_samples":
                    int(
                        nosign_count
                    ),

                "thresholds":
                    best,

                "locked_test_used":
                    False,

                "l28_temporal_stabilization":
                    False
            }


            with open(
                threshold_path,
                "w"
            ) as f:

                json.dump(
                    threshold_config,
                    f,
                    indent=2
                )


            print()
            print("=" * 110)
            print("L27 CALIBRATION RESULT")
            print("=" * 110)

            print()
            print(
                "VALID samples       :",
                valid_count
            )

            print(
                "NO_SIGN samples     :",
                nosign_count
            )

            print()
            print(
                "Confidence threshold:",
                f"{best['confidence_threshold']:.3f}"
            )

            print(
                "Margin threshold    :",
                f"{best['margin_threshold']:.3f}"
            )

            print(
                "Entropy threshold   :",
                f"{best['entropy_threshold']:.3f}"
            )

            print(
                "Hand ratio threshold:",
                f"{best['hand_ratio_threshold']:.3f}"
            )

            print()
            print(
                "Calibration balanced accuracy:",
                f"{best['balanced_accuracy']*100:.2f}%"
            )

            print(
                "Valid accept rate   :",
                f"{best['valid_accept_rate']*100:.2f}%"
            )

            print(
                "NO_SIGN reject rate :",
                f"{best['nosign_reject_rate']*100:.2f}%"
            )

            print()
            print(
                "Threshold file:"
            )

            print(
                threshold_path
            )

            print()
            print(
                "[PASS] Thresholds saved."
            )


finally:

    cap.release()

    cv2.destroyAllWindows()


print()
print(
    "[DONE] L27A calibration session ended."
)

CISLR — L27A NO_SIGN CALIBRATION

Model : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l26c_realtime_landmark_classifier/models/l26c_fast_landmark_bigru.xml
Input : [1,32,63]
Output: [1,10]

Controls:
 V = save current 32-frame window as VALID_SIGN
 N = save current 32-frame window as NO_SIGN
 C = calculate thresholds
 Q = quit

Recommendation:
 collect >=30 VALID and >=30 NO_SIGN windows

[SAVE] NO_SIGN    | raw=friday     | conf=0.422 | margin=0.151 | entropy=0.652 | hand=0.00
[SAVE] NO_SIGN    | raw=friday     | conf=0.422 | margin=0.151 | entropy=0.652 | hand=0.00
[SAVE] NO_SIGN    | raw=friday     | conf=0.422 | margin=0.151 | entropy=0.652 | hand=0.00
[SAVE] NO_SIGN    | raw=friday     | conf=0.422 | margin=0.151 | entropy=0.652 | hand=0.00
[SAVE] NO_SIGN    | raw=friday     | conf=0.422 | margin=0.151 | entropy=0.652 | hand=0.00
[SAVE] NO_SIGN    | raw=friday     | conf=0.422 | margin=0.151 | entropy=0.652 | hand=0.00
[SAVE] NO_SIGN    | raw=friday     | c

: 

In [3]:
# ==================================================================================================
# CISLR — L27 COMPLETE STANDALONE PIPELINE
# ==================================================================================================
#
# WORKS AFTER A FRESH KERNEL RESTART.
#
# Includes:
#   1. OpenVINO palm detector
#   2. 2016 MediaPipe-style palm anchors
#   3. Palm decoding
#   4. Weighted NMS
#   5. Palm -> rotated hand ROI
#   6. OpenVINO 21-point hand landmark detector
#   7. 63-D L26C landmark representation
#   8. Frozen OpenVINO BiGRU 10-class classifier
#   9. L27 calibration
#  10. Automatic rejection threshold search
#  11. Live SIGN / NO_SIGN rejection
#
# FIRST RUN:
#   No threshold JSON -> CALIBRATION MODE
#
# SECOND RUN:
#   Threshold JSON exists -> LIVE L27 MODE
#
# Calibration controls:
#   V = record VALID known sign
#   N = record NO_SIGN / unknown motion
#   C = calculate + save thresholds
#   Q = quit
#
# Live controls:
#   R = reset temporal sequence
#   Q = quit
#
# DOES NOT:
#   - use VideoMAE
#   - use locked test
#   - implement L28 stabilization
#
# ==================================================================================================

from pathlib import Path
from collections import deque
import cv2
import json
import math
import time
import numpy as np
import openvino as ov


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

LANDMARK_ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)

PALM_MODEL_PATH = (
    LANDMARK_ROOT /
    "models/pretrained/palm/palm_detection_full.tflite"
)

HAND_MODEL_PATH = (
    LANDMARK_ROOT /
    "models/pretrained/hand_landmark/hand_landmark_full.tflite"
)

CLASSIFIER_PATH = (
    LANDMARK_ROOT /
    "audit/l26c_realtime_landmark_classifier/models/"
    "l26c_fast_landmark_bigru.xml"
)

L27_ROOT = (
    LANDMARK_ROOT /
    "audit/l27_nosign_rejection"
)

CONFIG_ROOT = L27_ROOT / "config"
REPORT_ROOT = L27_ROOT / "reports"

CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

THRESHOLD_PATH = (
    CONFIG_ROOT /
    "l27_rejection_thresholds.json"
)

CALIBRATION_PATH = (
    REPORT_ROOT /
    "l27_calibration_samples.json"
)


# ==================================================================================================
# 1. BASIC CONFIGURATION
# ==================================================================================================

CAMERA_ID = 0

CAMERA_WIDTH = 640
CAMERA_HEIGHT = 480

PALM_SIZE = 192
LANDMARK_SIZE = 224

PALM_THRESHOLD = 0.50
NMS_THRESHOLD = 0.30

MAX_HANDS = 2

SEQUENCE_LENGTH = 32
FEATURE_DIM = 63


CLASS_NAMES = [
    "monday",
    "interest",
    "thursday",
    "christmas",
    "pencil",
    "mother",
    "tuesday",
    "friday",
    "rubber",
    "wash"
]


HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),

    (0,5),(5,6),(6,7),(7,8),

    (5,9),(9,10),(10,11),(11,12),

    (9,13),(13,14),(14,15),(15,16),

    (13,17),(17,18),(18,19),(19,20),

    (0,17)
]


# ==================================================================================================
# 2. VERIFY FILES
# ==================================================================================================

print("=" * 120)
print("CISLR — L27 STANDALONE INITIALIZATION")
print("=" * 120)


required_files = {
    "Palm detector": PALM_MODEL_PATH,
    "Landmark detector": HAND_MODEL_PATH,
    "L26C BiGRU": CLASSIFIER_PATH,
}


for name, path in required_files.items():

    print(
        f"{name:<22}: {path}"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\n{name} not found:\n{path}"
        )


print()
print("[PASS] Required model files found.")


# ==================================================================================================
# 3. OPENVINO
# ==================================================================================================

core = ov.Core()


print()
print(
    "OpenVINO version:",
    ov.__version__
)

print(
    "Available devices:",
    core.available_devices
)


# ==================================================================================================
# 4. LOAD PALM MODEL
# ==================================================================================================

print()
print("Loading palm detector...")


palm_model = core.read_model(
    str(PALM_MODEL_PATH)
)


palm_compiled = core.compile_model(
    palm_model,
    "CPU",
    {
        "PERFORMANCE_HINT":
            "LATENCY"
    }
)


palm_input = palm_compiled.input(0)

palm_outputs = [
    palm_compiled.output(i)
    for i in range(
        len(
            palm_compiled.outputs
        )
    )
]


print(
    "[PASS] Palm input:",
    palm_input.partial_shape
)


for i,o in enumerate(
    palm_outputs
):

    print(
        f"       output {i}:",
        o.partial_shape
    )


# ==================================================================================================
# 5. LOAD LANDMARK MODEL
# ==================================================================================================

print()
print("Loading landmark detector...")


hand_model = core.read_model(
    str(HAND_MODEL_PATH)
)


hand_compiled = core.compile_model(
    hand_model,
    "CPU",
    {
        "PERFORMANCE_HINT":
            "LATENCY"
    }
)


hand_input = hand_compiled.input(0)

hand_outputs = [
    hand_compiled.output(i)
    for i in range(
        len(
            hand_compiled.outputs
        )
    )
]


print(
    "[PASS] Landmark input:",
    hand_input.partial_shape
)


for i,o in enumerate(
    hand_outputs
):

    print(
        f"       output {i}:",
        o.partial_shape
    )


# ==================================================================================================
# 6. LOAD L26C CLASSIFIER
# ==================================================================================================

print()
print("Loading L26C temporal classifier...")


classifier_model = core.read_model(
    str(CLASSIFIER_PATH)
)


classifier_compiled = core.compile_model(
    classifier_model,
    "CPU",
    {
        "PERFORMANCE_HINT":
            "LATENCY"
    }
)


classifier_input = (
    classifier_compiled.input(0)
)

classifier_output = (
    classifier_compiled.output(0)
)


print(
    "[PASS] Classifier input :",
    classifier_input.partial_shape
)

print(
    "[PASS] Classifier output:",
    classifier_output.partial_shape
)


# ==================================================================================================
# 7. MODEL LAYOUT DETECTION
# ==================================================================================================

def get_static_shape(port):

    try:

        return tuple(
            int(x)
            for x in port.shape
        )

    except Exception:

        return None


PALM_INPUT_SHAPE = get_static_shape(
    palm_input
)

HAND_INPUT_SHAPE = get_static_shape(
    hand_input
)


def is_nhwc(shape):

    if shape is None:
        return True

    return (
        len(shape) == 4
        and
        shape[-1] == 3
    )


PALM_NHWC = is_nhwc(
    PALM_INPUT_SHAPE
)

HAND_NHWC = is_nhwc(
    HAND_INPUT_SHAPE
)


print()
print(
    "Palm layout:",
    "NHWC"
    if PALM_NHWC
    else "NCHW"
)

print(
    "Hand layout:",
    "NHWC"
    if HAND_NHWC
    else "NCHW"
)


# ==================================================================================================
# 8. GENERATE EXACT 2016 PALM ANCHORS
# ==================================================================================================
#
# L1 configuration:
#
# strides = [8, 16, 16, 16]
#
# Repeated stride groups are merged.
#
# stride 8:
#   24 x 24 x 2 = 1152
#
# stride 16:
#   12 x 12 x 6 = 864
#
# total:
#   2016
#
# ==================================================================================================

def generate_palm_anchors():

    anchors = []

    strides = [
        8,
        16,
        16,
        16
    ]


    layer_id = 0


    while layer_id < len(strides):

        stride = strides[
            layer_id
        ]


        last_same_stride = (
            layer_id
        )


        while (
            last_same_stride + 1
            <
            len(strides)
            and
            strides[
                last_same_stride + 1
            ]
            ==
            stride
        ):

            last_same_stride += 1


        number_of_layers = (
            last_same_stride
            -
            layer_id
            +
            1
        )


        anchors_per_cell = (
            2
            *
            number_of_layers
        )


        feature_map_size = int(
            math.ceil(
                PALM_SIZE /
                stride
            )
        )


        for y in range(
            feature_map_size
        ):

            for x in range(
                feature_map_size
            ):

                x_center = (
                    x + 0.5
                ) / feature_map_size

                y_center = (
                    y + 0.5
                ) / feature_map_size


                for _ in range(
                    anchors_per_cell
                ):

                    anchors.append(
                        [
                            x_center,
                            y_center,
                            1.0,
                            1.0
                        ]
                    )


        layer_id = (
            last_same_stride + 1
        )


    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )


    if anchors.shape != (
        2016,
        4
    ):

        raise RuntimeError(
            "Palm anchor generation failed. "
            f"Got {anchors.shape}, expected (2016,4)."
        )


    return anchors


PALM_ANCHORS = (
    generate_palm_anchors()
)


print()
print(
    "[PASS] Palm anchors:",
    PALM_ANCHORS.shape
)


# ==================================================================================================
# 9. SIGMOID
# ==================================================================================================

def sigmoid(x):

    x = np.clip(
        x,
        -80.0,
        80.0
    )

    return (
        1.0 /
        (
            1.0 +
            np.exp(-x)
        )
    )


# ==================================================================================================
# 10. LETTERBOX
# ==================================================================================================

def letterbox(
    frame,
    target_size
):

    h,w = frame.shape[:2]


    scale = min(
        target_size / w,
        target_size / h
    )


    new_w = int(
        round(
            w * scale
        )
    )

    new_h = int(
        round(
            h * scale
        )
    )


    resized = cv2.resize(
        frame,
        (
            new_w,
            new_h
        )
    )


    canvas = np.zeros(
        (
            target_size,
            target_size,
            3
        ),
        dtype=np.uint8
    )


    left = (
        target_size -
        new_w
    ) // 2

    top = (
        target_size -
        new_h
    ) // 2


    canvas[
        top:top+new_h,
        left:left+new_w
    ] = resized


    return (
        canvas,
        scale,
        left,
        top
    )


# ==================================================================================================
# 11. PREPARE INPUT
# ==================================================================================================

def prepare_model_input(
    rgb,
    nhwc=True
):

    x = (
        rgb.astype(
            np.float32
        )
        /
        255.0
    )


    if nhwc:

        x = x[
            None,
            ...
        ]

    else:

        x = np.transpose(
            x,
            (
                2,
                0,
                1
            )
        )[
            None,
            ...
        ]


    return x.astype(
        np.float32
    )


# ==================================================================================================
# 12. PALM OUTPUT IDENTIFICATION
# ==================================================================================================

def identify_palm_outputs(
    inference_result
):

    regression = None
    scores = None


    for output_port in palm_outputs:

        arr = np.asarray(
            inference_result[
                output_port
            ]
        )


        flat_shape = arr.shape


        if (
            arr.size
            ==
            2016 * 18
        ):

            regression = arr.reshape(
                2016,
                18
            )


        elif (
            arr.size
            ==
            2016
        ):

            scores = arr.reshape(
                2016
            )


    if regression is None:

        raise RuntimeError(
            "Could not identify palm regression output."
        )


    if scores is None:

        raise RuntimeError(
            "Could not identify palm score output."
        )


    return (
        regression.astype(
            np.float32
        ),
        scores.astype(
            np.float32
        )
    )


# ==================================================================================================
# 13. PALM DECODING
# ==================================================================================================

def decode_palms(
    regression,
    logits
):

    probabilities = sigmoid(
        logits
    )


    selected = np.where(
        probabilities
        >=
        PALM_THRESHOLD
    )[0]


    detections = []


    for idx in selected:

        raw = regression[
            idx
        ]


        anchor = PALM_ANCHORS[
            idx
        ]


        # ----------------------------------------------------------
        # MediaPipe palm decoder
        # x/y scale = 192
        # fixed anchor width/height = 1
        # ----------------------------------------------------------

        cx = (
            raw[0] /
            PALM_SIZE
            +
            anchor[0]
        )

        cy = (
            raw[1] /
            PALM_SIZE
            +
            anchor[1]
        )

        bw = (
            raw[2] /
            PALM_SIZE
        )

        bh = (
            raw[3] /
            PALM_SIZE
        )


        x1 = cx - bw / 2
        y1 = cy - bh / 2

        x2 = cx + bw / 2
        y2 = cy + bh / 2


        keypoints = []


        for k in range(7):

            kx = (
                raw[
                    4 + 2*k
                ]
                /
                PALM_SIZE
                +
                anchor[0]
            )

            ky = (
                raw[
                    5 + 2*k
                ]
                /
                PALM_SIZE
                +
                anchor[1]
            )


            keypoints.append(
                [
                    kx,
                    ky
                ]
            )


        detections.append(
            {
                "score":
                    float(
                        probabilities[
                            idx
                        ]
                    ),

                "box":
                    np.asarray(
                        [
                            x1,
                            y1,
                            x2,
                            y2
                        ],
                        dtype=np.float32
                    ),

                "keypoints":
                    np.asarray(
                        keypoints,
                        dtype=np.float32
                    )
            }
        )


    return detections


# ==================================================================================================
# 14. IOU
# ==================================================================================================

def box_iou(
    a,
    b
):

    x1 = max(
        float(a[0]),
        float(b[0])
    )

    y1 = max(
        float(a[1]),
        float(b[1])
    )

    x2 = min(
        float(a[2]),
        float(b[2])
    )

    y2 = min(
        float(a[3]),
        float(b[3])
    )


    iw = max(
        0.0,
        x2 - x1
    )

    ih = max(
        0.0,
        y2 - y1
    )


    intersection = (
        iw * ih
    )


    area_a = max(
        0.0,
        float(
            a[2] - a[0]
        )
    ) * max(
        0.0,
        float(
            a[3] - a[1]
        )
    )


    area_b = max(
        0.0,
        float(
            b[2] - b[0]
        )
    ) * max(
        0.0,
        float(
            b[3] - b[1]
        )
    )


    union = (
        area_a +
        area_b -
        intersection
    )


    if union <= 0:
        return 0.0


    return (
        intersection /
        union
    )


# ==================================================================================================
# 15. WEIGHTED NMS
# ==================================================================================================

def weighted_nms(
    detections
):

    if not detections:

        return []


    remaining = sorted(
        detections,
        key=lambda x:
            x["score"],
        reverse=True
    )


    output = []


    while remaining:

        reference = (
            remaining[0]
        )


        overlapping = []
        rest = []


        for detection in remaining:

            if (
                box_iou(
                    reference["box"],
                    detection["box"]
                )
                >
                NMS_THRESHOLD
            ):

                overlapping.append(
                    detection
                )

            else:

                rest.append(
                    detection
                )


        if len(
            overlapping
        ) == 1:

            merged = (
                overlapping[0]
            )


        else:

            weights = np.asarray(
                [
                    d["score"]
                    for d in overlapping
                ],
                dtype=np.float32
            )


            weights = (
                weights /
                np.sum(weights)
            )


            merged_box = np.sum(
                np.stack(
                    [
                        d["box"]
                        for d in overlapping
                    ]
                )
                *
                weights[:,None],
                axis=0
            )


            merged_keypoints = np.sum(
                np.stack(
                    [
                        d["keypoints"]
                        for d in overlapping
                    ]
                )
                *
                weights[
                    :,
                    None,
                    None
                ],
                axis=0
            )


            merged = {
                "score":
                    float(
                        max(
                            d["score"]
                            for d in overlapping
                        )
                    ),

                "box":
                    merged_box.astype(
                        np.float32
                    ),

                "keypoints":
                    merged_keypoints.astype(
                        np.float32
                    )
            }


        output.append(
            merged
        )


        remaining = rest


    return output[
        :MAX_HANDS
    ]


# ==================================================================================================
# 16. REMOVE PALM LETTERBOX
# ==================================================================================================

def palm_to_original_pixels(
    detection,
    scale,
    left,
    top
):

    det = {
        "score":
            detection[
                "score"
            ],

        "box":
            detection[
                "box"
            ].copy(),

        "keypoints":
            detection[
                "keypoints"
            ].copy()
    }


    box = det[
        "box"
    ]


    box[
        [0,2]
    ] = (
        box[
            [0,2]
        ]
        *
        PALM_SIZE
        -
        left
    ) / scale


    box[
        [1,3]
    ] = (
        box[
            [1,3]
        ]
        *
        PALM_SIZE
        -
        top
    ) / scale


    kp = det[
        "keypoints"
    ]


    kp[:,0] = (
        kp[:,0]
        *
        PALM_SIZE
        -
        left
    ) / scale


    kp[:,1] = (
        kp[:,1]
        *
        PALM_SIZE
        -
        top
    ) / scale


    return det


# ==================================================================================================
# 17. PALM -> ROTATED ROI
# ==================================================================================================

def normalize_angle(
    angle
):

    return (
        angle
        -
        2.0
        *
        math.pi
        *
        math.floor(
            (
                angle +
                math.pi
            )
            /
            (
                2.0 *
                math.pi
            )
        )
    )


def palm_to_roi(
    detection
):

    box = detection[
        "box"
    ]


    kp = detection[
        "keypoints"
    ]


    # L1:
    # wrist = keypoint 0
    # middle MCP proxy = keypoint 2

    wrist = kp[0]
    middle = kp[2]


    rotation = normalize_angle(

        math.pi / 2

        -

        math.atan2(
            -(
                middle[1]
                -
                wrist[1]
            ),
            (
                middle[0]
                -
                wrist[0]
            )
        )
    )


    rect_width = max(
        1.0,
        float(
            box[2] -
            box[0]
        )
    )

    rect_height = max(
        1.0,
        float(
            box[3] -
            box[1]
        )
    )


    center_x = (
        float(box[0]) +
        float(box[2])
    ) / 2.0

    center_y = (
        float(box[1]) +
        float(box[3])
    ) / 2.0


    shift_y = -0.5


    x_shift = (
        -
        rect_height
        *
        shift_y
        *
        math.sin(
            rotation
        )
    )


    y_shift = (
        rect_height
        *
        shift_y
        *
        math.cos(
            rotation
        )
    )


    center_x += x_shift
    center_y += y_shift


    rect_width *= 2.6
    rect_height *= 2.6


    # MediaPipe hand ROI is square.

    size = max(
        rect_width,
        rect_height
    )


    return {
        "cx":
            center_x,

        "cy":
            center_y,

        "width":
            size,

        "height":
            size,

        "rotation":
            rotation
    }


# ==================================================================================================
# 18. ROI CORNERS
# ==================================================================================================

def roi_corners(
    roi
):

    cx = roi["cx"]
    cy = roi["cy"]

    w = roi["width"]
    h = roi["height"]

    angle = roi[
        "rotation"
    ]


    local = np.asarray(
        [
            [
                -w/2,
                -h/2
            ],
            [
                w/2,
                -h/2
            ],
            [
                w/2,
                h/2
            ],
            [
                -w/2,
                h/2
            ]
        ],
        dtype=np.float32
    )


    c = math.cos(
        angle
    )

    s = math.sin(
        angle
    )


    rotation_matrix = np.asarray(
        [
            [c,-s],
            [s,c]
        ],
        dtype=np.float32
    )


    points = (
        local
        @
        rotation_matrix.T
    )


    points[:,0] += cx
    points[:,1] += cy


    return points.astype(
        np.float32
    )


# ==================================================================================================
# 19. WARP ROTATED ROI
# ==================================================================================================

def warp_hand_roi(
    frame,
    roi
):

    corners = roi_corners(
        roi
    )


    # TL, TR, BR, BL

    source = np.asarray(
        [
            corners[0],
            corners[1],
            corners[3]
        ],
        dtype=np.float32
    )


    destination = np.asarray(
        [
            [0,0],
            [
                LANDMARK_SIZE - 1,
                0
            ],
            [
                0,
                LANDMARK_SIZE - 1
            ]
        ],
        dtype=np.float32
    )


    M = cv2.getAffineTransform(
        source,
        destination
    )


    crop = cv2.warpAffine(
        frame,
        M,
        (
            LANDMARK_SIZE,
            LANDMARK_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT
    )


    M_inv = cv2.invertAffineTransform(
        M
    )


    return (
        crop,
        M,
        M_inv,
        corners
    )


# ==================================================================================================
# 20. LANDMARK OUTPUT IDENTIFICATION
# ==================================================================================================

def identify_landmark_output(
    result
):

    candidates = []


    for output_port in hand_outputs:

        arr = np.asarray(
            result[
                output_port
            ]
        )


        if arr.size == 63:

            candidates.append(
                arr.reshape(
                    21,
                    3
                ).astype(
                    np.float32
                )
            )


    if not candidates:

        raise RuntimeError(
            "No 63-value hand landmark output found."
        )


    # L1 rule:
    # screen landmarks generally have xy magnitude > 2,
    # world landmarks are much smaller.

    screen = max(
        candidates,
        key=lambda x:
            float(
                np.max(
                    np.abs(
                        x[:,:2]
                    )
                )
            )
    )


    return screen


# ==================================================================================================
# 21. INVERSE LANDMARK TRANSFORM
# ==================================================================================================

def inverse_landmarks(
    landmark_raw,
    M_inv,
    roi
):

    xy = landmark_raw[
        :,
        :2
    ].astype(
        np.float32
    )


    # Raw screen landmarks are in 224 ROI pixels.

    ones = np.ones(
        (
            21,
            1
        ),
        dtype=np.float32
    )


    homogeneous = np.concatenate(
        [
            xy,
            ones
        ],
        axis=1
    )


    projected = (
        homogeneous
        @
        M_inv.T
    )


    z = (
        landmark_raw[:,2]
        /
        LANDMARK_SIZE
        *
        roi["width"]
    )


    landmarks = np.concatenate(
        [
            projected,
            z[:,None]
        ],
        axis=1
    )


    return landmarks.astype(
        np.float32
    )


# ==================================================================================================
# 22. FULL HAND DETECTOR
# ==================================================================================================

def detect_hands(
    frame
):

    # ----------------------------------------------------------------------------------------------
    # PALM
    # ----------------------------------------------------------------------------------------------

    palm_start = (
        time.perf_counter()
    )


    palm_letterboxed, scale, left, top = (
        letterbox(
            frame,
            PALM_SIZE
        )
    )


    palm_rgb = cv2.cvtColor(
        palm_letterboxed,
        cv2.COLOR_BGR2RGB
    )


    palm_tensor = prepare_model_input(
        palm_rgb,
        PALM_NHWC
    )


    palm_result = (
        palm_compiled(
            {
                palm_input:
                    palm_tensor
            }
        )
    )


    regression, logits = (
        identify_palm_outputs(
            palm_result
        )
    )


    detections = decode_palms(
        regression,
        logits
    )


    detections = weighted_nms(
        detections
    )


    detections = [
        palm_to_original_pixels(
            d,
            scale,
            left,
            top
        )
        for d in detections
    ]


    palm_ms = (
        time.perf_counter()
        -
        palm_start
    ) * 1000.0


    # ----------------------------------------------------------------------------------------------
    # LANDMARKS
    # ----------------------------------------------------------------------------------------------

    landmark_start = (
        time.perf_counter()
    )


    results = []


    for detection in detections:

        try:

            roi = palm_to_roi(
                detection
            )


            crop, M, M_inv, corners = (
                warp_hand_roi(
                    frame,
                    roi
                )
            )


            crop_rgb = cv2.cvtColor(
                crop,
                cv2.COLOR_BGR2RGB
            )


            hand_tensor = prepare_model_input(
                crop_rgb,
                HAND_NHWC
            )


            landmark_result = (
                hand_compiled(
                    {
                        hand_input:
                            hand_tensor
                    }
                )
            )


            raw_landmarks = (
                identify_landmark_output(
                    landmark_result
                )
            )


            landmarks = inverse_landmarks(
                raw_landmarks,
                M_inv,
                roi
            )


            results.append(
                {
                    "landmarks":
                        landmarks,

                    "roi_corners":
                        corners,

                    "palm_score":
                        detection[
                            "score"
                        ]
                }
            )


        except Exception:

            continue


    landmark_ms = (
        time.perf_counter()
        -
        landmark_start
    ) * 1000.0


    return (
        results,
        palm_ms,
        landmark_ms
    )


# ==================================================================================================
# 23. DRAW HAND
# ==================================================================================================

def draw_hand(
    frame,
    result
):

    landmarks = result.get(
        "landmarks"
    )


    if landmarks is None:
        return


    points = np.asarray(
        landmarks[
            :,
            :2
        ],
        dtype=np.int32
    )


    # ROI

    corners = result.get(
        "roi_corners"
    )


    if corners is not None:

        cv2.polylines(
            frame,
            [
                np.asarray(
                    corners,
                    dtype=np.int32
                )
            ],
            True,
            (
                255,
                255,
                0
            ),
            1,
            cv2.LINE_AA
        )


    # Skeleton

    for a,b in HAND_CONNECTIONS:

        pa = tuple(
            points[a]
        )

        pb = tuple(
            points[b]
        )


        cv2.line(
            frame,
            pa,
            pb,
            (
                0,
                255,
                0
            ),
            2,
            cv2.LINE_AA
        )


    # Points

    for i,p in enumerate(
        points
    ):

        radius = (
            5
            if i == 0
            else 3
        )


        cv2.circle(
            frame,
            tuple(p),
            radius,
            (
                0,
                0,
                255
            ),
            -1,
            cv2.LINE_AA
        )


# ==================================================================================================
# 24. TEST HAND DETECTOR INITIALIZATION
# ==================================================================================================

print()
print(
    "[PASS] Palm + landmark pipeline rebuilt."
)


# ==================================================================================================
# 25. L26C FEATURE NORMALIZATION
# ==================================================================================================

def normalize_for_classifier(
    landmarks
):

    xyz = np.asarray(
        landmarks,
        dtype=np.float32
    ).copy()


    if xyz.shape != (
        21,
        3
    ):

        return None


    # Wrist centered.

    xyz -= xyz[
        0:1
    ]


    # Wrist -> middle MCP scale.

    scale = np.linalg.norm(
        xyz[9]
    )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = np.max(
            np.linalg.norm(
                xyz,
                axis=1
            )
        )


    if (
        not np.isfinite(scale)
        or
        scale < 1e-6
    ):

        scale = 1.0


    xyz /= scale


    feature = xyz.reshape(
        -1
    ).astype(
        np.float32
    )


    if feature.shape != (
        FEATURE_DIM,
    ):

        return None


    if not np.all(
        np.isfinite(
            feature
        )
    ):

        return None


    return feature


# ==================================================================================================
# 26. SOFTMAX
# ==================================================================================================

def softmax(
    x
):

    x = np.asarray(
        x,
        dtype=np.float32
    )


    x = (
        x -
        np.max(x)
    )


    e = np.exp(
        x
    )


    return (
        e /
        np.sum(e)
    )


# ==================================================================================================
# 27. CLASSIFY 32x63
# ==================================================================================================

def classify_sequence(
    sequence
):

    sequence = np.asarray(
        sequence,
        dtype=np.float32
    )


    if sequence.shape != (
        SEQUENCE_LENGTH,
        FEATURE_DIM
    ):

        raise ValueError(
            "Classifier expected "
            f"(32,63), got {sequence.shape}"
        )


    x = sequence[
        None,
        :,
        :
    ]


    t0 = time.perf_counter()


    result = classifier_compiled(
        {
            classifier_input:
                x
        }
    )


    inference_ms = (
        time.perf_counter()
        -
        t0
    ) * 1000.0


    logits = np.asarray(
        result[
            classifier_output
        ]
    ).reshape(
        -1
    )


    probabilities = softmax(
        logits
    )


    ranking = np.argsort(
        probabilities
    )[::-1]


    top1_id = int(
        ranking[0]
    )

    top2_id = int(
        ranking[1]
    )


    top1 = float(
        probabilities[
            top1_id
        ]
    )


    top2 = float(
        probabilities[
            top2_id
        ]
    )


    margin = (
        top1 -
        top2
    )


    entropy = float(
        -
        np.sum(
            probabilities
            *
            np.log(
                probabilities +
                1e-12
            )
        )
        /
        np.log(
            len(
                probabilities
            )
        )
    )


    return {
        "prediction":
            CLASS_NAMES[
                top1_id
            ],

        "confidence":
            top1,

        "top2":
            top2,

        "margin":
            margin,

        "entropy":
            entropy,

        "inference_ms":
            inference_ms
    }


# ==================================================================================================
# 28. CAMERA
# ==================================================================================================

def open_camera():

    cap = cv2.VideoCapture(
        CAMERA_ID
    )


    if not cap.isOpened():

        raise RuntimeError(
            "Could not open webcam."
        )


    cap.set(
        cv2.CAP_PROP_FRAME_WIDTH,
        CAMERA_WIDTH
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        CAMERA_HEIGHT
    )


    try:

        cap.set(
            cv2.CAP_PROP_BUFFERSIZE,
            1
        )

    except Exception:

        pass


    return cap


# ==================================================================================================
# 29. RECORD ONE INDEPENDENT CALIBRATION WINDOW
# ==================================================================================================

def record_calibration_window(
    cap,
    calibration_label
):

    features = []

    hand_flags = []


    last_feature = None


    print()
    print("-" * 100)

    print(
        "Recording:",
        calibration_label
    )


    if calibration_label == "VALID_SIGN":

        print(
            "Perform ONE of the 10 known signs naturally."
        )

    else:

        print(
            "Perform random / neutral / unknown movement."
        )


    print("-" * 100)


    while len(
        features
    ) < SEQUENCE_LENGTH:

        ok,frame = cap.read()


        if not ok:
            continue


        results, palm_ms, landmark_ms = (
            detect_hands(
                frame
            )
        )


        hand_present = False

        feature = None


        if results:

            feature = (
                normalize_for_classifier(
                    results[0][
                        "landmarks"
                    ]
                )
            )


            if feature is not None:

                hand_present = True

                last_feature = (
                    feature.copy()
                )


        hand_flags.append(
            int(
                hand_present
            )
        )


        if feature is not None:

            features.append(
                feature.copy()
            )


        elif last_feature is not None:

            features.append(
                last_feature.copy()
            )


        display = frame.copy()


        for result in results:

            draw_hand(
                display,
                result
            )


        cv2.rectangle(
            display,
            (
                0,
                0
            ),
            (
                display.shape[1],
                125
            ),
            (
                0,
                0,
                0
            ),
            -1
        )


        cv2.putText(
            display,
            (
                "RECORDING "
                +
                calibration_label
            ),
            (
                10,
                28
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (
                0,
                255,
                255
            ),
            2
        )


        cv2.putText(
            display,
            (
                f"Sequence "
                f"{len(features)}/32"
            ),
            (
                10,
                58
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.52,
            (
                255,
                255,
                255
            ),
            1
        )


        cv2.putText(
            display,
            (
                f"Palm {palm_ms:.1f} ms | "
                f"Landmark {landmark_ms:.1f} ms"
            ),
            (
                10,
                84
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (
                255,
                255,
                255
            ),
            1
        )


        cv2.putText(
            display,
            "Q = cancel recording",
            (
                10,
                108
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.42,
            (
                255,
                255,
                255
            ),
            1
        )


        cv2.imshow(
            "CISLR L27",
            display
        )


        key = (
            cv2.waitKey(1)
            &
            0xFF
        )


        if (
            key == ord("q")
            or
            key == 27
        ):

            return None


    sequence = np.stack(
        features,
        axis=0
    ).astype(
        np.float32
    )


    metrics = classify_sequence(
        sequence
    )


    metrics[
        "hand_ratio"
    ] = float(
        np.mean(
            hand_flags
        )
    )


    metrics[
        "label"
    ] = calibration_label


    print(
        f"[SAVED] {calibration_label:<10} | "
        f"raw={metrics['prediction']:<10} | "
        f"conf={metrics['confidence']:.3f} | "
        f"margin={metrics['margin']:.3f} | "
        f"entropy={metrics['entropy']:.3f} | "
        f"hand={metrics['hand_ratio']:.3f}"
    )


    return metrics


# ==================================================================================================
# 30. THRESHOLD SEARCH
# ==================================================================================================

def search_thresholds(
    records
):

    valid_records = [
        r
        for r in records
        if r["label"]
        ==
        "VALID_SIGN"
    ]


    nosign_records = [
        r
        for r in records
        if r["label"]
        ==
        "NO_SIGN"
    ]


    if (
        len(valid_records) < 10
        or
        len(nosign_records) < 10
    ):

        print()
        print(
            "[WAIT] Need at least "
            "10 VALID and 10 NO_SIGN samples."
        )

        return None


    confidence_grid = np.linspace(
        0.10,
        0.95,
        18
    )


    margin_grid = np.linspace(
        0.00,
        0.80,
        17
    )


    entropy_grid = np.linspace(
        0.20,
        1.00,
        17
    )


    hand_grid = [
        0.40,
        0.50,
        0.60,
        0.70,
        0.80
    ]


    best = None


    for conf_t in confidence_grid:

        for margin_t in margin_grid:

            for entropy_t in entropy_grid:

                for hand_t in hand_grid:

                    tp = 0
                    fn = 0
                    tn = 0
                    fp = 0


                    for record in records:

                        accepted = (

                            record[
                                "confidence"
                            ]
                            >=
                            conf_t

                            and

                            record[
                                "margin"
                            ]
                            >=
                            margin_t

                            and

                            record[
                                "entropy"
                            ]
                            <=
                            entropy_t

                            and

                            record[
                                "hand_ratio"
                            ]
                            >=
                            hand_t
                        )


                        actual_valid = (

                            record[
                                "label"
                            ]
                            ==
                            "VALID_SIGN"
                        )


                        if (
                            actual_valid
                            and
                            accepted
                        ):

                            tp += 1


                        elif (
                            actual_valid
                            and
                            not accepted
                        ):

                            fn += 1


                        elif (
                            not actual_valid
                            and
                            not accepted
                        ):

                            tn += 1


                        else:

                            fp += 1


                    valid_accept_rate = (

                        tp /
                        (
                            tp +
                            fn
                        )

                        if (
                            tp +
                            fn
                        )
                        else 0.0
                    )


                    nosign_reject_rate = (

                        tn /
                        (
                            tn +
                            fp
                        )

                        if (
                            tn +
                            fp
                        )
                        else 0.0
                    )


                    balanced_accuracy = (

                        valid_accept_rate
                        +
                        nosign_reject_rate

                    ) / 2.0


                    candidate = {

                        "confidence_threshold":
                            float(
                                conf_t
                            ),

                        "margin_threshold":
                            float(
                                margin_t
                            ),

                        "entropy_threshold":
                            float(
                                entropy_t
                            ),

                        "hand_ratio_threshold":
                            float(
                                hand_t
                            ),

                        "balanced_accuracy":
                            float(
                                balanced_accuracy
                            ),

                        "valid_accept_rate":
                            float(
                                valid_accept_rate
                            ),

                        "nosign_reject_rate":
                            float(
                                nosign_reject_rate
                            ),

                        "TP":
                            int(tp),

                        "FN":
                            int(fn),

                        "TN":
                            int(tn),

                        "FP":
                            int(fp)
                    }


                    if best is None:

                        best = candidate


                    elif (
                        candidate[
                            "balanced_accuracy"
                        ]
                        >
                        best[
                            "balanced_accuracy"
                        ]
                    ):

                        best = candidate


                    elif (
                        candidate[
                            "balanced_accuracy"
                        ]
                        ==
                        best[
                            "balanced_accuracy"
                        ]

                        and

                        candidate[
                            "FP"
                        ]
                        <
                        best[
                            "FP"
                        ]
                    ):

                        best = candidate


    return best


# ==================================================================================================
# 31. CALIBRATION MODE
# ==================================================================================================

if not THRESHOLD_PATH.exists():

    print()
    print("=" * 120)
    print("L27A — CALIBRATION MODE")
    print("=" * 120)

    print()
    print(
        "No saved L27 thresholds found."
    )

    print()
    print(
        "V = record VALID known sign"
    )

    print(
        "N = record NO_SIGN / unknown"
    )

    print(
        "C = calculate and SAVE thresholds"
    )

    print(
        "Q = quit"
    )

    print()
    print(
        "Recommended: 30+ VALID and 30+ NO_SIGN."
    )


    records = []


    if CALIBRATION_PATH.exists():

        try:

            with open(
                CALIBRATION_PATH
            ) as f:

                records = json.load(
                    f
                )


            print()
            print(
                f"[INFO] Recovered "
                f"{len(records)} previous samples."
            )


        except Exception:

            records = []


    cap = open_camera()


    try:

        while True:

            ok,frame = cap.read()


            if not ok:
                continue


            valid_count = sum(

                r["label"]
                ==
                "VALID_SIGN"

                for r in records
            )


            nosign_count = sum(

                r["label"]
                ==
                "NO_SIGN"

                for r in records
            )


            display = frame.copy()


            # Live hand display while waiting.

            try:

                preview_results, _, _ = (
                    detect_hands(
                        frame
                    )
                )


                for result in preview_results:

                    draw_hand(
                        display,
                        result
                    )


            except Exception:

                pass


            cv2.rectangle(
                display,
                (
                    0,
                    0
                ),
                (
                    display.shape[1],
                    150
                ),
                (
                    0,
                    0,
                    0
                ),
                -1
            )


            cv2.putText(
                display,
                "L27 CALIBRATION",
                (
                    10,
                    28
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (
                    255,
                    255,
                    255
                ),
                2
            )


            cv2.putText(
                display,
                (
                    f"VALID: "
                    f"{valid_count}"
                ),
                (
                    10,
                    58
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.50,
                (
                    0,
                    255,
                    0
                ),
                1
            )


            cv2.putText(
                display,
                (
                    f"NO_SIGN: "
                    f"{nosign_count}"
                ),
                (
                    10,
                    84
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.50,
                (
                    0,
                    165,
                    255
                ),
                1
            )


            cv2.putText(
                display,
                (
                    "V=VALID  N=NO_SIGN  "
                    "C=CALCULATE  Q=QUIT"
                ),
                (
                    10,
                    120
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.43,
                (
                    255,
                    255,
                    255
                ),
                1
            )


            cv2.imshow(
                "CISLR L27",
                display
            )


            key = (
                cv2.waitKey(1)
                &
                0xFF
            )


            # --------------------------------------------------------------------------------------
            # QUIT
            # --------------------------------------------------------------------------------------

            if (
                key == ord("q")
                or
                key == 27
            ):

                break


            # --------------------------------------------------------------------------------------
            # VALID
            # --------------------------------------------------------------------------------------

            elif key == ord("v"):

                sample = (
                    record_calibration_window(
                        cap,
                        "VALID_SIGN"
                    )
                )


                if sample is not None:

                    records.append(
                        sample
                    )


                    with open(
                        CALIBRATION_PATH,
                        "w"
                    ) as f:

                        json.dump(
                            records,
                            f,
                            indent=2
                        )


            # --------------------------------------------------------------------------------------
            # NO SIGN
            # --------------------------------------------------------------------------------------

            elif key == ord("n"):

                sample = (
                    record_calibration_window(
                        cap,
                        "NO_SIGN"
                    )
                )


                if sample is not None:

                    records.append(
                        sample
                    )


                    with open(
                        CALIBRATION_PATH,
                        "w"
                    ) as f:

                        json.dump(
                            records,
                            f,
                            indent=2
                        )


            # --------------------------------------------------------------------------------------
            # CALCULATE
            # --------------------------------------------------------------------------------------

            elif key == ord("c"):

                best = search_thresholds(
                    records
                )


                if best is None:
                    continue


                valid_count = sum(

                    r["label"]
                    ==
                    "VALID_SIGN"

                    for r in records
                )


                nosign_count = sum(

                    r["label"]
                    ==
                    "NO_SIGN"

                    for r in records
                )


                threshold_config = {

                    "stage":
                        "L27",

                    "status":
                        "DEVELOPMENT_CALIBRATION",

                    "classifier":
                        "L26C OpenVINO BiGRU",

                    "classifier_validation_accuracy":
                        0.60,

                    "sequence_length":
                        SEQUENCE_LENGTH,

                    "feature_dim":
                        FEATURE_DIM,

                    "valid_samples":
                        int(
                            valid_count
                        ),

                    "nosign_samples":
                        int(
                            nosign_count
                        ),

                    "thresholds":
                        best,

                    "locked_test_used":
                        False,

                    "videomae_used":
                        False,

                    "l28_stabilization":
                        False
                }


                with open(
                    THRESHOLD_PATH,
                    "w"
                ) as f:

                    json.dump(
                        threshold_config,
                        f,
                        indent=2
                    )


                print()
                print("=" * 120)
                print("L27 — CALIBRATION COMPLETE")
                print("=" * 120)

                print()
                print(
                    "VALID samples:",
                    valid_count
                )

                print(
                    "NO_SIGN samples:",
                    nosign_count
                )

                print()
                print(
                    "Confidence threshold:",
                    f"{best['confidence_threshold']:.3f}"
                )

                print(
                    "Margin threshold:",
                    f"{best['margin_threshold']:.3f}"
                )

                print(
                    "Entropy threshold:",
                    f"{best['entropy_threshold']:.3f}"
                )

                print(
                    "Hand ratio threshold:",
                    f"{best['hand_ratio_threshold']:.3f}"
                )

                print()
                print(
                    "Calibration balanced accuracy:",
                    f"{best['balanced_accuracy']*100:.2f}%"
                )

                print(
                    "Valid accept rate:",
                    f"{best['valid_accept_rate']*100:.2f}%"
                )

                print(
                    "NO_SIGN reject rate:",
                    f"{best['nosign_reject_rate']*100:.2f}%"
                )

                print()
                print(
                    "[PASS] Threshold file saved:"
                )

                print(
                    THRESHOLD_PATH
                )

                print()
                print(
                    "Now rerun THIS SAME CELL "
                    "for live L27."
                )


                break


    finally:

        cap.release()

        cv2.destroyAllWindows()


# ==================================================================================================
# 32. LIVE L27 MODE
# ==================================================================================================

else:

    print()
    print("=" * 120)
    print("L27B — LIVE NO_SIGN REJECTION")
    print("=" * 120)


    with open(
        THRESHOLD_PATH
    ) as f:

        threshold_config = (
            json.load(
                f
            )
        )


    thresholds = threshold_config[
        "thresholds"
    ]


    CONF_T = float(
        thresholds[
            "confidence_threshold"
        ]
    )


    MARGIN_T = float(
        thresholds[
            "margin_threshold"
        ]
    )


    ENTROPY_T = float(
        thresholds[
            "entropy_threshold"
        ]
    )


    HAND_T = float(
        thresholds[
            "hand_ratio_threshold"
        ]
    )


    print()
    print(
        f"Confidence >= {CONF_T:.3f}"
    )

    print(
        f"Margin     >= {MARGIN_T:.3f}"
    )

    print(
        f"Entropy    <= {ENTROPY_T:.3f}"
    )

    print(
        f"Hand ratio >= {HAND_T:.3f}"
    )


    feature_buffer = deque(
        maxlen=SEQUENCE_LENGTH
    )


    hand_buffer = deque(
        maxlen=SEQUENCE_LENGTH
    )


    last_feature = None


    raw_prediction = "-"

    final_prediction = (
        "NO_SIGN"
    )


    confidence = 0.0
    margin = 0.0
    entropy = 1.0

    classifier_ms = 0.0


    accepted_windows = 0
    rejected_windows = 0

    frame_count = 0


    fps_counter = 0

    fps_start = (
        time.perf_counter()
    )

    live_fps = 0.0


    cap = open_camera()


    print()
    print(
        "[PASS] Webcam opened."
    )

    print(
        "Q/ESC = quit | R = reset"
    )


    try:

        while True:

            ok,frame = cap.read()


            if not ok:
                continue


            frame_count += 1


            # --------------------------------------------------------------------------------------
            # OPENVINO HAND DETECTION
            # --------------------------------------------------------------------------------------

            results, palm_ms, landmark_ms = (
                detect_hands(
                    frame
                )
            )


            hand_present = False

            current_feature = None


            if results:

                current_feature = (
                    normalize_for_classifier(
                        results[0][
                            "landmarks"
                        ]
                    )
                )


                if current_feature is not None:

                    hand_present = True

                    last_feature = (
                        current_feature.copy()
                    )


            # --------------------------------------------------------------------------------------
            # HAND EVIDENCE
            # --------------------------------------------------------------------------------------

            hand_buffer.append(
                int(
                    hand_present
                )
            )


            hand_ratio = float(
                np.mean(
                    hand_buffer
                )
            )


            # --------------------------------------------------------------------------------------
            # TEMPORAL BUFFER
            # --------------------------------------------------------------------------------------

            if current_feature is not None:

                feature_buffer.append(
                    current_feature.copy()
                )


            elif last_feature is not None:

                feature_buffer.append(
                    last_feature.copy()
                )


            # --------------------------------------------------------------------------------------
            # DEFAULT = NO_SIGN
            # --------------------------------------------------------------------------------------

            final_prediction = (
                "NO_SIGN"
            )


            # --------------------------------------------------------------------------------------
            # CLASSIFICATION + L27 REJECTION
            # --------------------------------------------------------------------------------------

            if (
                len(
                    feature_buffer
                )
                ==
                SEQUENCE_LENGTH
            ):

                metrics = classify_sequence(
                    np.stack(
                        feature_buffer
                    )
                )


                raw_prediction = metrics[
                    "prediction"
                ]


                confidence = metrics[
                    "confidence"
                ]


                margin = metrics[
                    "margin"
                ]


                entropy = metrics[
                    "entropy"
                ]


                classifier_ms = metrics[
                    "inference_ms"
                ]


                accept = (

                    confidence
                    >=
                    CONF_T

                    and

                    margin
                    >=
                    MARGIN_T

                    and

                    entropy
                    <=
                    ENTROPY_T

                    and

                    hand_ratio
                    >=
                    HAND_T
                )


                if accept:

                    final_prediction = (
                        raw_prediction
                    )

                    accepted_windows += 1


                else:

                    final_prediction = (
                        "NO_SIGN"
                    )

                    rejected_windows += 1


            # --------------------------------------------------------------------------------------
            # DRAW
            # --------------------------------------------------------------------------------------

            display = frame.copy()


            for result in results:

                draw_hand(
                    display,
                    result
                )


            # --------------------------------------------------------------------------------------
            # FPS
            # --------------------------------------------------------------------------------------

            fps_counter += 1


            now = time.perf_counter()


            if (
                now -
                fps_start
                >=
                0.5
            ):

                live_fps = (

                    fps_counter
                    /
                    (
                        now -
                        fps_start
                    )
                )


                fps_counter = 0

                fps_start = now


            # --------------------------------------------------------------------------------------
            # UI
            # --------------------------------------------------------------------------------------

            overlay = display.copy()


            cv2.rectangle(
                overlay,
                (
                    0,
                    0
                ),
                (
                    display.shape[1],
                    205
                ),
                (
                    0,
                    0,
                    0
                ),
                -1
            )


            cv2.addWeighted(
                overlay,
                0.65,
                display,
                0.35,
                0,
                display
            )


            if (
                final_prediction
                ==
                "NO_SIGN"
            ):

                status_color = (
                    0,
                    165,
                    255
                )

            else:

                status_color = (
                    0,
                    255,
                    0
                )


            cv2.putText(
                display,
                (
                    f"L27 | FPS "
                    f"{live_fps:.1f}"
                ),
                (
                    10,
                    24
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (
                    255,
                    255,
                    255
                ),
                1,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                final_prediction.upper(),
                (
                    10,
                    62
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.90,
                status_color,
                2,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                (
                    f"Raw: "
                    f"{raw_prediction.upper()}"
                ),
                (
                    10,
                    92
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.46,
                (
                    255,
                    255,
                    255
                ),
                1,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                (
                    f"Conf "
                    f"{confidence:.3f} | "
                    f"Margin "
                    f"{margin:.3f} | "
                    f"Entropy "
                    f"{entropy:.3f}"
                ),
                (
                    10,
                    118
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.40,
                (
                    255,
                    255,
                    255
                ),
                1,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                (
                    f"Hand "
                    f"{hand_ratio:.2f} | "
                    f"Buffer "
                    f"{len(feature_buffer)}/32"
                ),
                (
                    10,
                    144
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.40,
                (
                    255,
                    255,
                    255
                ),
                1,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                (
                    f"Palm "
                    f"{palm_ms:.1f} ms | "
                    f"Landmark "
                    f"{landmark_ms:.1f} ms | "
                    f"BiGRU "
                    f"{classifier_ms:.2f} ms"
                ),
                (
                    10,
                    170
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.39,
                (
                    255,
                    255,
                    255
                ),
                1,
                cv2.LINE_AA
            )


            cv2.putText(
                display,
                "Q=QUIT | R=RESET",
                (
                    10,
                    194
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.38,
                (
                    255,
                    255,
                    255
                ),
                1,
                cv2.LINE_AA
            )


            cv2.imshow(
                "CISLR L27 - Live NO_SIGN Rejection",
                display
            )


            # --------------------------------------------------------------------------------------
            # KEYS
            # --------------------------------------------------------------------------------------

            key = (
                cv2.waitKey(1)
                &
                0xFF
            )


            if (
                key == ord("q")
                or
                key == 27
            ):

                break


            elif key == ord("r"):

                feature_buffer.clear()

                hand_buffer.clear()

                last_feature = None

                raw_prediction = "-"

                final_prediction = (
                    "NO_SIGN"
                )

                confidence = 0.0
                margin = 0.0
                entropy = 1.0


                print(
                    "[RESET] Temporal buffer cleared."
                )


    finally:

        cap.release()

        cv2.destroyAllWindows()


    print()
    print("=" * 120)
    print("L27 LIVE SESSION COMPLETE")
    print("=" * 120)

    print()
    print(
        "Frames:",
        frame_count
    )

    print(
        "Accepted windows:",
        accepted_windows
    )

    print(
        "Rejected windows:",
        rejected_windows
    )

    print()
    print(
        "Locked test used : NO"
    )

    print(
        "VideoMAE used    : NO"
    )

    print(
        "L28 stabilization: NO"
    )

CISLR — L27 STANDALONE INITIALIZATION
Palm detector         : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/palm/palm_detection_full.tflite
Landmark detector     : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/models/pretrained/hand_landmark/hand_landmark_full.tflite
L26C BiGRU            : /home/dipshikha/Music/cao/CISLR_RESEARCH/CISLR_LANDMARK_MODEL/audit/l26c_realtime_landmark_classifier/models/l26c_fast_landmark_bigru.xml

[PASS] Required model files found.

OpenVINO version: 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Available devices: ['CPU', 'GPU']

Loading palm detector...
[PASS] Palm input: [1,192,192,3]
       output 0: [1,2016,18]
       output 1: [1,2016,1]

Loading landmark detector...
[PASS] Landmark input: [1,224,224,3]
       output 0: [1,63]
       output 1: [1,1]
       output 2: [1,1]
       output 3: [1,63]

Loading L26C temporal classifier...
[PASS] Classifier input : [1,32,63]
[PASS] Classifier output: [1,10]

Palm

QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or sw


----------------------------------------------------------------------------------------------------
Recording: NO_SIGN
Perform random / neutral / unknown movement.
----------------------------------------------------------------------------------------------------
[SAVED] NO_SIGN    | raw=pencil     | conf=0.496 | margin=0.071 | entropy=0.427 | hand=0.364

----------------------------------------------------------------------------------------------------
Recording: NO_SIGN
Perform random / neutral / unknown movement.
----------------------------------------------------------------------------------------------------
[SAVED] NO_SIGN    | raw=friday     | conf=0.980 | margin=0.965 | entropy=0.052 | hand=0.125

----------------------------------------------------------------------------------------------------
Recording: NO_SIGN
Perform random / neutral / unknown movement.
----------------------------------------------------------------------------------------------------
[SAVED] NO_SI